In [1]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, TensorDataset
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# MNIST transform
transform = transforms.Compose([
    transforms.ToTensor()
])

print("Downloading / Loading MNIST from TorchVision...")

# Load dataset directly
train_full = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_full  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Convert to tensors
train_images = train_full.data.unsqueeze(1).float() / 255.0
train_labels = train_full.targets.long()

test_images  = test_full.data.unsqueeze(1).float() / 255.0
test_labels  = test_full.targets.long()

# Filter for digits 6 and 8
label1, label2 = 6, 8

train_mask = (train_labels == label1) | (train_labels == label2)
test_mask  = (test_labels == label1) | (test_labels == label2)

train_images = train_images[train_mask].to(device)
train_labels = train_labels[train_mask].to(device)

test_images = test_images[test_mask].to(device)
test_labels = test_labels[test_mask].to(device)

# Relabel: 6 → 0, 8 → 1
train_labels = torch.where(train_labels == label1,
                           torch.tensor(0).to(device),
                           torch.tensor(1).to(device))

test_labels = torch.where(test_labels == label1,
                          torch.tensor(0).to(device),
                          torch.tensor(1).to(device))

# Train / Val split
total_train = TensorDataset(train_images, train_labels)
train_size  = int(0.8 * len(total_train))
val_size    = len(total_train) - train_size

train_dataset, val_dataset = random_split(total_train, [train_size, val_size])

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size)
test_loader  = DataLoader(TensorDataset(test_images, test_labels), batch_size=batch_size)

print("MNIST loaded successfully!")
print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_labels))


Using device: cpu


100%|██████████| 9.91M/9.91M [00:02<00:00, 3.75MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 140kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.30MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.27MB/s]


MNIST loaded successfully!
Train: 9415
Val: 2354
Test: 1932


In [3]:
import urllib.request
import gzip
import shutil
import os

# EXACT folder your code references
base_dir = "/kaggle/input/mnist-dataset"

# Make sure the absolute path exists
os.makedirs(base_dir, exist_ok=True)

subfolders = [
    "train-images-idx3-ubyte",
    "train-labels-idx1-ubyte",
    "t10k-images-idx3-ubyte",
    "t10k-labels-idx1-ubyte"
]

for f in subfolders:
    os.makedirs(os.path.join(base_dir, f), exist_ok=True)

# Stable mirrors
files = {
    "train-images-idx3-ubyte": "https://storage.googleapis.com/cvdf-datasets/mnist/train-images-idx3-ubyte.gz",
    "train-labels-idx1-ubyte": "https://storage.googleapis.com/cvdf-datasets/mnist/train-labels-idx1-ubyte.gz",
    "t10k-images-idx3-ubyte": "https://storage.googleapis.com/cvdf-datasets/mnist/t10k-images-idx3-ubyte.gz",
    "t10k-labels-idx1-ubyte": "https://storage.googleapis.com/cvdf-datasets/mnist/t10k-labels-idx1-ubyte.gz"
}

def download_extract(url, out_file):
    gz_file = out_file + ".gz"
    print("Downloading", url)
    urllib.request.urlretrieve(url, gz_file)

    print("Extracting", gz_file)
    with gzip.open(gz_file, 'rb') as f_in:
        with open(out_file, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)

for fname, url in files.items():
    folder = os.path.join(base_dir, fname)
    out_file = os.path.join(folder, fname)
    download_extract(url, out_file)

print("DONE: IDX files installed in Kaggle-compatible structure.")


Extracting /kaggle/input/mnist-dataset\train-images-idx3-ubyte\train-images-idx3-ubyte.gz
Extracting /kaggle/input/mnist-dataset\train-labels-idx1-ubyte\train-labels-idx1-ubyte.gz
Extracting /kaggle/input/mnist-dataset\t10k-images-idx3-ubyte\t10k-images-idx3-ubyte.gz
Extracting /kaggle/input/mnist-dataset\t10k-labels-idx1-ubyte\t10k-labels-idx1-ubyte.gz
DONE: IDX files installed in Kaggle-compatible structure.


In [ ]:
import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import torch
from torch.utils.data import DataLoader, random_split, TensorDataset, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Specify the labels to filter
label1, label2 = 6,8

# File paths
data_dir = "/kaggle/input/mnist-dataset"
train_images_path = os.path.join(data_dir, "train-images-idx3-ubyte/train-images-idx3-ubyte")
train_labels_path = os.path.join(data_dir, "train-labels-idx1-ubyte/train-labels-idx1-ubyte")
test_images_path = os.path.join(data_dir, "t10k-images-idx3-ubyte/t10k-images-idx3-ubyte")
test_labels_path = os.path.join(data_dir, "t10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte")


def read_idx(filename):
    with open(filename, 'rb') as f:
        magic = int.from_bytes(f.read(4), byteorder='big')
        n_items = int.from_bytes(f.read(4), byteorder='big')
        if magic == 2051:  # Images
            rows = int.from_bytes(f.read(4), byteorder='big')
            cols = int.from_bytes(f.read(4), byteorder='big')
            data = np.frombuffer(f.read(), dtype=np.uint8).reshape(n_items, rows, cols)
        elif magic == 2049:  # Labels
            data = np.frombuffer(f.read(), dtype=np.uint8)
        else:
            raise ValueError("Invalid IDX file format!")
    return data


train_images = read_idx(train_images_path)
train_labels = read_idx(train_labels_path)
test_images = read_idx(test_images_path)
test_labels = read_idx(test_labels_path)

train_images = torch.tensor(train_images, dtype=torch.float32).unsqueeze(1) / 255.0
train_labels = torch.tensor(train_labels, dtype=torch.long)
test_images = torch.tensor(test_images, dtype=torch.float32).unsqueeze(1) / 255.0
test_labels = torch.tensor(test_labels, dtype=torch.long)

def filter_and_relabel(images, labels, label1, label2, device):
    idx = (labels == label1) | (labels == label2)
    filtered_images = images[idx]
    filtered_labels = labels[idx]

    filtered_labels = filtered_labels.to(device)
    zero_tensor = torch.tensor(0, device=device)
    one_tensor = torch.tensor(1, device=device)
    filtered_labels = torch.where(filtered_labels == label1, zero_tensor, one_tensor)

    return filtered_images.to(device), filtered_labels

train_images_filtered, train_labels_filtered = filter_and_relabel(train_images, train_labels, label1, label2, device)
test_images_filtered, test_labels_filtered = filter_and_relabel(test_images, test_labels, label1, label2, device)

total_train_data = TensorDataset(train_images_filtered, train_labels_filtered)

train_size = int(0.8 * len(total_train_data))  # 80% for training
val_size = len(total_train_data) - train_size  # 20% for validation
train_dataset, val_dataset = random_split(total_train_data, [train_size, val_size])
test_dataset = TensorDataset(test_images_filtered, test_labels_filtered)
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")
print(f"Test set size: {len(test_dataset)}")
import matplotlib.pyplot as plt

example_data, example_target = train_dataset[0]

print("Tensor representation:")
print(example_data)

example_image = example_data.squeeze().cpu().numpy()

plt.imshow(example_image, cmap="gray")
plt.title(f"Label: {example_target.item()}")
plt.axis("off")
plt.show()
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
def train_model(train_loader, val_loader, model, epochs=5, lr=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for data, targets in train_loader:
            data, targets = data.to(device), targets.to(device)
            optimizer.zero_grad()

            outputs = model(data)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

        if val_loader is not None:
            validate_model(val_loader, model)

    return model
def validate_model(val_loader, model):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for data, targets in val_loader:
            data, targets = data.to(device), targets.to(device)

            outputs = model(data)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == targets).sum().item()
            total += targets.size(0)

    print(f"Validation Accuracy: {correct / total:.4f}")
from sklearn.metrics import confusion_matrix

def test_model(test_loader, model):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()

    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for data, targets in test_loader:
            data, targets = data.to(device), targets.to(device)

            outputs = model(data)
            _, predicted = torch.max(outputs, 1)

            all_targets.extend(targets.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())

    cm = confusion_matrix(all_targets, all_predictions, labels=[0, 1])

    tp_class_0 = cm[0, 0]
    fn_class_0 = cm[0, 1]
    tp_class_1 = cm[1, 1]
    fn_class_1 = cm[1, 0]
    tpr_class_0 = tp_class_0 / (tp_class_0 + fn_class_0) if (tp_class_0 + fn_class_0) > 0 else 0.0
    tpr_class_1 = tp_class_1 / (tp_class_1 + fn_class_1) if (tp_class_1 + fn_class_1) > 0 else 0.0

    print(f"True Positive Rate for Class 0: {tpr_class_0:.4f}")
    print(f"True Positive Rate for Class 1: {tpr_class_1:.4f}")

    return tpr_class_0, tpr_class_1
model = SimpleCNN()
trained_model = train_model(train_loader, val_loader, model, epochs=5)
tpr_class_0, tpr_class_1 = test_model(test_loader, trained_model)
model_save_path = "MNIST_CNN_Orig_68.pth"
torch.save(model.state_dict(), model_save_path)
class VAE(nn.Module):
    def __init__(self, latent_dim=30):
        super(VAE, self).__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2_mu = nn.Linear(128, latent_dim)
        self.fc2_logvar = nn.Linear(128, latent_dim)

        self.fc3 = nn.Linear(latent_dim, 128)
        self.fc4 = nn.Linear(128, 32 * 7 * 7)
        self.conv_transpose1 = nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.conv_transpose2 = nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1)

        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(16)

    def encode(self, x):
        x = F.leaky_relu(self.conv1(x), negative_slope=0.1)
        x = self.bn1(F.leaky_relu(self.conv2(x), negative_slope=0.1))
        x = x.view(x.size(0), -1)
        x = F.leaky_relu(self.fc1(x), negative_slope=0.1)
        mu = self.fc2_mu(x)
        logvar = self.fc2_logvar(x)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        z = F.leaky_relu(self.fc3(z), negative_slope=0.1)
        z = F.leaky_relu(self.fc4(z), negative_slope=0.1)
        z = z.view(z.size(0), 32, 7, 7)
        z = self.bn2(F.leaky_relu(self.conv_transpose1(z), negative_slope=0.1))
        z = torch.sigmoid(self.conv_transpose2(z))
        return z

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

    def loss_function(self, recon_x, x, mu, logvar):
        BCE = F.binary_cross_entropy(recon_x.view(-1, 28 * 28), x.view(-1, 28 * 28), reduction='sum')
        KL = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        return BCE + KL
def train_vae(train_loader, vae_model, epochs=5, lr=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    vae_model.to(device)

    optimizer = torch.optim.Adam(vae_model.parameters(), lr=lr)

    best_model_state = None
    best_loss = float('inf')

    vae_model.train()
    for epoch in range(epochs):
        total_loss = 0

        for data, _ in train_loader:
            data = data.to(device)

            optimizer.zero_grad()

            recon_batch, mu, logvar = vae_model(data)
            loss = vae_model.loss_function(recon_batch, data, mu, logvar)
            loss.backward()

            total_loss += loss.item()
            optimizer.step()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            best_model_state = vae_model.state_dict()

    if best_model_state is not None:
        vae_model.load_state_dict(best_model_state)
    return vae_model
def test_vae(test_loader, vae_model):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    vae_model.to(device)
    vae_model.eval()

    with torch.no_grad():
        for data, _ in test_loader:
            data = data.to(device)

            recon_batch, _, _ = vae_model(data)

            # Plot the original and reconstructed images
            fig, axes = plt.subplots(1, 2)
            axes[0].imshow(data[0].cpu().squeeze(), cmap='gray')
            axes[0].set_title("Original Image")
            axes[1].imshow(recon_batch[0].cpu().squeeze(), cmap='gray')
            axes[1].set_title("Reconstructed Image")
            plt.show()

            break
vae = VAE(latent_dim=30)
trained_vae = train_vae(train_loader, vae, epochs=100)

# Test the VAE model and visualize reconstructed images
test_vae(test_loader, trained_vae)
vae_model_save_path = "MNIST_VAE_68_30LD.pth"
torch.save(vae.state_dict(), vae_model_save_path)
import random
import torch
from torch.utils.data import DataLoader
posnegdata = [[] for _ in range(4)]
posnegdata_test = [[] for _ in range(4)]

posnegtargets = [[] for _ in range(4)]
posnegtargets_test = [[] for _ in range(4)]

for input_data_batch, label_batch in train_loader:
    for pos_label in range(2):
        for neg_label in range(2):
            pos_indices = (label_batch == pos_label).nonzero(as_tuple=True)[0]
            neg_indices = (label_batch == neg_label).nonzero(as_tuple=True)[0]

            if len(pos_indices) > 0 and len(neg_indices) > 0:
                pos_data = input_data_batch[pos_indices]
                neg_data = input_data_batch[neg_indices]

                combined_data = torch.cat((pos_data, neg_data), dim=0)

                combined_labels = torch.cat((
                    torch.full((pos_data.size(0),), pos_label),  # Positive label
                    torch.full((neg_data.size(0),), neg_label)   # Negative label
                ))

                combined = list(zip(combined_data, combined_labels.tolist()))

                random.shuffle(combined)

                shuffled_data, shuffled_labels = zip(*combined)
                shuffled_data = torch.stack(shuffled_data)
                shuffled_labels = torch.tensor(shuffled_labels)

                posnegdata[pos_label * 2 + neg_label].append(shuffled_data)
                posnegtargets[pos_label * 2 + neg_label].append(shuffled_labels)

for i in range(4):
    if posnegdata[i]:
        posnegdata[i] = torch.cat(posnegdata[i], dim=0)
        posnegtargets[i] = torch.cat(posnegtargets[i], dim=0)


for input_data_batch, label_batch in test_loader:
    for pos_label in range(2):
        for neg_label in range(2):
            pos_indices = (label_batch == pos_label).nonzero(as_tuple=True)[0]
            neg_indices = (label_batch == neg_label).nonzero(as_tuple=True)[0]

            if len(pos_indices) > 0 and len(neg_indices) > 0:
                pos_data = input_data_batch[pos_indices]
                neg_data = input_data_batch[neg_indices]

                combined_data = torch.cat((pos_data, neg_data), dim=0)
                combined_labels = torch.cat((
                    torch.full((pos_data.size(0),), pos_label),
                    torch.full((neg_data.size(0),), neg_label)
                ))

                combined = list(zip(combined_data, combined_labels.tolist()))

                random.shuffle(combined)

                shuffled_data, shuffled_labels = zip(*combined)
                shuffled_data = torch.stack(shuffled_data)
                shuffled_labels = torch.tensor(shuffled_labels)

                posnegdata_test[pos_label * 2 + neg_label].append(shuffled_data)
                posnegtargets_test[pos_label * 2 + neg_label].append(shuffled_labels)

for i in range(4):
    if posnegdata_test[i]:
        posnegdata_test[i] = torch.cat(posnegdata_test[i], dim=0)
        posnegtargets_test[i] = torch.cat(posnegtargets_test[i], dim=0)
posindexvectors = [[] for _ in range(4)]
negindexvectors = [[] for _ in range(4)]

for pos_label in range(2):
    for neg_label in range(2):
        i = pos_label * 2 + neg_label
        targets = posnegtargets[i]

        if targets is not None:
            pos_index_vector = (targets == pos_label).tolist()
            neg_index_vector = (targets == neg_label).tolist()

            posindexvectors[i] = pos_index_vector
            negindexvectors[i] = neg_index_vector
posindexvectors_test = [[] for _ in range(4)]
negindexvectors_test = [[] for _ in range(4)]

for pos_label in range(2):
    for neg_label in range(2):
        i = pos_label * 2 + neg_label
        targets_test = posnegtargets_test[i]

        if targets_test is not None:
            pos_index_vector_test = (targets_test == pos_label).tolist()
            neg_index_vector_test = (targets_test == neg_label).tolist()

            posindexvectors_test[i] = pos_index_vector_test
            negindexvectors_test[i] = neg_index_vector_test
import torch

num_ones = (posnegtargets[2] == 1).sum().item()
num_zeros = (posnegtargets[2] == 0).sum().item()

print(f"Number of 1's: {num_ones}")
print(f"Number of 0's: {num_zeros}")
batch_size = 64
latent_dim = 30
mu_pos_all = []
std_pos_all = []
mu_neg_all = []
std_neg_all = []
deltawmean_all = []
deltawstddev_all = []

batch_size = 64
input_dim = 28 * 28

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for i in range(0, 2):
    for k in range(0, 2):
        index = i * 2 + k
        posindexvector = posindexvectors[index]
        negindexvector = negindexvectors[index]

        if len(posindexvector) == 0 or len(negindexvector) == 0:
            continue

        mu_pos_batches = []
        logvar_pos_batches = []
        for j in range(0, len(posindexvector), batch_size):
            torch.cuda.empty_cache()
            batch_indices = posindexvector[j:j + batch_size]
            batch_pos = posnegdata[index][j:j + batch_size][batch_indices]
            if len(batch_pos) != 0:
                batch_pos = batch_pos.view(-1, 1, 28, 28).to(device)
                with torch.no_grad():
                    mean_pos, logvar_pos = vae.encode(batch_pos)
                mu_pos_batches.append(mean_pos)
                logvar_pos_batches.append(logvar_pos)
            else:
                print(f'Batch with no positives {i}{k}')

        if mu_pos_batches:
            mu_pos_to_add = torch.cat(mu_pos_batches, dim=0)
            mu_pos = torch.cat(mu_pos_batches, dim=0).mean(dim=0)
            std_pos_to_add = torch.cat(logvar_pos_batches, dim=0)
            std_pos = torch.cat(logvar_pos_batches, dim=0).mean(dim=0)
        else:
            continue

        mu_neg_batches = []
        logvar_neg_batches = []
        for j in range(0, len(negindexvector), batch_size):
            torch.cuda.empty_cache()
            batch_indices = negindexvector[j:j + batch_size]
            batch_neg = posnegdata[index][j:j + batch_size][batch_indices]  # Use correct indices
            if len(batch_neg) != 0:
                batch_neg = batch_neg.view(-1, 1, 28, 28).to(device)  # Reshape and move to GPU
                with torch.no_grad():  # Disable gradient computation
                    mean_neg, logvar_neg = vae.encode(batch_neg)
                mu_neg_batches.append(mean_neg)
                logvar_neg_batches.append(logvar_neg)
            else:
                print(f'Batch with no negatives {i}{k}')


        if mu_neg_batches:
            mu_neg_to_add = torch.cat(mu_neg_batches, dim=0)
            mu_neg = torch.cat(mu_neg_batches, dim=0).mean(dim=0)
            std_neg_to_add = torch.cat(logvar_neg_batches, dim=0)
            std_neg = torch.cat(logvar_neg_batches, dim=0).mean(dim=0)
        else:
            continue

        deltawmean = mu_neg - mu_pos
        deltawstddev = std_neg - std_pos
        print(len(deltawmean))

        deltawmean = deltawmean.view(1, -1)
        deltawstddev = deltawstddev.view(1, -1)
        print(deltawmean.shape)


        mu_pos_all.append(mu_pos_to_add)
        std_pos_all.append(std_pos_to_add)
        mu_neg_all.append(mu_neg_to_add)
        std_neg_all.append(std_neg_to_add)
        deltawmean_all.append(deltawmean)
        print(len(deltawmean_all))
        deltawstddev_all.append(deltawstddev)
mu_pos_all_test = []
std_pos_all_test = []
mu_neg_all_test = []
std_neg_all_test = []
deltawmean_all_test = []
deltawstddev_all_test = []

batch_size = 64

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for i in range(0, 2):
    for k in range(0, 2):
        index = i * 2 + k
        posindexvector_test = posindexvectors_test[index]
        negindexvector_test = negindexvectors_test[index]

        if len(posindexvector_test) == 0 or len(negindexvector_test) == 0:
            continue

        mu_pos_batches_test = []
        logvar_pos_batches_test = []
        for j in range(0, len(posindexvector_test), batch_size):
            torch.cuda.empty_cache()
            batch_indices = posindexvector_test[j:j + batch_size]
            batch_pos = posnegdata_test[index][j:j + batch_size][batch_indices]
            if len(batch_pos) != 0:
                batch_pos = batch_pos.view(-1, 1, 28, 28).to(device)
                with torch.no_grad():
                    mean_pos, logvar_pos = vae.encode(batch_pos)
                mu_pos_batches_test.append(mean_pos)
                logvar_pos_batches_test.append(logvar_pos)
            else:
                print(f'Batch with no positives {i}{k}')

        if mu_pos_batches_test:
            mu_pos_test_add = torch.cat(mu_pos_batches_test, dim=0)
            mu_pos_test = torch.cat(mu_pos_batches_test, dim=0).mean(dim=0)
            std_pos_test_add = torch.cat(logvar_pos_batches_test, dim=0)
            std_pos_test = torch.cat(logvar_pos_batches_test, dim=0).mean(dim=0)
        else:
            continue

        mu_neg_batches_test = []
        logvar_neg_batches_test = []
        for j in range(0, len(negindexvector_test), batch_size):
            torch.cuda.empty_cache()
            batch_indices = negindexvector_test[j:j + batch_size]
            batch_neg = posnegdata_test[index][j:j + batch_size][batch_indices]
            if len(batch_neg) != 0:
                batch_neg = batch_neg.view(-1, 1, 28, 28).to(device)
                with torch.no_grad():
                    mean_neg, logvar_neg = vae.encode(batch_neg)
                mu_neg_batches_test.append(mean_neg)
                logvar_neg_batches_test.append(logvar_neg)
            else:
                print(f'Batch with no negatives {i}{k}')

        if mu_neg_batches_test:
            mu_neg_test_add = torch.cat(mu_neg_batches_test, dim=0)
            mu_neg_test = torch.cat(mu_neg_batches_test, dim=0).mean(dim=0)
            std_neg_test_add = torch.cat(logvar_neg_batches_test, dim=0)
            std_neg_test = torch.cat(logvar_neg_batches_test, dim=0).mean(dim=0)
        else:
            continue

        deltawmean_test = mu_neg_test - mu_pos_test
        deltawstddev_test = std_neg_test - std_pos_test
        print(len(deltawmean_test))

        deltawmean_test = deltawmean_test.view(1, -1)
        deltawstddev_test = deltawstddev_test.view(1, -1)
        print(deltawmean_test.shape)

        mu_pos_all_test.append(mu_pos_test_add)
        std_pos_all_test.append(std_pos_test_add)
        mu_neg_all_test.append(mu_neg_test_add)
        std_neg_all_test.append(std_neg_test_add)
        deltawmean_all_test.append(deltawmean_test)
        print(len(deltawmean_all_test))
        deltawstddev_all_test.append(deltawstddev_test)
# Particle class for PSO
class Particle:
    def __init__(self, bounds):
        self.position = np.random.uniform(bounds[:, 0], bounds[:, 1])
        self.velocity = np.random.uniform(bounds[:, 0], bounds[:, 1])
        self.momentum = np.zeros_like(self.velocity)
        self.pbest_position = self.position.copy()
        self.pbest_value = float('inf')

    def update_personal_best(self, fitness):
        if fitness < self.pbest_value:
            self.pbest_value = fitness
            self.pbest_position = self.position.copy()
# EMPSO class
class EMPSO:
    def __init__(self, fitness_function, bounds, num_particles, beta, c1, c2, max_iter):
        self.fitness_function = fitness_function
        self.bounds = bounds
        self.num_particles = num_particles
        self.beta = beta
        self.c1 = c1
        self.c2 = c2
        self.max_iter = max_iter
        self.swarm = [Particle(bounds) for _ in range(num_particles)]
        self.gbest_position = None
        self.gbest_value = float('inf')

    def optimize(self):
        print("Entering optimize()\n")
        for _ in range(self.max_iter):
            for particle in self.swarm:
                fitness = self.fitness_function(particle.position)
                particle.update_personal_best(fitness)

                if fitness < self.gbest_value:
                    self.gbest_value = fitness
                    self.gbest_position = particle.position.copy()
                print(f'fitness: {fitness}, gbest_value: {self.gbest_value}')

            for particle in self.swarm:
                r1 = np.random.rand()
                r2 = np.random.rand()
                particle.velocity = (self.beta * particle.momentum + (1 - self.beta) * particle.velocity +
                                     self.c1 * r1 * (particle.pbest_position - particle.position) +
                                     self.c2 * r2 * (self.gbest_position - particle.position))
                particle.position += particle.velocity
                particle.momentum = self.beta * particle.momentum + (1 - self.beta) * particle.velocity

                particle.position = np.clip(particle.position, self.bounds[:, 0], self.bounds[:, 1])
        print("Exiting optimize()\n")

        return self.gbest_position, self.gbest_value
import torch
# Frobenius Norm for cost
def calc_cost(weightwmean, weightwstddev):
    weightwmean = torch.tensor(weightwmean) if not isinstance(weightwmean, torch.Tensor) else weightwmean
    weightwstddev = torch.tensor(weightwstddev) if not isinstance(weightwstddev, torch.Tensor) else weightwstddev
    norm_mean = torch.norm(weightwmean, p='fro')
    norm_stddev = torch.norm(weightwstddev, p='fro')
    total_cost = (norm_mean/latent_dim) + (norm_stddev/latent_dim)

    return total_cost.item()
def adversary_payoff(posnegdata, posindexvector, negindexvector, mu_pos, std_pos, autoencodermodel, a_mean, a_stddev, posnegtargets_tensor, w, model):
    device = next(model.parameters()).device
    model.load_state_dict({k: v.to(device) for k, v in w.items()})

    a_mean = a_mean.reshape((latent_dim))
    a_mean_tensor = torch.tensor(a_mean, dtype=torch.float32).to(device)
    a_stddev = a_stddev.reshape((latent_dim))
    a_stddev_tensor = torch.tensor(a_stddev, dtype=torch.float32).to(device)

    posnegdata_tensor = torch.tensor(posnegdata, dtype=torch.float32).to(device)
    posindexvector_tensor = torch.tensor(posindexvector, dtype=torch.long).to(device)
    negindexvector_tensor = torch.tensor(negindexvector, dtype=torch.long).to(device)

    autoencodermodel.to(device)

    adversarial_data = adversarial_manipulation(posnegdata_tensor, posindexvector_tensor, negindexvector_tensor, mu_pos, std_pos, a_mean_tensor, a_stddev_tensor, autoencodermodel)

    advdata_data_tensor = torch.tensor(adversarial_data, dtype=torch.float32).to(device)
    adversarial_data_tensor = torch.clamp(advdata_data_tensor, 0.0, 1.0)  # Clip values to valid range

    posnegtargets_tensor = posnegtargets_tensor.to(device)

    outputs = model(adversarial_data_tensor)
    _, predicted = torch.max(outputs, 1)

    correct = (predicted == posnegtargets_tensor).sum().item()
    total = len(posnegtargets_tensor)

    true_positives = ((predicted == 1) & (posnegtargets_tensor == 1)).sum().item()
    false_negatives = ((predicted == 0) & (posnegtargets_tensor == 1)).sum().item()

    tpr_class_1 = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    error = 1 - tpr_class_1
    print(f"tp:{true_positives}")
    print(f"total:{true_positives + false_negatives}")
    cost = calc_cost(a_mean, a_stddev)
    payoff = 1 + error - cost

    print(error)
    print(cost)
    print(payoff)

    return payoff
# Fitness function for PSO
def fitness_function(posnegdata, posindexvector, negindexvector, mu_pos, std_pos, autoencodermodel, a_mean, a_stddev, posnegtargets_tensor, model_weights, model):
    return -adversary_payoff(posnegdata, posindexvector, negindexvector, mu_pos, std_pos, autoencodermodel, a_mean, a_stddev, posnegtargets_tensor, model_weights, model)
import random
def empso_optimizer_fn(model, weightwmean_best_sa, weightwstddev_best_sa,
                        optimize_mean, posnegdata, posnegtargets, posindexvector, negindexvector,
                        deltawmean, deltawstddev, encodedposwmean, encodedposwstddev, autoencodermodel):

    posnegtargets_tensor = torch.Tensor(posnegtargets)
    model_weights = {k: v.clone() for k, v in model.state_dict().items()}
    a_mean, a_stddev = weightwmean_best_sa.clone(), weightwstddev_best_sa.clone()
    if optimize_mean:
        deltawmean_cpu = deltawmean.cpu()
        bounds = np.array([[-0.2 * abs(deltawmean_cpu[0][i]), 0.2 * abs(deltawmean_cpu[0][i])] for i in range(latent_dim)])
    else:
        deltawstddev_cpu = deltawstddev.cpu()
        bounds = np.array([[-0.2 * abs(deltawstddev_cpu[0][i]), 0.2 * abs(deltawstddev_cpu[0][i])] for i in range(latent_dim)])

    if optimize_mean:
        def fitness_fn(a_mean):
            return fitness_function(posnegdata, posindexvector, negindexvector, encodedposwmean, encodedposwstddev, autoencodermodel, a_mean, a_stddev, posnegtargets_tensor, model_weights, model)
    else:
        def fitness_fn(a_stddev):
            return fitness_function(posnegdata, posindexvector, negindexvector, encodedposwmean, encodedposwstddev, autoencodermodel, a_mean, a_stddev, posnegtargets_tensor, model_weights, model)

    num_particles = 30
    beta = 0.7
    c1 = 2
    c2 = 2
    max_iter = 70
    em_pso = EMPSO(fitness_fn, bounds, num_particles, beta, c1, c2, max_iter)
    alpha_star, _ = em_pso.optimize()
    print(alpha_star)

    alpha_star_reshaped = torch.tensor(alpha_star).reshape((latent_dim))

    if optimize_mean:
        weightwmean_optimal = alpha_star_reshaped.clone()
        weightwstddev_optimal = weightwstddev_best_sa.clone()
        payoff_optimal = -1 * fitness_function(posnegdata, posindexvector, negindexvector, encodedposwmean, encodedposwstddev, autoencodermodel, alpha_star_reshaped, a_stddev, posnegtargets_tensor, model_weights, model)
    else:
        weightwstddev_optimal = alpha_star_reshaped.clone()
        weightwmean_optimal = weightwmean_best_sa.clone()
        payoff_optimal = -1 * fitness_function(posnegdata, posindexvector, negindexvector, encodedposwmean, encodedposwstddev, autoencodermodel, a_mean, alpha_star_reshaped, posnegtargets_tensor, model_weights, model)

    print(payoff_optimal)

    return (
    torch.tensor(weightwmean_optimal, dtype=torch.float32),
    torch.tensor(weightwstddev_optimal, dtype=torch.float32),
    payoff_optimal,
    )
def alternating_least_squares(payoff_curr, error_curr, model, weightwmean, weightwstddev, optimizemean, posnegdata, posnegtargets, posindexvector, negindexvector,
                        deltawmean, deltawstddev, encodedposwmean, encodedposwstddev, autoencodermodel):
    payoff_best,error_best = payoff_curr,error_curr
    weightwmean_best,weightwstddev_best = weightwmean.clone(),weightwstddev.clone()
    weightwmean_best_up,weightwstddev_temp,payoff_best = empso_optimizer_fn(model,weightwmean_best,weightwstddev_best,True,
                                                                                   posnegdata,posnegtargets,posindexvector,negindexvector,deltawmean,deltawstddev
                                                                                   ,mu_pos,std_pos,vae)

    weightwmean_best_up,weightwstddev_best_up,payoff_best = empso_optimizer_fn(model,weightwmean_best_up,weightwstddev_best,False,
                                                                                   posnegdata,posnegtargets,posindexvector,negindexvector,deltawmean,deltawstddev,
                                                                                   mu_pos,std_pos,vae)

    adversarial_data = adversarial_manipulation(posnegdata, posindexvector, negindexvector, encodedposwmean, encodedposwstddev, weightwmean_best_up, weightwstddev_best_up, autoencodermodel)
    advdata_tensor = torch.Tensor(adversarial_data).to(device)
    adversarial_data_tensor = torch.clamp(advdata_tensor, 0.0, 1.0)
    posnegtargets_tensor_ann = torch.Tensor(posnegtargets)
    combined_dataset_ann = ConcatDataset([
            torch.utils.data.TensorDataset(adversarial_data_tensor, posnegtargets_tensor_ann)
        ])

    combined_dataloader_ann = torch.utils.data.DataLoader(
            combined_dataset_ann,
            batch_size=batch_size,
            shuffle=True
        )

    _, tpr_best = test_model(combined_dataloader_ann, model)
    error_best = 1 - tpr_best
    payoff_curr,error_curr = payoff_best,error_best
    print(weightwmean_best_up)
    print(weightwstddev_best_up)
    return (weightwmean_best_up,weightwstddev_best_up, payoff_best, error_best)
def adversarial_manipulation(posnegdata, posindices, negindices, encoded_pos_mean, encoded_pos_stddev, weight_w_mean, weight_w_stddev, vae_model):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed_value = 42
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
    posnegdata = posnegdata.to(device)
    posindices = torch.tensor(posindices, dtype=torch.bool, device=device)
    negindices = torch.tensor(negindices, dtype=torch.bool, device=device)
    encoded_pos_mean = encoded_pos_mean.to(device)
    encoded_pos_stddev = encoded_pos_stddev.to(device)
    weight_w_mean = weight_w_mean.to(device)
    weight_w_stddev = weight_w_stddev.to(device)

    adv_data = torch.zeros(posnegdata.shape, device=device)
    encoded_adv_data = vae_model.reparameterize(encoded_pos_mean + weight_w_mean,
                                                encoded_pos_stddev + weight_w_stddev)
    decoded_adv_data = vae_model.decode(encoded_adv_data)
    adv_data[posindices] = decoded_adv_data
    true_indices = torch.nonzero(posindices, as_tuple=True)[0]

    if len(true_indices) > 1:
        a = true_indices[0]
        b = true_indices[1]
        if torch.equal(adv_data[a], adv_data[b]):
            print('Flag')
            print(adv_data[a])
            print(adv_data[b])

    adv_data[negindices] = posnegdata[negindices]

    return adv_data
import copy

model_orig = copy.deepcopy(model)
model_m = copy.deepcopy(model)
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import ConcatDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

game_iter = 0
for pos in range(0, 2):
    for neg in range(0, 2):
        game_iter = 0
        print(f'Iterations of {pos} {neg}')
        if pos == neg:
            continue
        if pos == 0:
            continue
        index = pos * 2 + neg
        error_manipulated = []
        error_secured = []
        iter = 0
        exitgame = False
        payoff_curr = -float('inf')
        error_curr = -float('inf')
        train_manip = None
        posnegdata_loop = posnegdata[index]
        posindexvector = posindexvectors[index]
        negindexvector = negindexvectors[index]
        mu_pos = mu_pos_all[index].to(DEVICE)
        std_pos = std_pos_all[index].to(DEVICE)
        deltawmean = deltawmean_all[index].to(DEVICE)
        deltawstddev = deltawstddev_all[index].to(DEVICE)
        weightwstddev = torch.zeros(1, deltawstddev.size(1), device=DEVICE)
        weightwmean = torch.zeros(1, deltawmean.size(1), device=DEVICE)
        A_mu, A_stddev = weightwmean, weightwstddev
        posnegtargets_game = posnegtargets[index]
        posnegtargets_test_game = posnegtargets_test[index]
        payoff_curr = float('-inf')
        error_curr = float('-inf')
        max_iter = 50
        max_error_game = 0.15
        while not exitgame:
            model_orig.eval()
            print(A_mu)
            print(A_stddev)
            adversarial_data = adversarial_manipulation(posnegdata_loop, posindexvector, negindexvector, mu_pos, std_pos, A_mu, A_stddev, vae)
            advdata_tensor = torch.Tensor(adversarial_data).to(DEVICE)
            adversarial_data_tensor = torch.clamp(advdata_tensor, 0.0, 1.0)

            posnegtargets_tensor_ann = torch.Tensor(posnegtargets_game).to(DEVICE)
            combined_dataset_ann = ConcatDataset([
                    torch.utils.data.TensorDataset(adversarial_data_tensor, posnegtargets_tensor_ann)
                ])

            combined_dataloader_ann = torch.utils.data.DataLoader(
                    combined_dataset_ann,
                    batch_size=batch_size,
                    shuffle=True
                )

            _, tpr_curr = test_model(combined_dataloader_ann, model_orig)
            error_curr = (1 - tpr_curr)
            payoff_curr = 1 + (error_curr)
            print(payoff_curr)
            print(error_curr)

            weightwmean_best, weightwstddev_best, payoff_best, error_best = alternating_least_squares(
                payoff_curr, error_curr, model_orig, A_mu, A_stddev,
                True, posnegdata_loop, posnegtargets_game, posindexvector,
                negindexvector, deltawmean, deltawstddev, mu_pos,
                std_pos, vae)

            print('Error After:', error_best)
            print('Payoff After:', payoff_best)

            if payoff_best - payoff_curr <= 0:
                exitgame = True
                break
            if error_best > max_error_game:
                exitgame = True

            adversarial_data_sec = adversarial_manipulation(posnegdata_loop, posindexvector, negindexvector, mu_pos, std_pos, weightwmean_best, weightwstddev_best, vae)

            advdata_tensor_sec = torch.Tensor(adversarial_data_sec).to(DEVICE)
            adversarial_data_tensor_sec = torch.clamp(advdata_tensor_sec, 0.0, 1.0)

            posnegtargets_tensor_ann_sec = torch.Tensor(posnegtargets_game).to(DEVICE)
            posnegtargets_tensor_sec = posnegtargets_tensor_ann_sec.long()
            advdata_final_sec = adversarial_data_tensor_sec.new_tensor(adversarial_data_tensor_sec.data)
            posnegtargets_final_sec = posnegtargets_tensor_sec.new_tensor(posnegtargets_tensor_sec.data)

            train_dataset = train_loader.dataset
            original_dataset = train_dataset.dataset
            train_data, train_targets = original_dataset.tensors

            adversarial_data_tensor_sec = advdata_final_sec.to(DEVICE)
            posnegtargets_tensor_sec = posnegtargets_final_sec.to(DEVICE)

            train_data_tensor = train_data.to(DEVICE)
            train_targets_tensor = train_targets.to(DEVICE)

            adversarial_dataset_sec = torch.utils.data.TensorDataset(adversarial_data_tensor_sec, posnegtargets_tensor_sec)
            train_dataset = torch.utils.data.TensorDataset(train_data_tensor, train_targets_tensor)

            combined_dataset_ann_sec = torch.utils.data.ConcatDataset([adversarial_dataset_sec, train_dataset])
            print(len(combined_dataset_ann_sec))

            combined_dataloader_ann_sec = torch.utils.data.DataLoader(
                combined_dataset_ann_sec,
                batch_size=batch_size,
                shuffle=True
            )

            model_orig = model_orig.to(DEVICE)
            model_orig.train()

            num_epochs = 5
            loss_fn = nn.CrossEntropyLoss()
            optimizer_orig = optim.Adam(model_orig.parameters(), lr=0.001)

            for epoch in range(num_epochs):
                print(f"Starting Training Epoch {epoch + 1}/{num_epochs}")

                running_loss = 0.0
                correct = 0
                total = 0

                model_orig.to(DEVICE)
                for batch_idx, (data, target) in enumerate(combined_dataloader_ann_sec):
                    data, target = data.to(DEVICE), target.to(DEVICE)

                    optimizer_orig.zero_grad()
                    output = model_orig(data)
                    loss = loss_fn(output, target)


                    if torch.isnan(loss).any() or torch.isinf(loss).any():
                        print(f"NaN or Inf loss detected at batch {batch_idx}. Loss: {loss}")
                        break


                    loss.backward()
                    optimizer_orig.step()

                    running_loss += loss.item()
                    _, predicted = torch.max(output, 1)
                    total += target.size(0)
                    correct += (predicted == target).sum().item()

                    if batch_idx % 10 == 0:
                        print(f"Batch {batch_idx}/{len(combined_dataloader_ann_sec)}, Loss: {loss.item():.4f}")

                average_loss = running_loss / len(combined_dataloader_ann_sec)
                accuracy = (correct / total) * 100

                print(f"Epoch [{epoch + 1}/{num_epochs}] Completed, Average Loss: {average_loss:.4f}, Accuracy: {accuracy:.2f}%")


            if train_manip is None:
                train_manip = combined_dataset_ann_sec
            else:
                train_manip = ConcatDataset([train_manip, combined_dataset_ann_sec])

            A_mu, A_stddev = weightwmean_best, weightwstddev_best
            game_iter = game_iter + 1
            payoff_curr = payoff_best
            error_curr = error_best
            if game_iter == 50:
                exitgame = True

/kaggle/input\mnist-dataset\t10k-images-idx3-ubyte\t10k-images-idx3-ubyte
/kaggle/input\mnist-dataset\t10k-images-idx3-ubyte\t10k-images-idx3-ubyte.gz
/kaggle/input\mnist-dataset\t10k-labels-idx1-ubyte\t10k-labels-idx1-ubyte
/kaggle/input\mnist-dataset\t10k-labels-idx1-ubyte\t10k-labels-idx1-ubyte.gz
/kaggle/input\mnist-dataset\train-images-idx3-ubyte\train-images-idx3-ubyte
/kaggle/input\mnist-dataset\train-images-idx3-ubyte\train-images-idx3-ubyte.gz
/kaggle/input\mnist-dataset\train-labels-idx1-ubyte\train-labels-idx1-ubyte
/kaggle/input\mnist-dataset\train-labels-idx1-ubyte\train-labels-idx1-ubyte.gz
Using device: cpu
Training set size: 9415
Validation set size: 2354
Test set size: 1932
Tensor representation:
tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.

In [2]:
# full_quantum_game_qiskit2_merged.py
import os
import time
import hashlib
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import confusion_matrix

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Prefer Aer Estimator (faster) if installed, else fallback to StatevectorEstimator
try:
    from qiskit_aer.primitives import Estimator as AerEstimator
    EstimatorClass = AerEstimator
    print("Using qiskit_aer Estimator (fast).")
except Exception:
    from qiskit.primitives import StatevectorEstimator as EstimatorClass
    print("Using qiskit.primitives.StatevectorEstimator (fallback).")

# -------------------------
# User-tunable settings
# -------------------------
latent_qubits = 8    # Option B default (6-10 recommended)
batch_size = 64
seed = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch device:", device)

random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

# -------------------------
# MNIST loader (torchvision preferred)
# -------------------------
def load_mnist_filtered(label1=6, label2=8, batch_size=64):
    try:
        from torchvision import datasets, transforms
        transform = transforms.Compose([transforms.ToTensor()])
        train_full = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
        test_full  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
        train_images = train_full.data.unsqueeze(1).float() / 255.0
        train_labels = train_full.targets.long()
        test_images = test_full.data.unsqueeze(1).float() / 255.0
        test_labels = test_full.targets.long()
        print("Loaded MNIST via torchvision.")
    except Exception as e:
        print("torchvision failed, falling back to sklearn.fetch_openml. Error:", e)
        from sklearn.datasets import fetch_openml
        mn = fetch_openml('mnist_784', version=1, as_frame=False)
        X = mn['data'].astype('float32') / 255.0
        y = mn['target'].astype(int)
        X = X.reshape(-1, 1, 28, 28)
        train_images, test_images = X[:60000], X[60000:]
        train_labels, test_labels = y[:60000], y[60000:]
        train_images = torch.from_numpy(train_images)
        train_labels = torch.from_numpy(train_labels).long()
        test_images = torch.from_numpy(test_images)
        test_labels = torch.from_numpy(test_labels).long()
        print("Loaded MNIST via sklearn.fetch_openml.")

    # filter digits and relabel label1->0, label2->1
    mask_train = (train_labels == label1) | (train_labels == label2)
    mask_test  = (test_labels  == label1) | (test_labels  == label2)

    train_images = train_images[mask_train]
    train_labels = train_labels[mask_train]
    test_images  = test_images[mask_test]
    test_labels  = test_labels[mask_test]

    train_labels = torch.where(train_labels == label1, torch.tensor(0, dtype=torch.long), torch.tensor(1, dtype=torch.long))
    test_labels  = torch.where(test_labels  == label1, torch.tensor(0, dtype=torch.long), torch.tensor(1, dtype=torch.long))

    total_train = TensorDataset(train_images, train_labels)
    train_size = int(0.8 * len(total_train))
    val_size = len(total_train) - train_size
    train_dataset, val_dataset = random_split(total_train, [train_size, val_size])
    test_dataset = TensorDataset(test_images, test_labels)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print("Prepared loaders: Train", len(train_dataset), "Val", len(val_dataset), "Test", len(test_dataset))
    return train_loader, val_loader, test_loader, train_images, train_labels, test_images, test_labels

train_loader, val_loader, test_loader, train_images_all, train_labels_all, test_images_all, test_labels_all = load_mnist_filtered(batch_size=batch_size)

# -------------------------
# Classical encoder (feature extractor) and decoder
# -------------------------
input_dim = 28 * 28

class ClassicalFeatureExtractor(nn.Module):
    def __init__(self, latent_qubits):
        super().__init__()
        self.fc = nn.Linear(input_dim, latent_qubits)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return torch.tanh(self.fc(x))  # map to [-1,1]

class ClassicalDecoder(nn.Module):
    def __init__(self, latent_qubits):
        super().__init__()
        self.fc1 = nn.Linear(latent_qubits, 128)
        self.fc2 = nn.Linear(128, input_dim)
    def forward(self, z):
        z = F.relu(self.fc1(z))
        out = torch.sigmoid(self.fc2(z))
        out = out.view(-1,1,28,28)
        return out

feature_extractor = ClassicalFeatureExtractor(latent_qubits).to(device)
classical_decoder = ClassicalDecoder(latent_qubits).to(device)

# -------------------------
# Circuit Caching utilities
# -------------------------
CIRCUIT_CACHE = {}
def _params_to_list(params):
    # handle numpy arrays / lists
    if isinstance(params, np.ndarray):
        return params.flatten().tolist()
    elif isinstance(params, (list, tuple)):
        return np.array(params).flatten().tolist()
    else:
        return np.array(params).flatten().tolist()

def cache_key(type_name, params, angles):
    p_list = _params_to_list(params)
    a_list = np.round(np.array(angles).flatten(), 6).tolist()
    raw = f"{type_name}|{p_list}|{a_list}"
    return hashlib.sha256(raw.encode()).hexdigest()

def get_encoder_circuit_cached(n_qubits, encoder_params, x_angles):
    key = cache_key("enc", encoder_params, x_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()  # return copy to avoid accidental mutation
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(x_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(encoder_params[i,0]), i)
        qc.rz(float(encoder_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

def get_decoder_circuit_cached(n_qubits, decoder_params, z_angles):
    key = cache_key("dec", decoder_params, z_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(z_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(decoder_params[i,0]), i)
        qc.rz(float(decoder_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

def get_classifier_circuit_cached(n_qubits, clf_params, x_angles):
    key = cache_key("clf", clf_params, x_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(x_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(clf_params[i,0]), i)
        qc.rz(float(clf_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

# -------------------------
# Observables: Z on each qubit and grouped
# -------------------------
z_ops = []
for i in range(latent_qubits):
    s = ['I'] * latent_qubits
    s[latent_qubits - 1 - i] = 'Z'   # consistent ordering
    z_ops.append(SparsePauliOp(''.join(s)))

# Group all Z into single observable H = sum_i Z_i
Z_GROUPED = z_ops[0]
for i in range(1, latent_qubits):
    Z_GROUPED = Z_GROUPED + z_ops[i]

# -------------------------
# Estimator instance
# -------------------------
def create_estimator():
    try:
        # Use Aer estimator if available (best)
        if EstimatorClass.__name__ == "Estimator":
            return EstimatorClass()  # qiskit_aer.Estimator
        else:
            return EstimatorClass()  # StatevectorEstimator
    except Exception:
        return EstimatorClass()

estimator = create_estimator()

# -------------------------
# Optimized measure_z_expectations_batch
#   - one grouped call, then one call per Z_i (but circuits are cached)
# -------------------------
def measure_z_expectations_batch(qc_list):
    """
    Input: qc_list: list of QuantumCircuit (length = n_circuits)
    Output: numpy array shape (n_circuits, latent_qubits)
    Strategy:
      - Make single grouped call (H = sum Z_i) to warm / cache simulator state
      - Then run one Estimator call per qubit (observable = Z_i repeated for all circuits)
        to retrieve each Z_i expectation; this is much faster than n_circuits * n_qubits calls.
    """
    if len(qc_list) == 0:
        return np.zeros((0, latent_qubits), dtype=float)

    n_circuits = len(qc_list)
    # 1) grouped call (warm/caching effect)
    grouped_obs_list = [Z_GROUPED] * n_circuits
    job = estimator.run(circuits=qc_list, observables=grouped_obs_list)
    _ = job.result()  # we don't use sum directly, but call warms internal state

    # 2) per-qubit calls (one call per qubit, each returns n_circuits values)
    values = np.zeros((n_circuits, latent_qubits), dtype=float)
    for q in range(latent_qubits):
        obs = z_ops[q]
        obs_list = [obs] * n_circuits
        job_q = estimator.run(circuits=qc_list, observables=obs_list)
        res_q = job_q.result()
        vals_q = np.array(res_q.values).reshape(-1)  # shape (n_circuits,)
        values[:, q] = vals_q

    return values

# -------------------------
# Quantum VAE (Estimator-based)
# -------------------------
class QuantumVAE:
    def __init__(self, n_qubits):
        self.n_qubits = n_qubits
        # small random init
        self.encoder_params = np.random.randn(n_qubits, 2) * 0.05
        self.decoder_params = np.random.randn(n_qubits, 2) * 0.05

    def encode(self, x_batch, angles_cache=None):
        x_batch = x_batch.detach().cpu()
        b = x_batch.size(0)
        if angles_cache is None:
            feats = feature_extractor(x_batch.to(device)).detach().cpu().numpy()  # (b, latent_qubits)
            angles = (feats + 1.0) * (np.pi / 2.0)
        else:
            angles = angles_cache

        qc_list = [get_encoder_circuit_cached(self.n_qubits, self.encoder_params, angles[i]) for i in range(b)]
        vals = measure_z_expectations_batch(qc_list)  # (b, n_qubits)
        mu_angles = np.arcsin(np.clip(vals, -1.0, 1.0))
        mu = torch.tensor(mu_angles, dtype=torch.float32).to(device)
        logvar = torch.zeros_like(mu).to(device)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        eps = torch.randn_like(mu)
        return mu + eps * torch.exp(0.5 * logvar)

    def decode(self, z_batch):
        z_np = z_batch.detach().cpu().numpy()
        qc_list = [get_decoder_circuit_cached(self.n_qubits, self.decoder_params, z_np[i]) for i in range(z_np.shape[0])]
        vals = measure_z_expectations_batch(qc_list)
        # map [-1,1] compressed to tensor
        return torch.tensor(vals, dtype=torch.float32).to(device)

    def forward(self, x_batch, angles_cache=None):
        mu, logvar = self.encode(x_batch, angles_cache)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

qvae = QuantumVAE(latent_qubits)

# -------------------------
# QuantumCNN wrapper (quantum features -> classical head)
# -------------------------
class QuantumCNN(nn.Module):
    def __init__(self, n_qubits, n_classes=2):
        super().__init__()
        self.n_qubits = n_qubits
        self.clf_params = np.random.randn(n_qubits, 2) * 0.05
        self.fc = nn.Linear(n_qubits, n_classes).to(device)

    def quantum_features_batch(self, x_batch, angles_cache=None):
        x_batch = x_batch.detach().cpu()
        b = x_batch.size(0)
        if angles_cache is None:
            feats = feature_extractor(x_batch.to(device)).detach().cpu().numpy()
            angles = (feats + 1.0) * (np.pi / 2.0)
        else:
            angles = angles_cache
        qc_list = [get_classifier_circuit_cached(self.n_qubits, self.clf_params, angles[i]) for i in range(b)]
        vals = measure_z_expectations_batch(qc_list)  # (b, n_qubits)
        return torch.tensor(vals, dtype=torch.float32).to(device)

    def forward(self, x_batch, angles_cache=None):
        feats_q = self.quantum_features_batch(x_batch, angles_cache)
        logits = self.fc(feats_q)
        return logits

qc_model = QuantumCNN(latent_qubits)

# -------------------------
# Utility metrics
# -------------------------
def test_model(loader, model):
    model.eval()
    all_targets = []
    all_preds = []
    with torch.no_grad():
        for data, targets in loader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs, 1)
            all_targets.extend(targets.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
    cm = confusion_matrix(all_targets, all_preds, labels=[0,1])
    if cm.size == 4:
        tp0, fn0 = cm[0,0], cm[0,1]
        tp1, fn1 = cm[1,1], cm[1,0]
    else:
        tp0 = fn0 = tp1 = fn1 = 0
    tpr0 = tp0 / (tp0 + fn0) if (tp0 + fn0) > 0 else 0.0
    tpr1 = tp1 / (tp1 + fn1) if (tp1 + fn1) > 0 else 0.0
    acc = (np.array(all_preds) == np.array(all_targets)).mean() if len(all_targets) else 0.0
    print(f"Acc: {acc:.4f}, TPR0: {tpr0:.4f}, TPR1: {tpr1:.4f}")
    return tpr0, tpr1

# -------------------------
# Adversarial latent functions
# -------------------------
def quantum_adversarial_latent(mu_pos, logvar_pos, A_mu, A_std):
    if isinstance(A_mu, np.ndarray):
        A_mu = torch.tensor(A_mu, dtype=torch.float32).to(device)
    if isinstance(A_std, np.ndarray):
        A_std = torch.tensor(A_std, dtype=torch.float32).to(device)
    mu_adv = mu_pos + A_mu
    logvar_adv = logvar_pos + A_std
    eps = torch.randn_like(mu_adv)
    z_adv = mu_adv + eps * torch.exp(0.5 * logvar_adv)
    return z_adv, mu_adv, logvar_adv

def calc_cost(a_mean, a_std):
    a_mean_t = torch.tensor(a_mean, dtype=torch.float32) if not isinstance(a_mean, torch.Tensor) else a_mean
    a_std_t = torch.tensor(a_std, dtype=torch.float32) if not isinstance(a_std, torch.Tensor) else a_std
    norm_mean = torch.norm(a_mean_t)
    norm_std = torch.norm(a_std_t)
    total_cost = (norm_mean / latent_qubits) + (norm_std / latent_qubits)
    return total_cost.item()

def adversary_payoff(posnegdata, posindices, negindices, mu_pos, std_pos, qvae_obj, a_mean, a_std, posnegtargets, model):
    # If mu_pos/std_pos are 1D (avg), make them 2D
    if mu_pos.dim() == 1:
        mu_pos = mu_pos.unsqueeze(0)
    if std_pos.dim() == 1:
        std_pos = std_pos.unsqueeze(0)
    z_adv, _, _ = quantum_adversarial_latent(mu_pos, std_pos, a_mean, a_std)
    # decode to images (optional)
    recon_images = classical_decoder(z_adv.to(device))
    # classify using classical head on latent z_adv (fast)
    logits = model.fc(z_adv.to(device))
    _, preds = torch.max(logits, 1)
    targets = torch.tensor(posnegtargets, dtype=torch.long).to(device)
    true_positives = ((preds == 1) & (targets == 1)).sum().item()
    false_negatives = ((preds == 0) & (targets == 1)).sum().item()
    tpr1 = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0.0
    error = 1.0 - tpr1
    cost = calc_cost(a_mean, a_std)
    payoff = 1.0 + error - cost
    return payoff

# -------------------------
# EMPSO optimizer (simplified)
# -------------------------
class Particle:
    def __init__(self, bounds):
        self.position = np.random.uniform(bounds[:,0], bounds[:,1])
        self.velocity = np.zeros_like(self.position)
        self.momentum = np.zeros_like(self.position)
        self.pbest_position = self.position.copy()
        self.pbest_value = float('inf')
    def update_personal_best(self, fitness):
        if fitness < self.pbest_value:
            self.pbest_value = fitness
            self.pbest_position = self.position.copy()

class EMPSO:
    def __init__(self, fitness_fn, bounds, num_particles=12, beta=0.7, c1=1.5, c2=1.5, max_iter=20):
        self.fitness_fn = fitness_fn
        self.bounds = bounds
        self.num_particles = num_particles
        self.beta = beta
        self.c1 = c1
        self.c2 = c2
        self.max_iter = max_iter
        self.swarm = [Particle(bounds) for _ in range(num_particles)]
        self.gbest_position = None
        self.gbest_value = float('inf')
    def optimize(self):
        for it in range(self.max_iter):
            for p in self.swarm:
                fitness = self.fitness_fn(p.position)
                p.update_personal_best(fitness)
                if fitness < self.gbest_value:
                    self.gbest_value = fitness
                    self.gbest_position = p.position.copy()
            for p in self.swarm:
                r1 = np.random.rand(len(p.position))
                r2 = np.random.rand(len(p.position))
                p.velocity = (self.beta * p.momentum +
                              (1 - self.beta) * p.velocity +
                              self.c1 * r1 * (p.pbest_position - p.position) +
                              self.c2 * r2 * (self.gbest_position - p.position))
                p.position += p.velocity
                p.momentum = self.beta * p.momentum + (1 - self.beta) * p.velocity
                p.position = np.clip(p.position, self.bounds[:,0], self.bounds[:,1])
        return self.gbest_position, self.gbest_value

# -------------------------
# Alternating least-squares adversarial game
# -------------------------
def alternating_least_squares_game(qc_model, qvae_obj, train_images_all, train_labels_all,
                                   max_game_iter=2, pso_iter=8, n_pos_samples=80):
    imgs = train_images_all
    labs = train_labels_all
    pos_mask = (labs == 1)
    neg_mask = (labs == 0)
    pos_imgs = imgs[pos_mask][:n_pos_samples].to(device)
    pos_labs = labs[pos_mask][:n_pos_samples].to(device)
    neg_imgs = imgs[neg_mask][:n_pos_samples].to(device)
    neg_labs = labs[neg_mask][:n_pos_samples].to(device)

    combined = torch.cat([pos_imgs, neg_imgs], dim=0)
    combined_targets = torch.cat([pos_labs, neg_labs], dim=0)
    pos_indices = (combined_targets.cpu().numpy() == 1).tolist()
    neg_indices = (combined_targets.cpu().numpy() == 0).tolist()

    A_mu = np.zeros(latent_qubits)
    A_std = np.zeros(latent_qubits)
    payoff_curr = -np.inf

    for game_iter in range(max_game_iter):
        # average latent for positives
        with torch.no_grad():
            mu_pos_all, logvar_pos_all = qvae_obj.encode(pos_imgs)
        mu_pos_avg = mu_pos_all.mean(dim=0)
        std_pos_avg = logvar_pos_all.mean(dim=0)

        # PSO bounds
        D = latent_qubits
        bounds = np.vstack([np.full(D, -0.5), np.full(D, 0.5)]).T

        # optimize mean
        def fit_mean(alpha_vec):
            return -adversary_payoff(combined, pos_indices, neg_indices, mu_pos_avg, std_pos_avg, qvae_obj, alpha_vec, A_std, combined_targets.cpu().numpy(), qc_model)

        em_mean = EMPSO(fit_mean, bounds, num_particles=10, max_iter=pso_iter)
        alpha_star_mean, _ = em_mean.optimize()

        # optimize std
        def fit_std(alpha_vec):
            return -adversary_payoff(combined, pos_indices, neg_indices, mu_pos_avg, std_pos_avg, qvae_obj, alpha_star_mean, alpha_vec, combined_targets.cpu().numpy(), qc_model)

        em_std = EMPSO(fit_std, bounds, num_particles=10, max_iter=pso_iter)
        alpha_star_std, _ = em_std.optimize()

        payoff_new = adversary_payoff(combined, pos_indices, neg_indices, mu_pos_avg, std_pos_avg, qvae_obj, alpha_star_mean, alpha_star_std, combined_targets.cpu().numpy(), qc_model)
        print(f"Game {game_iter}: payoff_curr={payoff_curr:.4f} payoff_new={payoff_new:.4f}")

        if payoff_new > payoff_curr:
            payoff_curr = payoff_new
            A_mu = alpha_star_mean.copy()
            A_std = alpha_star_std.copy()

            # build adversarial images from pos set
            z_adv, _, _ = quantum_adversarial_latent(mu_pos_all, logvar_pos_all, A_mu, A_std)
            adv_images = classical_decoder(z_adv.to(device))
            adv_targets = torch.ones(adv_images.size(0), dtype=torch.long).to(device)

            # retrain classical head with small subset + adv images
            subset_size = min(500, len(train_images_all))
            train_data_subset = train_images_all[:subset_size].to(device)
            train_targets_subset = train_labels_all[:subset_size].to(device)
            combined_images = torch.cat([train_data_subset, adv_images], dim=0)
            combined_targets2 = torch.cat([train_targets_subset, adv_targets], dim=0)
            combined_ds = TensorDataset(combined_images, combined_targets2)
            combined_loader = DataLoader(combined_ds, batch_size=32, shuffle=True)

            qc_model.train()
            optimizer = torch.optim.Adam(qc_model.fc.parameters(), lr=1e-3)
            loss_fn = nn.CrossEntropyLoss()
            for epoch in range(3):
                run_loss = 0.0
                for bimgs, btargets in combined_loader:
                    bimgs = bimgs.to(device); btargets = btargets.to(device)
                    with torch.no_grad():
                        mu_b, logvar_b = qvae_obj.encode(bimgs)
                        z_b = qvae_obj.reparameterize(mu_b, logvar_b)
                    logits = qc_model.fc(z_b.to(device))
                    loss = loss_fn(logits, btargets)
                    optimizer.zero_grad(); loss.backward(); optimizer.step()
                    run_loss += loss.item()
                print(f" Retrain epoch {epoch+1}, loss {run_loss/len(combined_loader):.4f}")
        else:
            print("No improvement — stopping.")
            break

    return A_mu, A_std, payoff_curr

# -------------------------
# Quick demo / run
# -------------------------
if __name__ == "__main__":
    print("Starting demo adversarial game (this will run several Estimator calls).")
    start = time.time()
    A_mu_final, A_std_final, final_payoff = alternating_least_squares_game(qc_model, qvae, train_images_all, train_labels_all,
                                                                           max_game_iter=2, pso_iter=6, n_pos_samples=60)
    print("Game done in {:.1f}s".format(time.time() - start))
    print("A_mu:", A_mu_final)
    print("A_std:", A_std_final)
    print("Final payoff:", final_payoff)

    print("Evaluating classifier on test:")
    test_model(test_loader, qc_model)

    # Save artifacts
    np.save("qvae_encoder_params.npy", qvae.encoder_params)
    np.save("qvae_decoder_params.npy", qvae.decoder_params)
    np.save("qcnn_clf_params.npy", qc_model.clf_params)
    torch.save(qc_model.fc.state_dict(), "qcnn_fc_head.pth")
    print("Params saved. Done.")


Using qiskit_aer Estimator (fast).
Torch device: cpu
Loaded MNIST via torchvision.
Prepared loaders: Train 9415 Val 2354 Test 1932
Starting demo adversarial game (this will run several Estimator calls).


C:\Users\Aaditya Rajput\AppData\Local\Temp\ipykernel_68000\1766341069.py:210: DeprecationWarning: Estimator has been deprecated as of Aer 0.15, please use EstimatorV2 instead.
  estimator = create_estimator()
C:\Users\Aaditya Rajput\AppData\Local\Temp\ipykernel_68000\1766341069.py:210: DeprecationWarning: Option approximation=False is deprecated as of qiskit-aer 0.13. It will be removed no earlier than 3 months after the release date. Instead, use BackendEstimator from qiskit.primitives.
  estimator = create_estimator()


Game 0: payoff_curr=-inf payoff_new=0.8930
 Retrain epoch 1, loss 0.7818
 Retrain epoch 2, loss 0.7620
 Retrain epoch 3, loss 0.7411
Game 1: payoff_curr=0.8930 payoff_new=1.9105
 Retrain epoch 1, loss 0.7453
 Retrain epoch 2, loss 0.7649
 Retrain epoch 3, loss 0.7608
Game done in 354.2s
A_mu: [ 0.00906552  0.07020369 -0.07072749 -0.01074888 -0.10662028  0.39572948
 -0.09350516 -0.11875553]
A_std: [ 0.09245658 -0.0025336  -0.00305625  0.0079356   0.17187662 -0.07383863
 -0.06158765 -0.15513694]
Final payoff: 1.9105465114116669
Evaluating classifier on test:
Acc: 0.4959, TPR0: 1.0000, TPR1: 0.0000
Params saved. Done.


In [1]:
import qiskit
print("Qiskit version:", qiskit.__version__)

import pkgutil
import qiskit.primitives
print("Available primitives:", [m.name for m in pkgutil.iter_modules(qiskit.primitives.__path__)])


Qiskit version: 2.2.3
Available primitives: ['backend_estimator_v2', 'backend_sampler_v2', 'base', 'containers', 'primitive_job', 'statevector_estimator', 'statevector_sampler', 'utils']


In [ ]:
# =========================
# Hard defense + quantum param-shift fine-tuning
# =========================

import math

def compute_features_and_grads_batch_for_clf_params(x_batch, clf_params_np, n_qubits, estimator,
                                                    get_classifier_circuit_cached, measure_z_expectations_batch):
    """
    Compute quantum features for the batch using clf_params_np (numpy array shape (n_qubits,2)).
    Returns:
      feats_tensor: torch.Tensor (batch, n_qubits) - features (expectations)
    """
    # Build angle encodings from feature_extractor (we keep same mapping as pipeline)
    with torch.no_grad():
        feats_classical = feature_extractor(x_batch.to(device)).detach().cpu().numpy()  # (B, D) in [-1,1]
    angles = (feats_classical + 1.0) * (np.pi / 2.0)  # (B, D)

    # Build circuit list using clf_params_np
    qc_list = [get_classifier_circuit_cached(n_qubits, clf_params_np, angles[i]) for i in range(angles.shape[0])]
    vals = measure_z_expectations_batch(qc_list)  # numpy (B, D)
    feats_tensor = torch.tensor(vals, dtype=torch.float32).to(device)  # return on device for autograd usage
    return feats_tensor, angles  # angles returned for reuse in shifted circuits

def parameter_shift_update(clf_params_np, x_batch, batch_labels, n_qubits,
                           qc_model, lr_q=1e-2, shift=np.pi/2):
    """
    Perform one parameter-shift update for clf_params_np using batch x_batch and batch_labels.
    - clf_params_np: numpy array shape (n_qubits,2)
    Returns updated clf_params_np (numpy).
    """
    # 1) compute baseline features with current parameters (feats) and obtain ∂L/∂feats via autograd
    feats_base, angles = compute_features_and_grads_batch_for_clf_params(x_batch, clf_params_np, n_qubits,
                                                                         estimator, get_classifier_circuit_cached, measure_z_expectations_batch)
    feats_base = feats_base.clone().detach().requires_grad_(True)  # require grad to compute dL/dfeat
    logits = qc_model.fc(feats_base)
    loss_fn = nn.CrossEntropyLoss()
    loss = loss_fn(logits, batch_labels.to(device))
    # backprop to get dL/dfeatures
    qc_model.fc.zero_grad()
    if feats_base.grad is not None:
        feats_base.grad.zero_()
    loss.backward(retain_graph=True)
    # feats_base.grad now holds dL/dfeature (on device)
    dL_dfeat = feats_base.grad.detach().cpu().numpy()  # shape (B, n_qubits)

    B = dL_dfeat.shape[0]
    # 2) For each parameter, compute parameter-shifted features and gradient
    # clf_params_np shape (n_qubits, 2) -> param index mapping: (q,0)=ry, (q,1)=rz
    grads_theta = np.zeros_like(clf_params_np, dtype=float)  # accumulate dL/dtheta

    # To reduce Estimator overhead we compute shifted features for all circuits per parameter
    # For each parameter p:
    for q in range(n_qubits):
        for p_idx in range(2):
            # create shifted parameter arrays
            clf_plus = clf_params_np.copy()
            clf_minus = clf_params_np.copy()
            clf_plus[q, p_idx] += shift
            clf_minus[q, p_idx] -= shift

            # compute features f_plus, f_minus (numpy arrays)
            f_plus_tensor, _ = compute_features_and_grads_batch_for_clf_params(x_batch, clf_plus, n_qubits,
                                                                               estimator, get_classifier_circuit_cached, measure_z_expectations_batch)
            f_minus_tensor, _ = compute_features_and_grads_batch_for_clf_params(x_batch, clf_minus, n_qubits,
                                                                                estimator, get_classifier_circuit_cached, measure_z_expectations_batch)
            f_plus = f_plus_tensor.detach().cpu().numpy()  # (B, D)
            f_minus = f_minus_tensor.detach().cpu().numpy()

            # parameter-shift derivative (per feature dimension): 0.5*(f_plus - f_minus)
            df_dtheta = 0.5 * (f_plus - f_minus)  # (B, D)

            # Chain rule: dL/dtheta = sum_over_batch sum_over_features (dL/dfeat * df/dtheta)
            # dL_dfeat: (B, D)
            term = np.sum(dL_dfeat * df_dtheta)  # scalar
            grads_theta[q, p_idx] = term / float(B)  # normalize by batch size

    # 3) Gradient descent step on clf_params
    clf_params_np = clf_params_np - lr_q * grads_theta

    return clf_params_np, loss.item()

# -------------------------
# Hard defense training with quantum fine-tuning (parameter-shift)
# -------------------------
def hard_defense_with_quantum_finetune(qc_model, qvae, feature_extractor, classical_decoder,
                                       train_images_all, train_labels_all,
                                       val_loader=None,
                                       epochs=6,
                                       adv_samples_per_pos=8,
                                       adv_radius=0.5,
                                       batch_size_def=64,
                                       collapse_penalty_coef=12.0,
                                       lr=5e-4,
                                       subset_size=3000,
                                       fine_tune_quantum=True,
                                       lr_q=1e-2,
                                       quantum_batches_per_epoch=4,
                                       paramshift_shift=np.pi/2):
    """
    Hard defense + quantum parameter-shift fine-tuning.
    - fine_tune_quantum: whether to run parameter-shift updates on qc_model.clf_params (numpy).
    - quantum_batches_per_epoch: number of minibatches per epoch to perform expensive param-shift updates (keeps cost bounded).
    """

    feature_extractor.train()
    qc_model.train()

    params = list(feature_extractor.parameters()) + list(qc_model.fc.parameters())
    optimizer = torch.optim.Adam(params, lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    device_local = device

    # Prepare training subsets
    total_clean = min(subset_size, len(train_images_all))
    clean_images = train_images_all[:total_clean].to(device_local)
    clean_labels = train_labels_all[:total_clean].to(device_local)

    pos_mask = (train_labels_all == 1)
    pos_imgs_all = train_images_all[pos_mask]
    if len(pos_imgs_all) == 0:
        raise RuntimeError("No positive-class examples found for adversarial generation.")
    pos_sample_count = min(200, len(pos_imgs_all))
    pos_imgs = pos_imgs_all[:pos_sample_count].to(device_local)

    # Convert quantum params to numpy for in-place updates
    clf_params_np = qc_model.clf_params.copy()

    for epoch in range(epochs):
        epoch_loss = 0.0
        n_batches = 0

        # Generate adversarial latent variants
        with torch.no_grad():
            mu_pos_all, logvar_pos_all = qvae.encode(pos_imgs)  # (P, D)
            mu_pos_rep = mu_pos_all.repeat_interleave(adv_samples_per_pos, dim=0)
            logvar_rep = logvar_pos_all.repeat_interleave(adv_samples_per_pos, dim=0)
            delta = torch.randn_like(mu_pos_rep, device=device_local) * adv_radius
            std_delta = torch.randn_like(logvar_rep, device=device_local) * (adv_radius * 0.3)
            mu_adv = mu_pos_rep + delta
            std_adv = torch.clamp(logvar_rep + std_delta, min=-3.0, max=3.0)
            eps = torch.randn_like(mu_adv)
            z_adv = mu_adv + eps * torch.exp(0.5 * std_adv)
            adv_images = classical_decoder(z_adv.to(device_local))
            adv_labels = torch.ones(adv_images.size(0), dtype=torch.long, device=device_local)

        # Build combined dataset: mix clean & adv
        idxs = torch.randperm(clean_images.size(0))
        clean_shuffled = clean_images[idxs]
        clean_shuffled_labels = clean_labels[idxs]
        combined_images = torch.cat([clean_shuffled, adv_images], dim=0)
        combined_labels = torch.cat([clean_shuffled_labels, adv_labels], dim=0)

        combined_ds = TensorDataset(combined_images, combined_labels)
        combined_loader = DataLoader(combined_ds, batch_size=batch_size_def, shuffle=True)

        # For controlling expensive quantum param updates we track how many have been done this epoch
        quantum_updates_done = 0

        for batch_idx, (batch_imgs, batch_labels) in enumerate(combined_loader):
            batch_imgs = batch_imgs.to(device_local)
            batch_labels = batch_labels.to(device_local)

            # Classical forward (trainable feature extractor)
            feats = feature_extractor(batch_imgs)  # (B, D) tensor on device
            # logits via current classical head
            logits = qc_model.fc(feats)
            ce_loss = loss_fn(logits, batch_labels)

            # collapse penalty
            probs1 = torch.softmax(logits, dim=1)[:, 1]
            mean_p1 = probs1.mean()
            collapse_penalty = collapse_penalty_coef * (mean_p1 - 0.5) ** 2

            # l2 reg
            l2_reg = 0.0
            for p in params:
                l2_reg += 1e-4 * torch.sum(p ** 2)

            loss = ce_loss + collapse_penalty + l2_reg

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

            # Quantum parameter-shift updates (expensive) on selected minibatches
            if fine_tune_quantum and (quantum_updates_done < quantum_batches_per_epoch):
                # We'll run parameter-shift on this minibatch
                try:
                    # Compute gradient w.r.t. quantum params (returns updated clf_params_np and the loss value)
                    clf_params_np, qloss = parameter_shift_update(clf_params_np, batch_imgs, batch_labels, latent_qubits,
                                                                  qc_model, lr_q=lr_q, shift=paramshift_shift)
                    # attach updated params back to qc_model
                    qc_model.clf_params = clf_params_np.copy()
                    quantum_updates_done += 1
                    # optional print
                    # print(f" Quantum param-shift update #{quantum_updates_done}, q_loss={qloss:.4f}")
                except Exception as e:
                    print("Parameter-shift update failed on minibatch:", e)

        avg_loss = epoch_loss / max(1, n_batches)
        print(f"[HardDef+QEpoch {epoch+1}/{epochs}] Avg loss: {avg_loss:.4f}, quantum_updates: {quantum_updates_done}")

        if val_loader is not None:
            print("Validation evaluation after epoch:")
            test_model(val_loader, qc_model)

    # Save back clf_params_np
    qc_model.clf_params = clf_params_np.copy()
    print("Hard defense + quantum fine-tune complete.")

# -------------------------
# Example call to run the hard defense + quantum finetune
# -------------------------
# Tune hyperparameters depending on CPU/time constraints.
hard_defense_with_quantum_finetune(qc_model, qvae, feature_extractor, classical_decoder,
                                   train_images_all, train_labels_all,
                                   val_loader=val_loader,
                                   epochs=4,
                                   adv_samples_per_pos=8,
                                   adv_radius=0.5,
                                   batch_size_def=64,
                                   collapse_penalty_coef=12.0,
                                   lr=5e-4,
                                   subset_size=3000,
                                   fine_tune_quantum=True,
                                   lr_q=1e-2,
                                   quantum_batches_per_epoch=3,
                                   paramshift_shift=math.pi/2)

# After training evaluate:
print("Final evaluation on test set:")
test_model(test_loader, qc_model)


In [1]:
# final_quantum_adversarial_game.py
import os, time, math, hashlib, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import confusion_matrix

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Prefer Aer Estimator (fast CPU) if installed; otherwise fallback to StatevectorEstimator
try:
    from qiskit_aer.primitives import Estimator as AerEstimator
    EstimatorClass = AerEstimator
    print("Using qiskit_aer Estimator (fast).")
except Exception:
    from qiskit.primitives import StatevectorEstimator as EstimatorClass
    print("Using qiskit.primitives.StatevectorEstimator (fallback).")

# ------------------------------
# Settings (tune these)
# ------------------------------
latent_qubits = 8    # 6-10 recommended for Option B
batch_size = 64
seed = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

# ------------------------------
# MNIST loader
# ------------------------------
def load_mnist_filtered(label1=6, label2=8, batch_size=64):
    try:
        from torchvision import datasets, transforms
        transform = transforms.Compose([transforms.ToTensor()])
        train_full = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
        test_full  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
        train_images = train_full.data.unsqueeze(1).float() / 255.0
        train_labels = train_full.targets.long()
        test_images = test_full.data.unsqueeze(1).float() / 255.0
        test_labels = test_full.targets.long()
        print("Loaded MNIST via torchvision.")
    except Exception as e:
        print("torchvision failed, falling back to sklearn.fetch_openml. Error:", e)
        from sklearn.datasets import fetch_openml
        mn = fetch_openml('mnist_784', version=1, as_frame=False)
        X = mn['data'].astype('float32') / 255.0
        y = mn['target'].astype(int)
        X = X.reshape(-1, 1, 28, 28)
        train_images, test_images = X[:60000], X[60000:]
        train_labels, test_labels = y[:60000], y[60000:]
        train_images = torch.from_numpy(train_images)
        train_labels = torch.from_numpy(train_labels).long()
        test_images = torch.from_numpy(test_images)
        test_labels = torch.from_numpy(test_labels).long()
        print("Loaded MNIST via sklearn.fetch_openml.")

    mask_train = (train_labels == label1) | (train_labels == label2)
    mask_test  = (test_labels  == label1) | (test_labels  == label2)

    train_images = train_images[mask_train]
    train_labels = train_labels[mask_train]
    test_images  = test_images[mask_test]
    test_labels  = test_labels[mask_test]

    train_labels = torch.where(train_labels == label1, torch.tensor(0, dtype=torch.long), torch.tensor(1, dtype=torch.long))
    test_labels  = torch.where(test_labels  == label1, torch.tensor(0, dtype=torch.long), torch.tensor(1, dtype=torch.long))

    total_train = TensorDataset(train_images, train_labels)
    train_size = int(0.8 * len(total_train))
    val_size = len(total_train) - train_size
    train_dataset, val_dataset = random_split(total_train, [train_size, val_size])
    test_dataset = TensorDataset(test_images, test_labels)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print("Loaders prepared: Train", len(train_dataset), "Val", len(val_dataset), "Test", len(test_dataset))
    return train_loader, val_loader, test_loader, train_images, train_labels, test_images, test_labels

train_loader, val_loader, test_loader, train_images_all, train_labels_all, test_images_all, test_labels_all = load_mnist_filtered(batch_size=batch_size)

# ------------------------------
# Classical feature extractor + decoder
# ------------------------------
input_dim = 28 * 28
class ClassicalFeatureExtractor(nn.Module):
    def __init__(self, latent_qubits):
        super().__init__()
        self.fc = nn.Linear(input_dim, latent_qubits)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return torch.tanh(self.fc(x))
class ClassicalDecoder(nn.Module):
    def __init__(self, latent_qubits):
        super().__init__()
        self.fc1 = nn.Linear(latent_qubits, 128)
        self.fc2 = nn.Linear(128, input_dim)
    def forward(self, z):
        z = F.relu(self.fc1(z))
        out = torch.sigmoid(self.fc2(z))
        out = out.view(-1,1,28,28)
        return out

feature_extractor = ClassicalFeatureExtractor(latent_qubits).to(device)
classical_decoder = ClassicalDecoder(latent_qubits).to(device)

# ------------------------------
# Circuit caching utilities
# ------------------------------
CIRCUIT_CACHE = {}
def _params_to_list(params):
    if isinstance(params, np.ndarray):
        return params.flatten().tolist()
    elif isinstance(params, (list, tuple)):
        return np.array(params).flatten().tolist()
    else:
        return np.array(params).flatten().tolist()

def cache_key(type_name, params, angles):
    p_list = _params_to_list(params)
    a_list = np.round(np.array(angles).flatten(), 6).tolist()
    raw = f"{type_name}|{p_list}|{a_list}"
    return hashlib.sha256(raw.encode()).hexdigest()

def get_encoder_circuit_cached(n_qubits, encoder_params, x_angles):
    key = cache_key("enc", encoder_params, x_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(x_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(encoder_params[i,0]), i)
        qc.rz(float(encoder_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

def get_decoder_circuit_cached(n_qubits, decoder_params, z_angles):
    key = cache_key("dec", decoder_params, z_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(z_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(decoder_params[i,0]), i)
        qc.rz(float(decoder_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

def get_classifier_circuit_cached(n_qubits, clf_params, x_angles):
    key = cache_key("clf", clf_params, x_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(x_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(clf_params[i,0]), i)
        qc.rz(float(clf_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

# ------------------------------
# Observables: Z individual + grouped
# ------------------------------
z_ops = []
for i in range(latent_qubits):
    s = ['I'] * latent_qubits
    s[latent_qubits - 1 - i] = 'Z'
    z_ops.append(SparsePauliOp(''.join(s)))
Z_GROUPED = z_ops[0]
for i in range(1, latent_qubits):
    Z_GROUPED = Z_GROUPED + z_ops[i]

# ------------------------------
# Estimator instance
# ------------------------------
def create_estimator():
    try:
        return EstimatorClass()
    except Exception:
        return EstimatorClass()
estimator = create_estimator()

# ------------------------------
# measure_z_expectations_batch (grouped + per-qubit calls with caching)
# ------------------------------
def measure_z_expectations_batch(qc_list):
    if len(qc_list) == 0:
        return np.zeros((0, latent_qubits), dtype=float)
    n_circuits = len(qc_list)
    # grouped warm call
    grouped_obs_list = [Z_GROUPED] * n_circuits
    job = estimator.run(circuits=qc_list, observables=grouped_obs_list)
    _ = job.result()
    # per-qubit calls (one call per qubit)
    values = np.zeros((n_circuits, latent_qubits), dtype=float)
    for q in range(latent_qubits):
        obs = z_ops[q]
        obs_list = [obs] * n_circuits
        job_q = estimator.run(circuits=qc_list, observables=obs_list)
        res_q = job_q.result()
        vals_q = np.array(res_q.values).reshape(-1)
        values[:, q] = vals_q
    return values

# ------------------------------
# QuantumVAE
# ------------------------------
class QuantumVAE:
    def __init__(self, n_qubits):
        self.n_qubits = n_qubits
        self.encoder_params = np.random.randn(n_qubits, 2) * 0.05
        self.decoder_params = np.random.randn(n_qubits, 2) * 0.05

    def encode(self, x_batch, angles_cache=None):
        x_batch = x_batch.detach().cpu()
        b = x_batch.size(0)
        if angles_cache is None:
            feats = feature_extractor(x_batch.to(device)).detach().cpu().numpy()
            angles = (feats + 1.0) * (np.pi / 2.0)
        else:
            angles = angles_cache
        qc_list = [get_encoder_circuit_cached(self.n_qubits, self.encoder_params, angles[i]) for i in range(b)]
        vals = measure_z_expectations_batch(qc_list)
        mu_angles = np.arcsin(np.clip(vals, -1.0, 1.0))
        mu = torch.tensor(mu_angles, dtype=torch.float32).to(device)
        logvar = torch.zeros_like(mu).to(device)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        eps = torch.randn_like(mu)
        return mu + eps * torch.exp(0.5 * logvar)

    def decode(self, z_batch):
        z_np = z_batch.detach().cpu().numpy()
        qc_list = [get_decoder_circuit_cached(self.n_qubits, self.decoder_params, z_np[i]) for i in range(z_np.shape[0])]
        vals = measure_z_expectations_batch(qc_list)
        return torch.tensor(vals, dtype=torch.float32).to(device)

    def forward(self, x_batch, angles_cache=None):
        mu, logvar = self.encode(x_batch, angles_cache)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

qvae = QuantumVAE(latent_qubits)

# ------------------------------
# QuantumCNN wrapper (quantum features -> classical head)
# ------------------------------
class QuantumCNN(nn.Module):
    def __init__(self, n_qubits, n_classes=2):
        super().__init__()
        self.n_qubits = n_qubits
        self.clf_params = np.random.randn(n_qubits, 2) * 0.05
        self.fc = nn.Linear(n_qubits, n_classes).to(device)

    def quantum_features_batch(self, x_batch, angles_cache=None):
        x_batch = x_batch.detach().cpu()
        b = x_batch.size(0)
        if angles_cache is None:
            feats = feature_extractor(x_batch.to(device)).detach().cpu().numpy()
            angles = (feats + 1.0) * (np.pi / 2.0)
        else:
            angles = angles_cache
        qc_list = [get_classifier_circuit_cached(self.n_qubits, self.clf_params, angles[i]) for i in range(b)]
        vals = measure_z_expectations_batch(qc_list)
        return torch.tensor(vals, dtype=torch.float32).to(device)

    def forward(self, x_batch, angles_cache=None):
        feats_q = self.quantum_features_batch(x_batch, angles_cache)
        logits = self.fc(feats_q)
        return logits

qc_model = QuantumCNN(latent_qubits)

# ------------------------------
# Metrics & test
# ------------------------------
def test_model(loader, model):
    model.eval()
    all_targets = []
    all_preds = []
    with torch.no_grad():
        for data, targets in loader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs, 1)
            all_targets.extend(targets.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
    cm = confusion_matrix(all_targets, all_preds, labels=[0,1])
    if cm.size == 4:
        tp0, fn0 = cm[0,0], cm[0,1]
        tp1, fn1 = cm[1,1], cm[1,0]
    else:
        tp0 = fn0 = tp1 = fn1 = 0
    tpr0 = tp0 / (tp0 + fn0) if (tp0 + fn0) > 0 else 0.0
    tpr1 = tp1 / (tp1 + fn1) if (tp1 + fn1) > 0 else 0.0
    acc = (np.array(all_preds) == np.array(all_targets)).mean() if len(all_targets) else 0.0
    print(f"Acc: {acc:.4f}, TPR0: {tpr0:.4f}, TPR1: {tpr1:.4f}")
    return tpr0, tpr1

# ------------------------------
# Adversarial utilities (latent)
# ------------------------------
def quantum_adversarial_latent(mu_pos, logvar_pos, A_mu, A_std):
    if isinstance(A_mu, np.ndarray):
        A_mu = torch.tensor(A_mu, dtype=torch.float32).to(device)
    if isinstance(A_std, np.ndarray):
        A_std = torch.tensor(A_std, dtype=torch.float32).to(device)
    mu_adv = mu_pos + A_mu
    logvar_adv = logvar_pos + A_std
    eps = torch.randn_like(mu_adv)
    z_adv = mu_adv + eps * torch.exp(0.5 * logvar_adv)
    return z_adv, mu_adv, logvar_adv

def calc_cost(a_mean, a_std):
    a_mean_t = torch.tensor(a_mean, dtype=torch.float32) if not isinstance(a_mean, torch.Tensor) else a_mean
    a_std_t = torch.tensor(a_std, dtype=torch.float32) if not isinstance(a_std, torch.Tensor) else a_std
    norm_mean = torch.norm(a_mean_t)
    norm_std = torch.norm(a_std_t)
    total_cost = (norm_mean / latent_qubits) + (norm_std / latent_qubits)
    return total_cost.item()

def adversary_payoff(posnegdata, posindices, negindices, mu_pos, std_pos, qvae_obj, a_mean, a_std, posnegtargets, model):
    if mu_pos.dim() == 1:
        mu_pos = mu_pos.unsqueeze(0)
    if std_pos.dim() == 1:
        std_pos = std_pos.unsqueeze(0)
    z_adv, _, _ = quantum_adversarial_latent(mu_pos, std_pos, a_mean, a_std)
    recon_images = classical_decoder(z_adv.to(device))
    logits = model.fc(z_adv.to(device))
    _, preds = torch.max(logits, 1)
    targets = torch.tensor(posnegtargets, dtype=torch.long).to(device)
    true_positives = ((preds == 1) & (targets == 1)).sum().item()
    false_negatives = ((preds == 0) & (targets == 1)).sum().item()
    tpr1 = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0.0
    error = 1.0 - tpr1
    cost = calc_cost(a_mean, a_std)
    payoff = 1.0 + error - cost
    return payoff

# ------------------------------
# EMPSO optimizer
# ------------------------------
class Particle:
    def __init__(self, bounds):
        self.position = np.random.uniform(bounds[:,0], bounds[:,1])
        self.velocity = np.zeros_like(self.position)
        self.momentum = np.zeros_like(self.position)
        self.pbest_position = self.position.copy()
        self.pbest_value = float('inf')
    def update_personal_best(self, fitness):
        if fitness < self.pbest_value:
            self.pbest_value = fitness
            self.pbest_position = self.position.copy()

class EMPSO:
    def __init__(self, fitness_fn, bounds, num_particles=12, beta=0.7, c1=1.5, c2=1.5, max_iter=20):
        self.fitness_fn = fitness_fn
        self.bounds = bounds
        self.num_particles = num_particles
        self.beta = beta
        self.c1 = c1
        self.c2 = c2
        self.max_iter = max_iter
        self.swarm = [Particle(bounds) for _ in range(num_particles)]
        self.gbest_position = None
        self.gbest_value = float('inf')
    def optimize(self):
        for it in range(self.max_iter):
            for p in self.swarm:
                fitness = self.fitness_fn(p.position)
                p.update_personal_best(fitness)
                if fitness < self.gbest_value:
                    self.gbest_value = fitness
                    self.gbest_position = p.position.copy()
            for p in self.swarm:
                r1 = np.random.rand(len(p.position))
                r2 = np.random.rand(len(p.position))
                p.velocity = (self.beta * p.momentum +
                              (1 - self.beta) * p.velocity +
                              self.c1 * r1 * (p.pbest_position - p.position) +
                              self.c2 * r2 * (self.gbest_position - p.position))
                p.position += p.velocity
                p.momentum = self.beta * p.momentum + (1 - self.beta) * p.velocity
                p.position = np.clip(p.position, self.bounds[:,0], self.bounds[:,1])
        return self.gbest_position, self.gbest_value

# ------------------------------
# Alternating least squares adversarial game
# ------------------------------
def alternating_least_squares_game(qc_model, qvae_obj, train_images_all, train_labels_all,
                                   max_game_iter=2, pso_iter=8, n_pos_samples=80):
    imgs = train_images_all
    labs = train_labels_all
    pos_mask = (labs == 1)
    neg_mask = (labs == 0)
    pos_imgs = imgs[pos_mask][:n_pos_samples].to(device)
    pos_labs = labs[pos_mask][:n_pos_samples].to(device)
    neg_imgs = imgs[neg_mask][:n_pos_samples].to(device)
    neg_labs = labs[neg_mask][:n_pos_samples].to(device)

    combined = torch.cat([pos_imgs, neg_imgs], dim=0)
    combined_targets = torch.cat([pos_labs, neg_labs], dim=0)
    pos_indices = (combined_targets.cpu().numpy() == 1).tolist()
    neg_indices = (combined_targets.cpu().numpy() == 0).tolist()

    A_mu = np.zeros(latent_qubits)
    A_std = np.zeros(latent_qubits)
    payoff_curr = -np.inf

    for game_iter in range(max_game_iter):
        with torch.no_grad():
            mu_pos_all, logvar_pos_all = qvae_obj.encode(pos_imgs)
        mu_pos_avg = mu_pos_all.mean(dim=0)
        std_pos_avg = logvar_pos_all.mean(dim=0)

        D = latent_qubits
        bounds = np.vstack([np.full(D, -0.5), np.full(D, 0.5)]).T

        def fit_mean(alpha_vec):
            return -adversary_payoff(combined, pos_indices, neg_indices, mu_pos_avg, std_pos_avg, qvae_obj, alpha_vec, A_std, combined_targets.cpu().numpy(), qc_model)
        em_mean = EMPSO(fit_mean, bounds, num_particles=10, max_iter=pso_iter)
        alpha_star_mean, _ = em_mean.optimize()

        def fit_std(alpha_vec):
            return -adversary_payoff(combined, pos_indices, neg_indices, mu_pos_avg, std_pos_avg, qvae_obj, alpha_star_mean, alpha_vec, combined_targets.cpu().numpy(), qc_model)
        em_std = EMPSO(fit_std, bounds, num_particles=10, max_iter=pso_iter)
        alpha_star_std, _ = em_std.optimize()

        payoff_new = adversary_payoff(combined, pos_indices, neg_indices, mu_pos_avg, std_pos_avg, qvae_obj, alpha_star_mean, alpha_star_std, combined_targets.cpu().numpy(), qc_model)
        print(f"Game {game_iter}: payoff_curr={payoff_curr:.4f} payoff_new={payoff_new:.4f}")

        if payoff_new > payoff_curr:
            payoff_curr = payoff_new
            A_mu = alpha_star_mean.copy()
            A_std = alpha_star_std.copy()
            z_adv, _, _ = quantum_adversarial_latent(mu_pos_all, logvar_pos_all, A_mu, A_std)
            adv_images = classical_decoder(z_adv.to(device))
            adv_targets = torch.ones(adv_images.size(0), dtype=torch.long).to(device)

            subset_size = min(500, len(train_images_all))
            train_data_subset = train_images_all[:subset_size].to(device)
            train_targets_subset = train_labels_all[:subset_size].to(device)
            combined_images = torch.cat([train_data_subset, adv_images], dim=0)
            combined_targets2 = torch.cat([train_targets_subset, adv_targets], dim=0)
            combined_ds = TensorDataset(combined_images, combined_targets2)
            combined_loader = DataLoader(combined_ds, batch_size=32, shuffle=True)

            qc_model.train()
            optimizer = torch.optim.Adam(qc_model.fc.parameters(), lr=1e-3)
            loss_fn = nn.CrossEntropyLoss()
            for epoch in range(3):
                run_loss = 0.0
                for bimgs, btargets in combined_loader:
                    bimgs = bimgs.to(device); btargets = btargets.to(device)
                    with torch.no_grad():
                        mu_b, logvar_b = qvae_obj.encode(bimgs)
                        z_b = qvae_obj.reparameterize(mu_b, logvar_b)
                    logits = qc_model.fc(z_b.to(device))
                    loss = loss_fn(logits, btargets)
                    optimizer.zero_grad(); loss.backward(); optimizer.step()
                    run_loss += loss.item()
                print(f" Retrain epoch {epoch+1}, loss {run_loss/len(combined_loader):.4f}")
        else:
            print("No improvement — stopping.")
            break

    return A_mu, A_std, payoff_curr

# ------------------------------
# Parameter-shift helpers (quantum finetuning)
# ------------------------------
def compute_features_and_grads_batch_for_clf_params(x_batch, clf_params_np, n_qubits):
    with torch.no_grad():
        feats_classical = feature_extractor(x_batch.to(device)).detach().cpu().numpy()
    angles = (feats_classical + 1.0) * (np.pi / 2.0)
    qc_list = [get_classifier_circuit_cached(n_qubits, clf_params_np, angles[i]) for i in range(angles.shape[0])]
    vals = measure_z_expectations_batch(qc_list)
    feats_tensor = torch.tensor(vals, dtype=torch.float32).to(device)
    return feats_tensor, angles

def parameter_shift_update(clf_params_np, x_batch, batch_labels, n_qubits,
                           qc_model, lr_q=1e-2, shift=np.pi/2):
    feats_base, angles = compute_features_and_grads_batch_for_clf_params(x_batch, clf_params_np, n_qubits)
    feats_base = feats_base.clone().detach().requires_grad_(True)
    logits = qc_model.fc(feats_base)
    loss_fn = nn.CrossEntropyLoss()
    loss = loss_fn(logits, batch_labels.to(device))
    qc_model.fc.zero_grad()
    if feats_base.grad is not None:
        feats_base.grad.zero_()
    loss.backward(retain_graph=True)
    dL_dfeat = feats_base.grad.detach().cpu().numpy()  # (B, D)
    B = dL_dfeat.shape[0]
    grads_theta = np.zeros_like(clf_params_np, dtype=float)
    for q in range(n_qubits):
        for p_idx in range(2):
            clf_plus = clf_params_np.copy()
            clf_minus = clf_params_np.copy()
            clf_plus[q, p_idx] += shift
            clf_minus[q, p_idx] -= shift
            f_plus_tensor, _ = compute_features_and_grads_batch_for_clf_params(x_batch, clf_plus, n_qubits)
            f_minus_tensor, _ = compute_features_and_grads_batch_for_clf_params(x_batch, clf_minus, n_qubits)
            f_plus = f_plus_tensor.detach().cpu().numpy()
            f_minus = f_minus_tensor.detach().cpu().numpy()
            df_dtheta = 0.5 * (f_plus - f_minus)
            term = np.sum(dL_dfeat * df_dtheta)
            grads_theta[q, p_idx] = term / float(B)
    clf_params_np = clf_params_np - lr_q * grads_theta
    return clf_params_np, loss.item()

# ------------------------------
# Hard defense + quantum finetune
# ------------------------------
def hard_defense_with_quantum_finetune(qc_model, qvae, feature_extractor, classical_decoder,
                                       train_images_all, train_labels_all,
                                       val_loader=None,
                                       epochs=4,
                                       adv_samples_per_pos=8,
                                       adv_radius=0.5,
                                       batch_size_def=64,
                                       collapse_penalty_coef=12.0,
                                       lr=5e-4,
                                       subset_size=3000,
                                       fine_tune_quantum=True,
                                       lr_q=1e-2,
                                       quantum_batches_per_epoch=3,
                                       paramshift_shift=math.pi/2):
    feature_extractor.train(); qc_model.train()
    params = list(feature_extractor.parameters()) + list(qc_model.fc.parameters())
    optimizer = torch.optim.Adam(params, lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    device_local = device
    total_clean = min(subset_size, len(train_images_all))
    clean_images = train_images_all[:total_clean].to(device_local)
    clean_labels = train_labels_all[:total_clean].to(device_local)
    pos_mask = (train_labels_all == 1)
    pos_imgs_all = train_images_all[pos_mask]
    if len(pos_imgs_all) == 0:
        raise RuntimeError("No positive-class examples found.")
    pos_sample_count = min(200, len(pos_imgs_all))
    pos_imgs = pos_imgs_all[:pos_sample_count].to(device_local)
    clf_params_np = qc_model.clf_params.copy()
    for epoch in range(epochs):
        epoch_loss = 0.0; n_batches = 0; quantum_updates_done = 0
        with torch.no_grad():
            mu_pos_all, logvar_pos_all = qvae.encode(pos_imgs)
            mu_pos_rep = mu_pos_all.repeat_interleave(adv_samples_per_pos, dim=0)
            logvar_rep = logvar_pos_all.repeat_interleave(adv_samples_per_pos, dim=0)
            delta = torch.randn_like(mu_pos_rep, device=device_local) * adv_radius
            std_delta = torch.randn_like(logvar_rep, device=device_local) * (adv_radius * 0.3)
            mu_adv = mu_pos_rep + delta
            std_adv = torch.clamp(logvar_rep + std_delta, min=-3.0, max=3.0)
            eps = torch.randn_like(mu_adv)
            z_adv = mu_adv + eps * torch.exp(0.5 * std_adv)
            adv_images = classical_decoder(z_adv.to(device_local))
            adv_labels = torch.ones(adv_images.size(0), dtype=torch.long, device=device_local)
        idxs = torch.randperm(clean_images.size(0))
        clean_shuffled = clean_images[idxs]; clean_shuffled_labels = clean_labels[idxs]
        combined_images = torch.cat([clean_shuffled, adv_images], dim=0)
        combined_labels = torch.cat([clean_shuffled_labels, adv_labels], dim=0)
        combined_ds = TensorDataset(combined_images, combined_labels)
        combined_loader = DataLoader(combined_ds, batch_size=batch_size_def, shuffle=True)
        for batch_idx, (batch_imgs, batch_labels) in enumerate(combined_loader):
            batch_imgs = batch_imgs.to(device_local); batch_labels = batch_labels.to(device_local)
            feats = feature_extractor(batch_imgs)
            logits = qc_model.fc(feats)
            ce_loss = loss_fn(logits, batch_labels)
            probs1 = torch.softmax(logits, dim=1)[:, 1]
            mean_p1 = probs1.mean()
            collapse_penalty = collapse_penalty_coef * (mean_p1 - 0.5) ** 2
            l2_reg = 0.0
            for p in params:
                l2_reg += 1e-4 * torch.sum(p ** 2)
            loss = ce_loss + collapse_penalty + l2_reg
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            epoch_loss += loss.item(); n_batches += 1
            if fine_tune_quantum and (quantum_updates_done < quantum_batches_per_epoch):
                try:
                    clf_params_np, qloss = parameter_shift_update(clf_params_np, batch_imgs, batch_labels, latent_qubits, qc_model, lr_q=lr_q, shift=paramshift_shift)
                    qc_model.clf_params = clf_params_np.copy()
                    quantum_updates_done += 1
                except Exception as e:
                    print("Parameter-shift update error:", e)
        avg_loss = epoch_loss / max(1, n_batches)
        print(f"[HardDef+QEpoch {epoch+1}/{epochs}] Avg loss: {avg_loss:.4f}, quantum_updates: {quantum_updates_done}")
        if val_loader is not None:
            print("Validation eval:")
            test_model(val_loader, qc_model)
    qc_model.clf_params = clf_params_np.copy()
    print("Hard defense + quantum fine-tune complete.")

# ------------------------------
# Demo run: adversarial game then hard defense + quantum finetune
# ------------------------------
if __name__ == "__main__":
    print("Running adversarial ALS game (short demo):")
    start = time.time()
    A_mu_final, A_std_final, final_payoff = alternating_least_squares_game(qc_model, qvae, train_images_all, train_labels_all,
                                                                           max_game_iter=2, pso_iter=6, n_pos_samples=60)
    print("Game finished in {:.1f}s".format(time.time() - start))
    print("A_mu:", A_mu_final)
    print("A_std:", A_std_final)
    print("Final payoff:", final_payoff)
    print("Test eval BEFORE defense:")
    test_model(test_loader, qc_model)

    print("\nStarting hard defense + quantum fine-tuning (this is expensive):")
    hard_defense_with_quantum_finetune(qc_model, qvae, feature_extractor, classical_decoder,
                                       train_images_all, train_labels_all,
                                       val_loader=val_loader,
                                       epochs=3,
                                       adv_samples_per_pos=8,
                                       adv_radius=0.5,
                                       batch_size_def=64,
                                       collapse_penalty_coef=12.0,
                                       lr=5e-4,
                                       subset_size=2000,
                                       fine_tune_quantum=True,
                                       lr_q=1e-2,
                                       quantum_batches_per_epoch=2,
                                       paramshift_shift=math.pi/2)

    print("Final evaluation on test set AFTER defense:")
    test_model(test_loader, qc_model)

    # Save artifacts
    np.save("qvae_encoder_params.npy", qvae.encoder_params)
    np.save("qvae_decoder_params.npy", qvae.decoder_params)
    np.save("qcnn_clf_params.npy", qc_model.clf_params)
    torch.save(qc_model.fc.state_dict(), "qcnn_fc_head.pth")
    print("Saved parameters. Done.")


Using qiskit_aer Estimator (fast).
Device: cpu
Loaded MNIST via torchvision.
Loaders prepared: Train 9415 Val 2354 Test 1932
Running adversarial ALS game (short demo):


C:\Users\Aaditya Rajput\AppData\Local\Temp\ipykernel_75124\990898259.py:194: DeprecationWarning: Estimator has been deprecated as of Aer 0.15, please use EstimatorV2 instead.
  estimator = create_estimator()
C:\Users\Aaditya Rajput\AppData\Local\Temp\ipykernel_75124\990898259.py:194: DeprecationWarning: Option approximation=False is deprecated as of qiskit-aer 0.13. It will be removed no earlier than 3 months after the release date. Instead, use BackendEstimator from qiskit.primitives.
  estimator = create_estimator()


Game 0: payoff_curr=-inf payoff_new=0.8930
 Retrain epoch 1, loss 0.7816
 Retrain epoch 2, loss 0.7619
 Retrain epoch 3, loss 0.7411
Game 1: payoff_curr=0.8930 payoff_new=1.9105
 Retrain epoch 1, loss 0.7455
 Retrain epoch 2, loss 0.7651
 Retrain epoch 3, loss 0.7609
Game finished in 351.7s
A_mu: [ 0.00906552  0.07020369 -0.07072749 -0.01074888 -0.10662028  0.39572948
 -0.09350516 -0.11875553]
A_std: [ 0.09245658 -0.0025336  -0.00305625  0.0079356   0.17187662 -0.07383863
 -0.06158765 -0.15513694]
Final payoff: 1.9105465114116669
Test eval BEFORE defense:
Acc: 0.4959, TPR0: 1.0000, TPR1: 0.0000

Starting hard defense + quantum fine-tuning (this is expensive):
[HardDef+QEpoch 1/3] Avg loss: 0.5431, quantum_updates: 2
Validation eval:
Acc: 0.5119, TPR0: 0.9959, TPR1: 0.0000
[HardDef+QEpoch 2/3] Avg loss: 0.3994, quantum_updates: 2
Validation eval:
Acc: 0.5153, TPR0: 1.0000, TPR1: 0.0026
[HardDef+QEpoch 3/3] Avg loss: 0.3546, quantum_updates: 2
Validation eval:
Acc: 0.5144, TPR0: 0.9983, 

In [2]:
# final_quantum_adversarial_game_alpha_star.py
import os, time, math, hashlib, random, traceback
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import confusion_matrix

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Operator

# Prefer Aer Estimator (fast CPU) if installed; otherwise fallback to StatevectorEstimator
try:
    from qiskit_aer.primitives import Estimator as AerEstimator
    EstimatorClass = AerEstimator
    print("Using qiskit_aer Estimator (fast).")
except Exception:
    from qiskit.primitives import StatevectorEstimator as EstimatorClass
    EstimatorClass = EstimatorClass
    print("Using qiskit.primitives.StatevectorEstimator (fallback).")

# ------------------------------
# Settings (tune these)
# ------------------------------
latent_qubits = 8    # 6-10 recommended for Option B
batch_size = 64
seed = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

# ------------------------------
# MNIST loader
# ------------------------------
def load_mnist_filtered(label1=6, label2=8, batch_size=64):
    try:
        from torchvision import datasets, transforms
        transform = transforms.Compose([transforms.ToTensor()])
        train_full = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
        test_full  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
        train_images = train_full.data.unsqueeze(1).float() / 255.0
        train_labels = train_full.targets.long()
        test_images = test_full.data.unsqueeze(1).float() / 255.0
        test_labels = test_full.targets.long()
        print("Loaded MNIST via torchvision.")
    except Exception as e:
        print("torchvision failed, falling back to sklearn.fetch_openml. Error:", e)
        from sklearn.datasets import fetch_openml
        mn = fetch_openml('mnist_784', version=1, as_frame=False)
        X = mn['data'].astype('float32') / 255.0
        y = mn['target'].astype(int)
        X = X.reshape(-1, 1, 28, 28)
        train_images, test_images = X[:60000], X[60000:]
        train_labels, test_labels = y[:60000], y[60000:]
        train_images = torch.from_numpy(train_images)
        train_labels = torch.from_numpy(train_labels).long()
        test_images = torch.from_numpy(test_images)
        test_labels = torch.from_numpy(test_labels).long()
        print("Loaded MNIST via sklearn.fetch_openml.")

    mask_train = (train_labels == label1) | (train_labels == label2)
    mask_test  = (test_labels  == label1) | (test_labels  == label2)

    train_images = train_images[mask_train]
    train_labels = train_labels[mask_train]
    test_images  = test_images[mask_test]
    test_labels  = test_labels[mask_test]

    train_labels = torch.where(train_labels == label1, torch.tensor(0, dtype=torch.long), torch.tensor(1, dtype=torch.long))
    test_labels  = torch.where(test_labels  == label1, torch.tensor(0, dtype=torch.long), torch.tensor(1, dtype=torch.long))

    total_train = TensorDataset(train_images, train_labels)
    train_size = int(0.8 * len(total_train))
    val_size = len(total_train) - train_size
    train_dataset, val_dataset = random_split(total_train, [train_size, val_size])
    test_dataset = TensorDataset(test_images, test_labels)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print("Loaders prepared: Train", len(train_dataset), "Val", len(val_dataset), "Test", len(test_dataset))
    return train_loader, val_loader, test_loader, train_images, train_labels, test_images, test_labels

train_loader, val_loader, test_loader, train_images_all, train_labels_all, test_images_all, test_labels_all = load_mnist_filtered(batch_size=batch_size)

# ------------------------------
# Classical feature extractor + decoder
# ------------------------------
input_dim = 28 * 28
class ClassicalFeatureExtractor(nn.Module):
    def __init__(self, latent_qubits):
        super().__init__()
        self.fc = nn.Linear(input_dim, latent_qubits)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return torch.tanh(self.fc(x))
class ClassicalDecoder(nn.Module):
    def __init__(self, latent_qubits):
        super().__init__()
        self.fc1 = nn.Linear(latent_qubits, 128)
        self.fc2 = nn.Linear(128, input_dim)
    def forward(self, z):
        z = F.relu(self.fc1(z))
        out = torch.sigmoid(self.fc2(z))
        out = out.view(-1,1,28,28)
        return out

feature_extractor = ClassicalFeatureExtractor(latent_qubits).to(device)
classical_decoder = ClassicalDecoder(latent_qubits).to(device)

# ------------------------------
# Circuit caching utilities
# ------------------------------
CIRCUIT_CACHE = {}
def _params_to_list(params):
    if isinstance(params, np.ndarray):
        return params.flatten().tolist()
    elif isinstance(params, (list, tuple)):
        return np.array(params).flatten().tolist()
    else:
        return np.array(params).flatten().tolist()

def cache_key(type_name, params, angles):
    p_list = _params_to_list(params)
    a_list = np.round(np.array(angles).flatten(), 6).tolist()
    raw = f"{type_name}|{p_list}|{a_list}"
    return hashlib.sha256(raw.encode()).hexdigest()

def get_encoder_circuit_cached(n_qubits, encoder_params, x_angles):
    key = cache_key("enc", encoder_params, x_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(x_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(encoder_params[i,0]), i)
        qc.rz(float(encoder_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

def get_decoder_circuit_cached(n_qubits, decoder_params, z_angles):
    key = cache_key("dec", decoder_params, z_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(z_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(decoder_params[i,0]), i)
        qc.rz(float(decoder_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

def get_classifier_circuit_cached(n_qubits, clf_params, x_angles):
    key = cache_key("clf", clf_params, x_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(x_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(clf_params[i,0]), i)
        qc.rz(float(clf_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

# ------------------------------
# Observables: Z individual + grouped
# ------------------------------
z_ops = []
for i in range(latent_qubits):
    s = ['I'] * latent_qubits
    s[latent_qubits - 1 - i] = 'Z'
    z_ops.append(SparsePauliOp(''.join(s)))
Z_GROUPED = z_ops[0]
for i in range(1, latent_qubits):
    Z_GROUPED = Z_GROUPED + z_ops[i]

# ------------------------------
# Estimator instance
# ------------------------------
def create_estimator():
    try:
        return EstimatorClass()
    except Exception:
        return EstimatorClass()
estimator = create_estimator()

# ------------------------------
# measure_z_expectations_batch (grouped + per-qubit calls with caching)
# ------------------------------
def measure_z_expectations_batch(qc_list):
    if len(qc_list) == 0:
        return np.zeros((0, latent_qubits), dtype=float)
    n_circuits = len(qc_list)
    # grouped warm call
    grouped_obs_list = [Z_GROUPED] * n_circuits
    job = estimator.run(circuits=qc_list, observables=grouped_obs_list)
    _ = job.result()
    # per-qubit calls (one call per qubit)
    values = np.zeros((n_circuits, latent_qubits), dtype=float)
    for q in range(latent_qubits):
        obs = z_ops[q]
        obs_list = [obs] * n_circuits
        job_q = estimator.run(circuits=qc_list, observables=obs_list)
        res_q = job_q.result()
        vals_q = np.array(res_q.values).reshape(-1)
        values[:, q] = vals_q
    return values

# ------------------------------
# QuantumVAE
# ------------------------------
class QuantumVAE:
    def __init__(self, n_qubits):
        self.n_qubits = n_qubits
        self.encoder_params = np.random.randn(n_qubits, 2) * 0.05
        self.decoder_params = np.random.randn(n_qubits, 2) * 0.05

    def encode(self, x_batch, angles_cache=None):
        x_batch = x_batch.detach().cpu()
        b = x_batch.size(0)
        if angles_cache is None:
            feats = feature_extractor(x_batch.to(device)).detach().cpu().numpy()
            angles = (feats + 1.0) * (np.pi / 2.0)
        else:
            angles = angles_cache
        qc_list = [get_encoder_circuit_cached(self.n_qubits, self.encoder_params, angles[i]) for i in range(b)]
        vals = measure_z_expectations_batch(qc_list)
        mu_angles = np.arcsin(np.clip(vals, -1.0, 1.0))
        mu = torch.tensor(mu_angles, dtype=torch.float32).to(device)
        logvar = torch.zeros_like(mu).to(device)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        eps = torch.randn_like(mu)
        return mu + eps * torch.exp(0.5 * logvar)

    def decode(self, z_batch):
        z_np = z_batch.detach().cpu().numpy()
        qc_list = [get_decoder_circuit_cached(self.n_qubits, self.decoder_params, z_np[i]) for i in range(z_np.shape[0])]
        vals = measure_z_expectations_batch(qc_list)
        return torch.tensor(vals, dtype=torch.float32).to(device)

    def forward(self, x_batch, angles_cache=None):
        mu, logvar = self.encode(x_batch, angles_cache)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

qvae = QuantumVAE(latent_qubits)

# ------------------------------
# QuantumCNN wrapper (quantum features -> classical head)
# ------------------------------
class QuantumCNN(nn.Module):
    def __init__(self, n_qubits, n_classes=2):
        super().__init__()
        self.n_qubits = n_qubits
        self.clf_params = np.random.randn(n_qubits, 2) * 0.05
        self.fc = nn.Linear(n_qubits, n_classes).to(device)

    def quantum_features_batch(self, x_batch, angles_cache=None):
        x_batch = x_batch.detach().cpu()
        b = x_batch.size(0)
        if angles_cache is None:
            feats = feature_extractor(x_batch.to(device)).detach().cpu().numpy()
            angles = (feats + 1.0) * (np.pi / 2.0)
        else:
            angles = angles_cache
        qc_list = [get_classifier_circuit_cached(self.n_qubits, self.clf_params, angles[i]) for i in range(b)]
        vals = measure_z_expectations_batch(qc_list)
        return torch.tensor(vals, dtype=torch.float32).to(device)

    def forward(self, x_batch, angles_cache=None):
        feats_q = self.quantum_features_batch(x_batch, angles_cache)
        logits = self.fc(feats_q)
        return logits

qc_model = QuantumCNN(latent_qubits)

# ------------------------------
# Metrics & test
# ------------------------------
def test_model(loader, model):
    model.eval()
    all_targets = []
    all_preds = []
    with torch.no_grad():
        for data, targets in loader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs, 1)
            all_targets.extend(targets.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
    cm = confusion_matrix(all_targets, all_preds, labels=[0,1])
    if cm.size == 4:
        tp0, fn0 = cm[0,0], cm[0,1]
        tp1, fn1 = cm[1,1], cm[1,0]
    else:
        tp0 = fn0 = tp1 = fn1 = 0
    tpr0 = tp0 / (tp0 + fn0) if (tp0 + fn0) > 0 else 0.0
    tpr1 = tp1 / (tp1 + fn1) if (tp1 + fn1) > 0 else 0.0
    acc = (np.array(all_preds) == np.array(all_targets)).mean() if len(all_targets) else 0.0
    print(f"Acc: {acc:.4f}, TPR0: {tpr0:.4f}, TPR1: {tpr1:.4f}")
    return tpr0, tpr1

# ------------------------------
# Adversarial utilities (latent)
# ------------------------------
def quantum_adversarial_latent(mu_pos, logvar_pos, A_mu, A_std):
    if isinstance(A_mu, np.ndarray):
        A_mu = torch.tensor(A_mu, dtype=torch.float32).to(device)
    if isinstance(A_std, np.ndarray):
        A_std = torch.tensor(A_std, dtype=torch.float32).to(device)
    mu_adv = mu_pos + A_mu
    logvar_adv = logvar_pos + A_std
    eps = torch.randn_like(mu_adv)
    z_adv = mu_adv + eps * torch.exp(0.5 * logvar_adv)
    return z_adv, mu_adv, logvar_adv

def calc_cost(a_mean, a_std):
    a_mean_t = torch.tensor(a_mean, dtype=torch.float32) if not isinstance(a_mean, torch.Tensor) else a_mean
    a_std_t = torch.tensor(a_std, dtype=torch.float32) if not isinstance(a_std, torch.Tensor) else a_std
    norm_mean = torch.norm(a_mean_t)
    norm_std = torch.norm(a_std_t)
    total_cost = (norm_mean / latent_qubits) + (norm_std / latent_qubits)
    return total_cost.item()

def adversary_payoff(posnegdata, posindices, negindices, mu_pos, std_pos, qvae_obj, a_mean, a_std, posnegtargets, model):
    if mu_pos.dim() == 1:
        mu_pos = mu_pos.unsqueeze(0)
    if std_pos.dim() == 1:
        std_pos = std_pos.unsqueeze(0)
    z_adv, _, _ = quantum_adversarial_latent(mu_pos, std_pos, a_mean, a_std)
    recon_images = classical_decoder(z_adv.to(device))
    logits = model.fc(z_adv.to(device))
    _, preds = torch.max(logits, 1)
    targets = torch.tensor(posnegtargets, dtype=torch.long).to(device)
    true_positives = ((preds == 1) & (targets == 1)).sum().item()
    false_negatives = ((preds == 0) & (targets == 1)).sum().item()
    tpr1 = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0.0
    error = 1.0 - tpr1
    cost = calc_cost(a_mean, a_std)
    payoff = 1.0 + error - cost
    return payoff

# ------------------------------
# EMPSO optimizer
# ------------------------------
class Particle:
    def __init__(self, bounds):
        self.position = np.random.uniform(bounds[:,0], bounds[:,1])
        self.velocity = np.zeros_like(self.position)
        self.momentum = np.zeros_like(self.position)
        self.pbest_position = self.position.copy()
        self.pbest_value = float('inf')
    def update_personal_best(self, fitness):
        if fitness < self.pbest_value:
            self.pbest_value = fitness
            self.pbest_position = self.position.copy()

class EMPSO:
    def __init__(self, fitness_fn, bounds, num_particles=12, beta=0.7, c1=1.5, c2=1.5, max_iter=20):
        self.fitness_fn = fitness_fn
        self.bounds = bounds
        self.num_particles = num_particles
        self.beta = beta
        self.c1 = c1
        self.c2 = c2
        self.max_iter = max_iter
        self.swarm = [Particle(bounds) for _ in range(num_particles)]
        self.gbest_position = None
        self.gbest_value = float('inf')
    def optimize(self):
        for it in range(self.max_iter):
            for p in self.swarm:
                fitness = self.fitness_fn(p.position)
                p.update_personal_best(fitness)
                if fitness < self.gbest_value:
                    self.gbest_value = fitness
                    self.gbest_position = p.position.copy()
            for p in self.swarm:
                r1 = np.random.rand(len(p.position))
                r2 = np.random.rand(len(p.position))
                p.velocity = (self.beta * p.momentum +
                              (1 - self.beta) * p.velocity +
                              self.c1 * r1 * (p.pbest_position - p.position) +
                              self.c2 * r2 * (self.gbest_position - p.position))
                p.position += p.velocity
                p.momentum = self.beta * p.momentum + (1 - self.beta) * p.velocity
                p.position = np.clip(p.position, self.bounds[:,0], self.bounds[:,1])
        return self.gbest_position, self.gbest_value

# ------------------------------
# Alternating least squares adversarial game
# ------------------------------
def alternating_least_squares_game(qc_model, qvae_obj, train_images_all, train_labels_all,
                                   max_game_iter=2, pso_iter=8, n_pos_samples=80):
    imgs = train_images_all
    labs = train_labels_all
    pos_mask = (labs == 1)
    neg_mask = (labs == 0)
    pos_imgs = imgs[pos_mask][:n_pos_samples].to(device)
    pos_labs = labs[pos_mask][:n_pos_samples].to(device)
    neg_imgs = imgs[neg_mask][:n_pos_samples].to(device)
    neg_labs = labs[neg_mask][:n_pos_samples].to(device)

    combined = torch.cat([pos_imgs, neg_imgs], dim=0)
    combined_targets = torch.cat([pos_labs, neg_labs], dim=0)
    pos_indices = (combined_targets.cpu().numpy() == 1).tolist()
    neg_indices = (combined_targets.cpu().numpy() == 0).tolist()

    A_mu = np.zeros(latent_qubits)
    A_std = np.zeros(latent_qubits)
    payoff_curr = -np.inf

    for game_iter in range(max_game_iter):
        with torch.no_grad():
            mu_pos_all, logvar_pos_all = qvae_obj.encode(pos_imgs)
        mu_pos_avg = mu_pos_all.mean(dim=0)
        std_pos_avg = logvar_pos_all.mean(dim=0)

        D = latent_qubits
        bounds = np.vstack([np.full(D, -0.5), np.full(D, 0.5)]).T

        def fit_mean(alpha_vec):
            return -adversary_payoff(combined, pos_indices, neg_indices, mu_pos_avg, std_pos_avg, qvae_obj, alpha_vec, A_std, combined_targets.cpu().numpy(), qc_model)
        em_mean = EMPSO(fit_mean, bounds, num_particles=10, max_iter=pso_iter)
        alpha_star_mean, _ = em_mean.optimize()

        def fit_std(alpha_vec):
            return -adversary_payoff(combined, pos_indices, neg_indices, mu_pos_avg, std_pos_avg, qvae_obj, alpha_star_mean, alpha_vec, combined_targets.cpu().numpy(), qc_model)
        em_std = EMPSO(fit_std, bounds, num_particles=10, max_iter=pso_iter)
        alpha_star_std, _ = em_std.optimize()

        payoff_new = adversary_payoff(combined, pos_indices, neg_indices, mu_pos_avg, std_pos_avg, qvae_obj, alpha_star_mean, alpha_star_std, combined_targets.cpu().numpy(), qc_model)
        print(f"Game {game_iter}: payoff_curr={payoff_curr:.4f} payoff_new={payoff_new:.4f}")

        if payoff_new > payoff_curr:
            payoff_curr = payoff_new
            A_mu = alpha_star_mean.copy()
            A_std = alpha_star_std.copy()
            z_adv, _, _ = quantum_adversarial_latent(mu_pos_all, logvar_pos_all, A_mu, A_std)
            adv_images = classical_decoder(z_adv.to(device))
            adv_targets = torch.ones(adv_images.size(0), dtype=torch.long).to(device)

            subset_size = min(500, len(train_images_all))
            train_data_subset = train_images_all[:subset_size].to(device)
            train_targets_subset = train_labels_all[:subset_size].to(device)
            combined_images = torch.cat([train_data_subset, adv_images], dim=0)
            combined_targets2 = torch.cat([train_targets_subset, adv_targets], dim=0)
            combined_ds = TensorDataset(combined_images, combined_targets2)
            combined_loader = DataLoader(combined_ds, batch_size=32, shuffle=True)

            qc_model.train()
            optimizer = torch.optim.Adam(qc_model.fc.parameters(), lr=1e-3)
            loss_fn = nn.CrossEntropyLoss()
            for epoch in range(3):
                run_loss = 0.0
                for bimgs, btargets in combined_loader:
                    bimgs = bimgs.to(device); btargets = btargets.to(device)
                    with torch.no_grad():
                        mu_b, logvar_b = qvae_obj.encode(bimgs)
                        z_b = qvae_obj.reparameterize(mu_b, logvar_b)
                    logits = qc_model.fc(z_b.to(device))
                    loss = loss_fn(logits, btargets)
                    optimizer.zero_grad(); loss.backward(); optimizer.step()
                    run_loss += loss.item()
                print(f" Retrain epoch {epoch+1}, loss {run_loss/len(combined_loader):.4f}")
        else:
            print("No improvement — stopping.")
            break

    return A_mu, A_std, payoff_curr

# ------------------------------
# Parameter-shift helpers (quantum finetuning)
# ------------------------------
def compute_features_and_grads_batch_for_clf_params(x_batch, clf_params_np, n_qubits):
    with torch.no_grad():
        feats_classical = feature_extractor(x_batch.to(device)).detach().cpu().numpy()
    angles = (feats_classical + 1.0) * (np.pi / 2.0)
    qc_list = [get_classifier_circuit_cached(n_qubits, clf_params_np, angles[i]) for i in range(angles.shape[0])]
    vals = measure_z_expectations_batch(qc_list)
    feats_tensor = torch.tensor(vals, dtype=torch.float32).to(device)
    return feats_tensor, angles

def parameter_shift_update(clf_params_np, x_batch, batch_labels, n_qubits,
                           qc_model, lr_q=1e-2, shift=np.pi/2):
    feats_base, angles = compute_features_and_grads_batch_for_clf_params(x_batch, clf_params_np, n_qubits)
    feats_base = feats_base.clone().detach().requires_grad_(True)
    logits = qc_model.fc(feats_base)
    loss_fn = nn.CrossEntropyLoss()
    loss = loss_fn(logits, batch_labels.to(device))
    qc_model.fc.zero_grad()
    if feats_base.grad is not None:
        feats_base.grad.zero_()
    loss.backward(retain_graph=True)
    dL_dfeat = feats_base.grad.detach().cpu().numpy()  # (B, D)
    B = dL_dfeat.shape[0]
    grads_theta = np.zeros_like(clf_params_np, dtype=float)
    for q in range(n_qubits):
        for p_idx in range(2):
            clf_plus = clf_params_np.copy()
            clf_minus = clf_params_np.copy()
            clf_plus[q, p_idx] += shift
            clf_minus[q, p_idx] -= shift
            f_plus_tensor, _ = compute_features_and_grads_batch_for_clf_params(x_batch, clf_plus, n_qubits)
            f_minus_tensor, _ = compute_features_and_grads_batch_for_clf_params(x_batch, clf_minus, n_qubits)
            f_plus = f_plus_tensor.detach().cpu().numpy()
            f_minus = f_minus_tensor.detach().cpu().numpy()
            df_dtheta = 0.5 * (f_plus - f_minus)
            term = np.sum(dL_dfeat * df_dtheta)
            grads_theta[q, p_idx] = term / float(B)
    clf_params_np = clf_params_np - lr_q * grads_theta
    return clf_params_np, loss.item()

# ------------------------------
# Hard defense + quantum finetune
# ------------------------------
def hard_defense_with_quantum_finetune(qc_model, qvae, feature_extractor, classical_decoder,
                                       train_images_all, train_labels_all,
                                       val_loader=None,
                                       epochs=4,
                                       adv_samples_per_pos=8,
                                       adv_radius=0.5,
                                       batch_size_def=64,
                                       collapse_penalty_coef=12.0,
                                       lr=5e-4,
                                       subset_size=3000,
                                       fine_tune_quantum=True,
                                       lr_q=1e-2,
                                       quantum_batches_per_epoch=3,
                                       paramshift_shift=math.pi/2):
    feature_extractor.train(); qc_model.train()
    params = list(feature_extractor.parameters()) + list(qc_model.fc.parameters())
    optimizer = torch.optim.Adam(params, lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    device_local = device
    total_clean = min(subset_size, len(train_images_all))
    clean_images = train_images_all[:total_clean].to(device_local)
    clean_labels = train_labels_all[:total_clean].to(device_local)
    pos_mask = (train_labels_all == 1)
    pos_imgs_all = train_images_all[pos_mask]
    if len(pos_imgs_all) == 0:
        raise RuntimeError("No positive-class examples found.")
    pos_sample_count = min(200, len(pos_imgs_all))
    pos_imgs = pos_imgs_all[:pos_sample_count].to(device_local)
    clf_params_np = qc_model.clf_params.copy()
    for epoch in range(epochs):
        epoch_loss = 0.0; n_batches = 0; quantum_updates_done = 0
        with torch.no_grad():
            mu_pos_all, logvar_pos_all = qvae.encode(pos_imgs)
            mu_pos_rep = mu_pos_all.repeat_interleave(adv_samples_per_pos, dim=0)
            logvar_rep = logvar_pos_all.repeat_interleave(adv_samples_per_pos, dim=0)
            delta = torch.randn_like(mu_pos_rep, device=device_local) * adv_radius
            std_delta = torch.randn_like(logvar_rep, device=device_local) * (adv_radius * 0.3)
            mu_adv = mu_pos_rep + delta
            std_adv = torch.clamp(logvar_rep + std_delta, min=-3.0, max=3.0)
            eps = torch.randn_like(mu_adv)
            z_adv = mu_adv + eps * torch.exp(0.5 * std_adv)
            adv_images = classical_decoder(z_adv.to(device_local))
            adv_labels = torch.ones(adv_images.size(0), dtype=torch.long, device=device_local)
        idxs = torch.randperm(clean_images.size(0))
        clean_shuffled = clean_images[idxs]; clean_shuffled_labels = clean_labels[idxs]
        combined_images = torch.cat([clean_shuffled, adv_images], dim=0)
        combined_labels = torch.cat([clean_shuffled_labels, adv_labels], dim=0)
        combined_ds = TensorDataset(combined_images, combined_labels)
        combined_loader = DataLoader(combined_ds, batch_size=batch_size_def, shuffle=True)
        for batch_idx, (batch_imgs, batch_labels) in enumerate(combined_loader):
            batch_imgs = batch_imgs.to(device_local); batch_labels = batch_labels.to(device_local)
            feats = feature_extractor(batch_imgs)
            logits = qc_model.fc(feats)
            ce_loss = loss_fn(logits, batch_labels)
            probs1 = torch.softmax(logits, dim=1)[:, 1]
            mean_p1 = probs1.mean()
            collapse_penalty = collapse_penalty_coef * (mean_p1 - 0.5) ** 2
            l2_reg = 0.0
            for p in params:
                l2_reg += 1e-4 * torch.sum(p ** 2)
            loss = ce_loss + collapse_penalty + l2_reg
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            epoch_loss += loss.item(); n_batches += 1
            if fine_tune_quantum and (quantum_updates_done < quantum_batches_per_epoch):
                try:
                    clf_params_np, qloss = parameter_shift_update(clf_params_np, batch_imgs, batch_labels, latent_qubits, qc_model, lr_q=lr_q, shift=paramshift_shift)
                    qc_model.clf_params = clf_params_np.copy()
                    quantum_updates_done += 1
                except Exception as e:
                    print("Parameter-shift update error:", e)
        avg_loss = epoch_loss / max(1, n_batches)
        print(f"[HardDef+QEpoch {epoch+1}/{epochs}] Avg loss: {avg_loss:.4f}, quantum_updates: {quantum_updates_done}")
        if val_loader is not None:
            print("Validation eval:")
            test_model(val_loader, qc_model)
    qc_model.clf_params = clf_params_np.copy()
    print("Hard defense + quantum fine-tune complete.")

# ------------------------------
# Demo run: adversarial game then hard defense + quantum finetune
# ------------------------------
if __name__ == "__main__":
    print("Running adversarial ALS game (short demo):")
    start = time.time()
    A_mu_final, A_std_final, final_payoff = alternating_least_squares_game(qc_model, qvae, train_images_all, train_labels_all,
                                                                           max_game_iter=2, pso_iter=6, n_pos_samples=60)
    print("Game finished in {:.1f}s".format(time.time() - start))
    print("A_mu:", A_mu_final)
    print("A_std:", A_std_final)
    print("Final payoff:", final_payoff)
    print("Test eval BEFORE defense:")
    baseline_tpr0, baseline_tpr1 = test_model(test_loader, qc_model)

    # ------------------------------
    # ALPHA★ POST-GAME ATTACK EVALUATION (inserted here)
    # ------------------------------
    print("\n=== ALPHA★ POST-GAME ATTACK EVALUATION ===")
    # Use a subset of positive test samples for evaluation (or all if small)
    pos_mask_test = (test_labels_all == 1)
    pos_imgs_test = test_images_all[pos_mask_test]
    if pos_imgs_test.size(0) == 0:
        print("No positive test samples found; skipping alpha-star attack evaluation.")
    else:
        n_attack_samples = min(120, pos_imgs_test.size(0))  # pick up to 120 positives
        pos_imgs_test = pos_imgs_test[:n_attack_samples].to(device)

        with torch.no_grad():
            mu_pos_test, logvar_pos_test = qvae.encode(pos_imgs_test)

        # Create alpha-star manipulated latents using final A_mu_final and A_std_final
        # Use the same format as quantum_adversarial_latent
        z_star, mu_star, logvar_star = quantum_adversarial_latent(mu_pos_test, logvar_pos_test, A_mu_final, A_std_final)

        # Evaluate classifier directly on alpha-star latents (fast)
        qc_model.eval()
        with torch.no_grad():
            logits_star = qc_model.fc(z_star.to(device))
            _, preds_star = torch.max(logits_star, 1)
            true_labels_star = torch.ones_like(preds_star)  # these were positive-class samples

            tp1 = ((preds_star == 1) & (true_labels_star == 1)).sum().item()
            fn1 = ((preds_star == 0) & (true_labels_star == 1)).sum().item()
            tpr1_attack = tp1 / (tp1 + fn1 + 1e-12)
            asr = 1.0 - tpr1_attack  # attack success rate (reduction in TPR1)

        print("\n--- Alpha★ Attack Results (on latent inputs) ---")
        print(f"Num attack samples: {n_attack_samples}")
        print(f"TPR1 on alpha★ manipulated positives (latent eval): {tpr1_attack:.4f}")
        print(f"Attack Success Rate (ASR): {asr:.4f}")
        print(f"Payoff at convergence: {final_payoff:.4f}")

        # Also evaluate on reconstructed attacked images (pixel-space)
        with torch.no_grad():
            recon_star = classical_decoder(z_star.to(device))
            # We need to evaluate classifier either on latent z or re-encode reconstructed images.
            # For pixel-based evaluation, re-run encoder -> classifier to be consistent:
            mu_re, logvar_re = qvae.encode(recon_star)
            z_re = qvae.reparameterize(mu_re, logvar_re)
            logits_re = qc_model.fc(z_re.to(device))
            _, preds_re = torch.max(logits_re, 1)
            acc_recon = (preds_re == torch.ones_like(preds_re)).float().mean().item()

        print("\nClassifier performance on reconstructed alpha★ images (pixel-space):")
        print(f"Accuracy (should be low if attack succeeds): {acc_recon:.4f}")

    # ------------------------------
    # Baseline model performance reporting (clean and attacked) saved to disk
    # ------------------------------
    baseline_metrics = {
        "baseline_tpr0": float(baseline_tpr0),
        "baseline_tpr1": float(baseline_tpr1),
        "A_mu_final": A_mu_final.tolist() if isinstance(A_mu_final, np.ndarray) else A_mu_final,
        "A_std_final": A_std_final.tolist() if isinstance(A_std_final, np.ndarray) else A_std_final,
        "final_payoff": float(final_payoff),
    }
    # attempt to store attack metrics if computed
    try:
        baseline_metrics["tpr1_attack"] = float(tpr1_attack)
        baseline_metrics["asr"] = float(asr)
        baseline_metrics["acc_recon"] = float(acc_recon)
    except Exception:
        pass

    import json
    with open("baseline_and_attack_metrics.json", "w") as f:
        json.dump(baseline_metrics, f, indent=2)
    print("Saved baseline + attack metrics -> baseline_and_attack_metrics.json")

    # ------------------------------
    # Now run hard defense + quantum fine-tuning
    # ------------------------------
    print("\nStarting hard defense + quantum fine-tuning (this is expensive):")
    hard_defense_with_quantum_finetune(qc_model, qvae, feature_extractor, classical_decoder,
                                       train_images_all, train_labels_all,
                                       val_loader=val_loader,
                                       epochs=3,
                                       adv_samples_per_pos=8,
                                       adv_radius=0.5,
                                       batch_size_def=64,
                                       collapse_penalty_coef=12.0,
                                       lr=5e-4,
                                       subset_size=2000,
                                       fine_tune_quantum=True,
                                       lr_q=1e-2,
                                       quantum_batches_per_epoch=2,
                                       paramshift_shift=math.pi/2)

    print("Final evaluation on test set AFTER defense:")
    test_model(test_loader, qc_model)

    # Save artifacts
    np.save("qvae_encoder_params.npy", qvae.encoder_params)
    np.save("qvae_decoder_params.npy", qvae.decoder_params)
    np.save("qcnn_clf_params.npy", qc_model.clf_params)
    torch.save(qc_model.fc.state_dict(), "qcnn_fc_head.pth")
    print("Saved parameters. Done.")


Using qiskit_aer Estimator (fast).
Device: cpu
Loaded MNIST via torchvision.
Loaders prepared: Train 9415 Val 2354 Test 1932


C:\Users\Aaditya Rajput\AppData\Local\Temp\ipykernel_75124\3960447829.py:195: DeprecationWarning: Estimator has been deprecated as of Aer 0.15, please use EstimatorV2 instead.
  estimator = create_estimator()
C:\Users\Aaditya Rajput\AppData\Local\Temp\ipykernel_75124\3960447829.py:195: DeprecationWarning: Option approximation=False is deprecated as of qiskit-aer 0.13. It will be removed no earlier than 3 months after the release date. Instead, use BackendEstimator from qiskit.primitives.
  estimator = create_estimator()


Running adversarial ALS game (short demo):
Game 0: payoff_curr=-inf payoff_new=0.8930
 Retrain epoch 1, loss 0.7829
 Retrain epoch 2, loss 0.7622
 Retrain epoch 3, loss 0.7418
Game 1: payoff_curr=0.8930 payoff_new=1.9105
 Retrain epoch 1, loss 0.7463
 Retrain epoch 2, loss 0.7656
 Retrain epoch 3, loss 0.7602
Game finished in 501.7s
A_mu: [ 0.00906552  0.07020369 -0.07072749 -0.01074888 -0.10662028  0.39572948
 -0.09350516 -0.11875553]
A_std: [ 0.09245658 -0.0025336  -0.00305625  0.0079356   0.17187662 -0.07383863
 -0.06158765 -0.15513694]
Final payoff: 1.9105465114116669
Test eval BEFORE defense:
Acc: 0.4959, TPR0: 1.0000, TPR1: 0.0000

=== ALPHA★ POST-GAME ATTACK EVALUATION ===

--- Alpha★ Attack Results (on latent inputs) ---
Num attack samples: 120
TPR1 on alpha★ manipulated positives (latent eval): 0.3167
Attack Success Rate (ASR): 0.6833
Payoff at convergence: 1.9105

Classifier performance on reconstructed alpha★ images (pixel-space):
Accuracy (should be low if attack succeeds):

In [3]:
# ======================================================
# final_quantum_adversarial_game_SA2.py
# Full script with Simulated Annealing replacing PSO
# ======================================================

import os, time, math, hashlib, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import confusion_matrix

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Prefer Aer Estimator
try:
    from qiskit_aer.primitives import Estimator as AerEstimator
    EstimatorClass = AerEstimator
    print("Using qiskit_aer Estimator.")
except Exception:
    from qiskit.primitives import StatevectorEstimator as EstimatorClass
    print("Using StatevectorEstimator fallback.")

# ------------------------------
# Settings
# ------------------------------
latent_qubits = 8
batch_size = 64
seed = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

# ------------------------------
# MNIST Loader (6 vs 8)
# ------------------------------
def load_mnist_filtered(label1=6, label2=8, batch_size=64):
    try:
        from torchvision import datasets, transforms
        transform = transforms.Compose([transforms.ToTensor()])
        train_full = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
        test_full  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

        train_images = train_full.data.unsqueeze(1).float() / 255.0
        train_labels = train_full.targets.long()
        test_images = test_full.data.unsqueeze(1).float() / 255.0
        test_labels = test_full.targets.long()

        print("Loaded MNIST via torchvision.")
    except Exception as e:
        print("torchvision failed, falling back to sklearn:", e)
        from sklearn.datasets import fetch_openml
        mn = fetch_openml('mnist_784', version=1, as_frame=False)
        X = mn['data'].astype('float32') / 255.0
        y = mn['target'].astype(int)
        X = X.reshape(-1, 1, 28, 28)

        train_images, test_images = X[:60000], X[60000:]
        train_labels, test_labels = y[:60000], y[60000:]

        train_images = torch.from_numpy(train_images)
        train_labels = torch.from_numpy(train_labels).long()
        test_images = torch.from_numpy(test_images)
        test_labels = torch.from_numpy(test_labels).long()

        print("Loaded MNIST via sklearn.")

    mask_train = (train_labels == label1) | (train_labels == label2)
    mask_test  = (test_labels == label1) | (test_labels == label2)

    train_images = train_images[mask_train]
    train_labels = train_labels[mask_train]
    test_images  = test_images[mask_test]
    test_labels  = test_labels[mask_test]

    train_labels = torch.where(train_labels == label1, torch.tensor(0), torch.tensor(1))
    test_labels  = torch.where(test_labels  == label1, torch.tensor(0), torch.tensor(1))

    total_train = TensorDataset(train_images, train_labels)
    train_size = int(0.8 * len(total_train))
    val_size = len(total_train) - train_size

    train_dataset, val_dataset = random_split(total_train, [train_size, val_size])
    test_dataset = TensorDataset(test_images, test_labels)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print("Datasets prepared.")
    return train_loader, val_loader, test_loader, train_images, train_labels, test_images, test_labels


train_loader, val_loader, test_loader, train_images_all, train_labels_all, test_images_all, test_labels_all = \
    load_mnist_filtered(batch_size=batch_size)

# ------------------------------
# Classical feature extractor + decoder
# ------------------------------
input_dim = 28 * 28

class ClassicalFeatureExtractor(nn.Module):
    def __init__(self, latent_qubits):
        super().__init__()
        self.fc = nn.Linear(input_dim, latent_qubits)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return torch.tanh(self.fc(x))

class ClassicalDecoder(nn.Module):
    def __init__(self, latent_qubits):
        super().__init__()
        self.fc1 = nn.Linear(latent_qubits, 128)
        self.fc2 = nn.Linear(128, input_dim)

    def forward(self, z):
        z = F.relu(self.fc1(z))
        out = torch.sigmoid(self.fc2(z))
        return out.view(-1,1,28,28)

feature_extractor = ClassicalFeatureExtractor(latent_qubits).to(device)
classical_decoder = ClassicalDecoder(latent_qubits).to(device)

# ---------------------------------------------------------
# Part 2/5
# Quantum circuit builders, estimator, measure, models, SA
# ---------------------------------------------------------

# ------------------------------
# Circuit caching utilities
# ------------------------------
CIRCUIT_CACHE = {}
def _params_to_list(params):
    if isinstance(params, np.ndarray):
        return params.flatten().tolist()
    elif isinstance(params, (list, tuple)):
        return np.array(params).flatten().tolist()
    else:
        return np.array(params).flatten().tolist()

def cache_key(type_name, params, angles):
    p_list = _params_to_list(params)
    a_list = np.round(np.array(angles).flatten(), 6).tolist()
    raw = f"{type_name}|{p_list}|{a_list}"
    return hashlib.sha256(raw.encode()).hexdigest()

def get_encoder_circuit_cached(n_qubits, encoder_params, x_angles):
    key = cache_key("enc", encoder_params, x_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(x_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(encoder_params[i,0]), i)
        qc.rz(float(encoder_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

def get_decoder_circuit_cached(n_qubits, decoder_params, z_angles):
    key = cache_key("dec", decoder_params, z_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(z_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(decoder_params[i,0]), i)
        qc.rz(float(decoder_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

def get_classifier_circuit_cached(n_qubits, clf_params, x_angles):
    key = cache_key("clf", clf_params, x_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(x_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(clf_params[i,0]), i)
        qc.rz(float(clf_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

# ------------------------------
# Observables: Z individual + grouped
# ------------------------------
z_ops = []
for i in range(latent_qubits):
    s = ['I'] * latent_qubits
    s[latent_qubits - 1 - i] = 'Z'
    z_ops.append(SparsePauliOp(''.join(s)))
Z_GROUPED = z_ops[0]
for i in range(1, latent_qubits):
    Z_GROUPED = Z_GROUPED + z_ops[i]

# ------------------------------
# Estimator instance (wrap factory)
# ------------------------------
def create_estimator():
    try:
        return EstimatorClass()
    except Exception as e:
        # fallback attempt
        try:
            return EstimatorClass()
        except Exception as e2:
            raise RuntimeError("Could not create estimator: " + str(e2))

estimator = create_estimator()

# ------------------------------
# measure_z_expectations_batch
# ------------------------------
def measure_z_expectations_batch(qc_list):
    """
    Batched measurement of single-qubit Z expectation values.
    Uses grouped warm call then per-qubit calls (keeps compatibility with your Estimator).
    Returns numpy array shape (n_circuits, latent_qubits)
    """
    if len(qc_list) == 0:
        return np.zeros((0, latent_qubits), dtype=float)

    n_circuits = len(qc_list)

    # Warm grouped call (may help backend cache)
    grouped_obs_list = [Z_GROUPED] * n_circuits
    job = estimator.run(circuits=qc_list, observables=grouped_obs_list)
    _ = job.result()

    # Then get per-qubit expectations
    values = np.zeros((n_circuits, latent_qubits), dtype=float)
    for q in range(latent_qubits):
        obs = z_ops[q]
        obs_list = [obs] * n_circuits
        job_q = estimator.run(circuits=qc_list, observables=obs_list)
        res_q = job_q.result()
        vals_q = np.array(res_q.values).reshape(-1)
        values[:, q] = vals_q
    return values

# ------------------------------
# QuantumVAE (uses measure function)
# ------------------------------
class QuantumVAE:
    def __init__(self, n_qubits):
        self.n_qubits = n_qubits
        self.encoder_params = np.random.randn(n_qubits, 2) * 0.05
        self.decoder_params = np.random.randn(n_qubits, 2) * 0.05

    def encode(self, x_batch, angles_cache=None):
        x_batch = x_batch.detach().cpu()
        b = x_batch.size(0)
        if angles_cache is None:
            feats = feature_extractor(x_batch.to(device)).detach().cpu().numpy()
            angles = (feats + 1.0) * (np.pi / 2.0)
        else:
            angles = angles_cache
        qc_list = [get_encoder_circuit_cached(self.n_qubits, self.encoder_params, angles[i]) for i in range(b)]
        vals = measure_z_expectations_batch(qc_list)
        mu_angles = np.arcsin(np.clip(vals, -1.0, 1.0))
        mu = torch.tensor(mu_angles, dtype=torch.float32).to(device)
        logvar = torch.zeros_like(mu).to(device)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        eps = torch.randn_like(mu)
        return mu + eps * torch.exp(0.5 * logvar)

    def decode(self, z_batch):
        z_np = z_batch.detach().cpu().numpy()
        qc_list = [get_decoder_circuit_cached(self.n_qubits, self.decoder_params, z_np[i]) for i in range(z_np.shape[0])]
        vals = measure_z_expectations_batch(qc_list)
        return torch.tensor(vals, dtype=torch.float32).to(device)

    def forward(self, x_batch, angles_cache=None):
        mu, logvar = self.encode(x_batch, angles_cache)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

qvae = QuantumVAE(latent_qubits)

# ------------------------------
# QuantumCNN wrapper (quantum features -> classical head)
# ------------------------------
class QuantumCNN(nn.Module):
    def __init__(self, n_qubits, n_classes=2):
        super().__init__()
        self.n_qubits = n_qubits
        self.clf_params = np.random.randn(n_qubits, 2) * 0.05
        self.fc = nn.Linear(n_qubits, n_classes).to(device)

    def quantum_features_batch(self, x_batch, angles_cache=None):
        x_batch = x_batch.detach().cpu()
        b = x_batch.size(0)
        if angles_cache is None:
            feats = feature_extractor(x_batch.to(device)).detach().cpu().numpy()
            angles = (feats + 1.0) * (np.pi / 2.0)
        else:
            angles = angles_cache
        qc_list = [get_classifier_circuit_cached(self.n_qubits, self.clf_params, angles[i]) for i in range(b)]
        vals = measure_z_expectations_batch(qc_list)
        return torch.tensor(vals, dtype=torch.float32).to(device)

    def forward(self, x_batch, angles_cache=None):
        feats_q = self.quantum_features_batch(x_batch, angles_cache)
        logits = self.fc(feats_q)
        return logits

qc_model = QuantumCNN(latent_qubits)

# ------------------------------
# Metrics & test (same as before)
# ------------------------------
def test_model(loader, model):
    model.eval()
    all_targets = []
    all_preds = []
    with torch.no_grad():
        for data, targets in loader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs, 1)
            all_targets.extend(targets.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
    cm = confusion_matrix(all_targets, all_preds, labels=[0,1])
    if cm.size == 4:
        tp0, fn0 = cm[0,0], cm[0,1]
        tp1, fn1 = cm[1,1], cm[1,0]
    else:
        tp0 = fn0 = tp1 = fn1 = 0
    tpr0 = tp0 / (tp0 + fn0) if (tp0 + fn0) > 0 else 0.0
    tpr1 = tp1 / (tp1 + fn1) if (tp1 + fn1) > 0 else 0.0
    acc = (np.array(all_preds) == np.array(all_targets)).mean() if len(all_targets) else 0.0
    print(f"Acc: {acc:.4f}, TPR0: {tpr0:.4f}, TPR1: {tpr1:.4f}")
    return tpr0, tpr1

# ------------------------------
# Adversarial utilities (latent)
# ------------------------------
def quantum_adversarial_latent(mu_pos, logvar_pos, A_mu, A_std):
    if isinstance(A_mu, np.ndarray):
        A_mu = torch.tensor(A_mu, dtype=torch.float32).to(device)
    if isinstance(A_std, np.ndarray):
        A_std = torch.tensor(A_std, dtype=torch.float32).to(device)
    mu_adv = mu_pos + A_mu
    logvar_adv = logvar_pos + A_std
    eps = torch.randn_like(mu_adv)
    z_adv = mu_adv + eps * torch.exp(0.5 * logvar_adv)
    return z_adv, mu_adv, logvar_adv

def calc_cost(a_mean, a_std):
    a_mean_t = torch.tensor(a_mean, dtype=torch.float32) if not isinstance(a_mean, torch.Tensor) else a_mean
    a_std_t = torch.tensor(a_std, dtype=torch.float32) if not isinstance(a_std, torch.Tensor) else a_std
    norm_mean = torch.norm(a_mean_t)
    norm_std = torch.norm(a_std_t)
    total_cost = (norm_mean / latent_qubits) + (norm_std / latent_qubits)
    return total_cost.item()

def adversary_payoff(posnegdata, posindices, negindices, mu_pos, std_pos, qvae_obj, a_mean, a_std, posnegtargets, model):
    """
    Compute payoff = 1 + error - cost
    error computed as 1 - TPR1 on the manipulated positives (compared to targets)
    posnegtargets: array/list of targets for combined data (1 for pos, 0 for neg)
    model: qc_model (assumed to use fc on latent z)
    """
    if mu_pos.dim() == 1:
        mu_pos = mu_pos.unsqueeze(0)
    if std_pos.dim() == 1:
        std_pos = std_pos.unsqueeze(0)
    # Build adversarial latents for the batch (mu_pos shape: [B, D])
    z_adv, _, _ = quantum_adversarial_latent(mu_pos, std_pos, a_mean, a_std)
    logits = model.fc(z_adv.to(device))
    _, preds = torch.max(logits, 1)
    targets = torch.tensor(posnegtargets, dtype=torch.long).to(device)
    true_positives = ((preds == 1) & (targets == 1)).sum().item()
    false_negatives = ((preds == 0) & (targets == 1)).sum().item()
    tpr1 = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0.0
    error = 1.0 - tpr1
    cost = calc_cost(a_mean, a_std)
    payoff = 1.0 + error - cost
    return payoff

# ------------------------------
# Simulated Annealing optimizer (replacement for PSO)
# ------------------------------
class SimulatedAnnealing:
    def __init__(self, fitness_fn, bounds,
                 max_iter=400,
                 T_start=2.5,
                 T_end=1e-3,
                 alpha=0.98,
                 step_scale=0.12):
        """
        fitness_fn: function taking vector x and returning scalar (to MINIMIZE)
        bounds: np.array shape (D,2)
        """
        self.fitness_fn = fitness_fn
        self.bounds = bounds
        self.max_iter = int(max_iter)
        self.T_start = float(T_start)
        self.T_end = float(T_end)
        self.alpha = float(alpha)
        self.step_scale = float(step_scale)

    def clip(self, x):
        return np.clip(x, self.bounds[:,0], self.bounds[:,1])

    def optimize(self):
        D = self.bounds.shape[0]
        x = np.random.uniform(self.bounds[:,0], self.bounds[:,1])
        fx = self.fitness_fn(x)

        best_x = x.copy()
        best_fx = fx
        T = self.T_start

        for it in range(self.max_iter):
            step = np.random.randn(D) * self.step_scale
            x_new = self.clip(x + step)
            fx_new = self.fitness_fn(x_new)

            dE = fx_new - fx
            if dE < 0:
                accept = True
            else:
                accept = np.random.rand() < np.exp(-dE / max(T, 1e-12))

            if accept:
                x = x_new
                fx = fx_new
                if fx < best_fx:
                    best_fx = fx
                    best_x = x.copy()

            T = max(self.T_end, T * self.alpha)

        return best_x, best_fx

# ------------------------------
# Alternating least squares adversarial game (uses SA instead of PSO)
# ------------------------------
def alternating_least_squares_game(qc_model, qvae_obj, train_images_all, train_labels_all,
                                   max_game_iter=2, pso_iter=100, n_pos_samples=80):
    """
    Note: pso_iter argument repurposed as SA iteration budget (max_iter for SA).
    """
    imgs = train_images_all
    labs = train_labels_all
    pos_mask = (labs == 1)
    neg_mask = (labs == 0)
    pos_imgs = imgs[pos_mask][:n_pos_samples].to(device)
    pos_labs = labs[pos_mask][:n_pos_samples].to(device)
    neg_imgs = imgs[neg_mask][:n_pos_samples].to(device)
    neg_labs = labs[neg_mask][:n_pos_samples].to(device)

    combined = torch.cat([pos_imgs, neg_imgs], dim=0)
    combined_targets = torch.cat([pos_labs, neg_labs], dim=0)
    pos_indices = (combined_targets.cpu().numpy() == 1).tolist()
    neg_indices = (combined_targets.cpu().numpy() == 0).tolist()

    A_mu = np.zeros(latent_qubits)
    A_std = np.zeros(latent_qubits)
    payoff_curr = -np.inf

    for game_iter in range(max_game_iter):
        # encode positives to get mu_pos and std
        with torch.no_grad():
            mu_pos_all, logvar_pos_all = qvae_obj.encode(pos_imgs)
        mu_pos_avg = mu_pos_all.mean(dim=0)
        std_pos_avg = logvar_pos_all.mean(dim=0)

        D = latent_qubits
        bounds = np.vstack([np.full(D, -0.5), np.full(D, 0.5)]).T

        # Fitness functions expect vectors; they MINIMIZE; adversary_payoff returns payoff (higher is better)
        # So fitness = -payoff
        def fit_mean(alpha_vec):
            return -adversary_payoff(combined, pos_indices, neg_indices, mu_pos_avg, std_pos_avg,
                                     qvae_obj, alpha_vec, A_std, combined_targets.cpu().numpy(), qc_model)

        sa_mean = SimulatedAnnealing(fit_mean, bounds,
                                     max_iter=pso_iter,
                                     T_start=2.0, T_end=1e-4, alpha=0.985, step_scale=0.12)
        alpha_star_mean, best_mean_val = sa_mean.optimize()

        def fit_std(alpha_vec):
            return -adversary_payoff(combined, pos_indices, neg_indices, mu_pos_avg, std_pos_avg,
                                     qvae_obj, alpha_star_mean, alpha_vec, combined_targets.cpu().numpy(), qc_model)

        sa_std = SimulatedAnnealing(fit_std, bounds,
                                    max_iter=pso_iter,
                                    T_start=2.0, T_end=1e-4, alpha=0.985, step_scale=0.12)
        alpha_star_std, best_std_val = sa_std.optimize()

        payoff_new = adversary_payoff(combined, pos_indices, neg_indices, mu_pos_avg, std_pos_avg,
                                      qvae_obj, alpha_star_mean, alpha_star_std, combined_targets.cpu().numpy(), qc_model)
        print(f"Game {game_iter}: payoff_curr={payoff_curr:.4f} payoff_new={payoff_new:.4f}")

        if payoff_new > payoff_curr:
            payoff_curr = payoff_new
            A_mu = alpha_star_mean.copy()
            A_std = alpha_star_std.copy()

            # build adversarial images from full mu_pos_all set
            z_adv, _, _ = quantum_adversarial_latent(mu_pos_all, logvar_pos_all, A_mu, A_std)
            adv_images = classical_decoder(z_adv.to(device))
            adv_targets = torch.ones(adv_images.size(0), dtype=torch.long).to(device)

            subset_size = min(500, len(train_images_all))
            train_data_subset = train_images_all[:subset_size].to(device)
            train_targets_subset = train_labels_all[:subset_size].to(device)
            combined_images = torch.cat([train_data_subset, adv_images], dim=0)
            combined_targets2 = torch.cat([train_targets_subset, adv_targets], dim=0)
            combined_ds = TensorDataset(combined_images, combined_targets2)
            combined_loader = DataLoader(combined_ds, batch_size=32, shuffle=True)

            qc_model.train()
            optimizer = torch.optim.Adam(qc_model.fc.parameters(), lr=1e-3)
            loss_fn = nn.CrossEntropyLoss()
            for epoch in range(3):
                run_loss = 0.0
                for bimgs, btargets in combined_loader:
                    bimgs = bimgs.to(device); btargets = btargets.to(device)
                    with torch.no_grad():
                        mu_b, logvar_b = qvae_obj.encode(bimgs)
                        z_b = qvae_obj.reparameterize(mu_b, logvar_b)
                    logits = qc_model.fc(z_b.to(device))
                    loss = loss_fn(logits, btargets)
                    optimizer.zero_grad(); loss.backward(); optimizer.step()
                    run_loss += loss.item()
                print(f" Retrain epoch {epoch+1}, loss {run_loss/len(combined_loader):.4f}")
        else:
            print("No improvement — stopping.")
            break

    return A_mu, A_std, payoff_curr

# End of Part 2/5
# ---------------------------------------------------------
# Part 3/5
# Parameter-shift helpers, hard defense, main demo + alpha★ eval
# ---------------------------------------------------------

# ------------------------------
# Parameter-shift helpers (quantum finetuning)
# ------------------------------
def compute_features_and_grads_batch_for_clf_params(x_batch, clf_params_np, n_qubits):
    with torch.no_grad():
        feats_classical = feature_extractor(x_batch.to(device)).detach().cpu().numpy()
    angles = (feats_classical + 1.0) * (np.pi / 2.0)
    qc_list = [get_classifier_circuit_cached(n_qubits, clf_params_np, angles[i]) for i in range(angles.shape[0])]
    vals = measure_z_expectations_batch(qc_list)
    feats_tensor = torch.tensor(vals, dtype=torch.float32).to(device)
    return feats_tensor, angles

def parameter_shift_update(clf_params_np, x_batch, batch_labels, n_qubits,
                           qc_model, lr_q=1e-2, shift=np.pi/2):
    feats_base, angles = compute_features_and_grads_batch_for_clf_params(x_batch, clf_params_np, n_qubits)
    feats_base = feats_base.clone().detach().requires_grad_(True)
    logits = qc_model.fc(feats_base)
    loss_fn = nn.CrossEntropyLoss()
    loss = loss_fn(logits, batch_labels.to(device))
    qc_model.fc.zero_grad()
    if feats_base.grad is not None:
        feats_base.grad.zero_()
    loss.backward(retain_graph=True)
    dL_dfeat = feats_base.grad.detach().cpu().numpy()  # (B, D)
    B = dL_dfeat.shape[0]
    grads_theta = np.zeros_like(clf_params_np, dtype=float)
    for q in range(n_qubits):
        for p_idx in range(2):
            clf_plus = clf_params_np.copy()
            clf_minus = clf_params_np.copy()
            clf_plus[q, p_idx] += shift
            clf_minus[q, p_idx] -= shift
            f_plus_tensor, _ = compute_features_and_grads_batch_for_clf_params(x_batch, clf_plus, n_qubits)
            f_minus_tensor, _ = compute_features_and_grads_batch_for_clf_params(x_batch, clf_minus, n_qubits)
            f_plus = f_plus_tensor.detach().cpu().numpy()
            f_minus = f_minus_tensor.detach().cpu().numpy()
            df_dtheta = 0.5 * (f_plus - f_minus)
            term = np.sum(dL_dfeat * df_dtheta)
            grads_theta[q, p_idx] = term / float(B)
    clf_params_np = clf_params_np - lr_q * grads_theta
    return clf_params_np, loss.item()

# ------------------------------
# Hard defense + quantum finetune
# ------------------------------
def hard_defense_with_quantum_finetune(qc_model, qvae, feature_extractor, classical_decoder,
                                       train_images_all, train_labels_all,
                                       val_loader=None,
                                       epochs=4,
                                       adv_samples_per_pos=8,
                                       adv_radius=0.5,
                                       batch_size_def=64,
                                       collapse_penalty_coef=12.0,
                                       lr=5e-4,
                                       subset_size=3000,
                                       fine_tune_quantum=True,
                                       lr_q=1e-2,
                                       quantum_batches_per_epoch=3,
                                       paramshift_shift=math.pi/2):
    feature_extractor.train(); qc_model.train()
    params = list(feature_extractor.parameters()) + list(qc_model.fc.parameters())
    optimizer = torch.optim.Adam(params, lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    device_local = device
    total_clean = min(subset_size, len(train_images_all))
    clean_images = train_images_all[:total_clean].to(device_local)
    clean_labels = train_labels_all[:total_clean].to(device_local)
    pos_mask = (train_labels_all == 1)
    pos_imgs_all = train_images_all[pos_mask]
    if len(pos_imgs_all) == 0:
        raise RuntimeError("No positive-class examples found.")
    pos_sample_count = min(200, len(pos_imgs_all))
    pos_imgs = pos_imgs_all[:pos_sample_count].to(device_local)
    clf_params_np = qc_model.clf_params.copy()
    for epoch in range(epochs):
        epoch_loss = 0.0; n_batches = 0; quantum_updates_done = 0
        with torch.no_grad():
            mu_pos_all, logvar_pos_all = qvae.encode(pos_imgs)
            mu_pos_rep = mu_pos_all.repeat_interleave(adv_samples_per_pos, dim=0)
            logvar_rep = logvar_pos_all.repeat_interleave(adv_samples_per_pos, dim=0)
            delta = torch.randn_like(mu_pos_rep, device=device_local) * adv_radius
            std_delta = torch.randn_like(logvar_rep, device=device_local) * (adv_radius * 0.3)
            mu_adv = mu_pos_rep + delta
            std_adv = torch.clamp(logvar_rep + std_delta, min=-3.0, max=3.0)
            eps = torch.randn_like(mu_adv)
            z_adv = mu_adv + eps * torch.exp(0.5 * std_adv)
            adv_images = classical_decoder(z_adv.to(device_local))
            adv_labels = torch.ones(adv_images.size(0), dtype=torch.long, device=device_local)
        idxs = torch.randperm(clean_images.size(0))
        clean_shuffled = clean_images[idxs]; clean_shuffled_labels = clean_labels[idxs]
        combined_images = torch.cat([clean_shuffled, adv_images], dim=0)
        combined_labels = torch.cat([clean_shuffled_labels, adv_labels], dim=0)
        combined_ds = TensorDataset(combined_images, combined_labels)
        combined_loader = DataLoader(combined_ds, batch_size=batch_size_def, shuffle=True)
        for batch_idx, (batch_imgs, batch_labels) in enumerate(combined_loader):
            batch_imgs = batch_imgs.to(device_local); batch_labels = batch_labels.to(device_local)
            feats = feature_extractor(batch_imgs)
            logits = qc_model.fc(feats)
            ce_loss = loss_fn(logits, batch_labels)
            probs1 = torch.softmax(logits, dim=1)[:, 1]
            mean_p1 = probs1.mean()
            collapse_penalty = collapse_penalty_coef * (mean_p1 - 0.5) ** 2
            l2_reg = 0.0
            for p in params:
                l2_reg += 1e-4 * torch.sum(p ** 2)
            loss = ce_loss + collapse_penalty + l2_reg
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            epoch_loss += loss.item(); n_batches += 1
            if fine_tune_quantum and (quantum_updates_done < quantum_batches_per_epoch):
                try:
                    clf_params_np, qloss = parameter_shift_update(clf_params_np, batch_imgs, batch_labels, latent_qubits, qc_model, lr_q=lr_q, shift=paramshift_shift)
                    qc_model.clf_params = clf_params_np.copy()
                    quantum_updates_done += 1
                except Exception as e:
                    print("Parameter-shift update error:", e)
        avg_loss = epoch_loss / max(1, n_batches)
        print(f"[HardDef+QEpoch {epoch+1}/{epochs}] Avg loss: {avg_loss:.4f}, quantum_updates: {quantum_updates_done}")
        if val_loader is not None:
            print("Validation eval:")
            test_model(val_loader, qc_model)
    qc_model.clf_params = clf_params_np.copy()
    print("Hard defense + quantum fine-tune complete.")

# ------------------------------
# Demo run: adversarial game then hard defense + quantum finetune
# ------------------------------
if __name__ == "__main__":
    print("Running adversarial ALS game (short demo) with Simulated Annealing adversary:")
    start = time.time()
    A_mu_final, A_std_final, final_payoff = alternating_least_squares_game(qc_model, qvae, train_images_all, train_labels_all,
                                                                           max_game_iter=2, pso_iter=150, n_pos_samples=60)
    print("Game finished in {:.1f}s".format(time.time() - start))
    print("A_mu:", A_mu_final)
    print("A_std:", A_std_final)
    print("Final payoff:", final_payoff)

    print("Test eval BEFORE defense (baseline):")
    baseline_tpr0, baseline_tpr1 = test_model(test_loader, qc_model)

    # ------------------------------
    # ALPHA★ POST-GAME ATTACK EVALUATION
    # ------------------------------
    print("\n=== ALPHA★ POST-GAME ATTACK EVALUATION ===")
    pos_mask_test = (test_labels_all == 1)
    pos_imgs_test = test_images_all[pos_mask_test]
    if pos_imgs_test.size(0) == 0:
        print("No positive test samples found; skipping alpha-star attack evaluation.")
        tpr1_attack = None; asr = None; acc_recon = None
    else:
        n_attack_samples = min(120, pos_imgs_test.size(0))
        pos_imgs_test = pos_imgs_test[:n_attack_samples].to(device)

        with torch.no_grad():
            mu_pos_test, logvar_pos_test = qvae.encode(pos_imgs_test)

        z_star, mu_star, logvar_star = quantum_adversarial_latent(mu_pos_test, logvar_pos_test, A_mu_final, A_std_final)

        qc_model.eval()
        with torch.no_grad():
            logits_star = qc_model.fc(z_star.to(device))
            _, preds_star = torch.max(logits_star, 1)
            true_labels_star = torch.ones_like(preds_star)
            tp1 = ((preds_star == 1) & (true_labels_star == 1)).sum().item()
            fn1 = ((preds_star == 0) & (true_labels_star == 1)).sum().item()
            tpr1_attack = tp1 / (tp1 + fn1 + 1e-12)
            asr = 1.0 - tpr1_attack

        print("\n--- Alpha★ Attack Results (latent eval) ---")
        print(f"Num attack samples: {n_attack_samples}")
        print(f"TPR1 on alpha★ manipulated positives (latent): {tpr1_attack:.4f}")
        print(f"Attack Success Rate (ASR): {asr:.4f}")
        print(f"Payoff at convergence: {final_payoff:.4f}")

        with torch.no_grad():
            recon_star = classical_decoder(z_star.to(device))
            mu_re, logvar_re = qvae.encode(recon_star)
            z_re = qvae.reparameterize(mu_re, logvar_re)
            logits_re = qc_model.fc(z_re.to(device))
            _, preds_re = torch.max(logits_re, 1)
            acc_recon = (preds_re == torch.ones_like(preds_re)).float().mean().item()

        print("\nClassifier performance on reconstructed alpha★ images (pixel-space):")
        print(f"Accuracy on reconstructions: {acc_recon:.4f}")

    # Save baseline and attack metrics
    metrics = {
        "baseline_tpr0": float(baseline_tpr0),
        "baseline_tpr1": float(baseline_tpr1),
        "A_mu_final": A_mu_final.tolist() if isinstance(A_mu_final, np.ndarray) else A_mu_final,
        "A_std_final": A_std_final.tolist() if isinstance(A_std_final, np.ndarray) else A_std_final,
        "final_payoff": float(final_payoff),
        "tpr1_attack": (float(tpr1_attack) if tpr1_attack is not None else None),
        "asr": (float(asr) if asr is not None else None),
        "acc_recon": (float(acc_recon) if 'acc_recon' in locals() else None)
    }
    import json
    with open("metrics_alpha_star_SA2.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print("Saved metrics -> metrics_alpha_star_SA2.json")

    # ------------------------------
    # Proceed to hard defense + quantum finetune
    # ------------------------------
    print("\nStarting hard defense + quantum fine-tuning (this is expensive):")
    hard_defense_with_quantum_finetune(qc_model, qvae, feature_extractor, classical_decoder,
                                       train_images_all, train_labels_all,
                                       val_loader=val_loader,
                                       epochs=3,
                                       adv_samples_per_pos=8,
                                       adv_radius=0.5,
                                       batch_size_def=64,
                                       collapse_penalty_coef=12.0,
                                       lr=5e-4,
                                       subset_size=2000,
                                       fine_tune_quantum=True,
                                       lr_q=1e-2,
                                       quantum_batches_per_epoch=2,
                                       paramshift_shift=math.pi/2)

    print("Final evaluation on test set AFTER defense:")
    test_model(test_loader, qc_model)

    # ------------------------------
    # Save artifacts
    # ------------------------------
    np.save("qvae_encoder_params_SA2.npy", qvae.encoder_params)
    np.save("qvae_decoder_params_SA2.npy", qvae.decoder_params)
    np.save("qcnn_clf_params_SA2.npy", qc_model.clf_params)
    torch.save(qc_model.fc.state_dict(), "qcnn_fc_head_SA2.pth")
    print("Saved parameters. Done.")

# ---------------------------------------------------------
# Part 4/5
# Additional utilities, numerics, profiling toggles
# ---------------------------------------------------------

# Optional: Stable exponential for annealing acceptance
def safe_exp(x):
    """
    Numerically stable exp for large negative or small values.
    Clips exponent to avoid overflow.
    """
    x = np.clip(x, -50, 50)  # avoid large overflow
    return np.exp(x)

# Optional: Mini profiler toggle
ENABLE_PROFILING = False

class Timer:
    """
    Simple wall-clock profiler context manager.
    Usage:
        with Timer("Encoding"):
            mu, logvar = qvae.encode(x)
    """
    def __init__(self, name="timer"):
        self.name = name
    def __enter__(self):
        if ENABLE_PROFILING:
            self.start = time.time()
        return self
    def __exit__(self, exc_type, exc, tb):
        if ENABLE_PROFILING:
            dt = time.time() - self.start
            print(f"[PROFILE] {self.name}: {dt:.4f}s")

# -----------------------------------
# Enhanced debug print (optional)
# -----------------------------------
DEBUG = False

def debug(*args):
    """
    Print only when debugging is enabled.
    """
    if DEBUG:
        print("[DEBUG]", *args)

# -----------------------------------
# Gradient clipping helper
# -----------------------------------
def clip_gradients(model, clip_value=5.0):
    """
    Clip gradients of all parameters to avoid exploding gradient issues.
    """
    for p in model.parameters():
        if p.grad is not None:
            p.grad.data.clamp_(-clip_value, clip_value)

# -----------------------------------
# Angle sanitization helper
# -----------------------------------
def sanitize_angles(a):
    """
    Ensures numerical finiteness of angle vectors.
    """
    a = np.array(a, dtype=float)
    a[~np.isfinite(a)] = 0.0
    a = np.clip(a, -10*np.pi, 10*np.pi)
    return a

# -----------------------------------
# Robust z reparameterization
# -----------------------------------
def safe_reparameterize(mu, logvar):
    """
    VAE reparameterization with numeric safety.
    """
    with torch.no_grad():
        eps = torch.randn_like(mu)
        logvar = torch.clamp(logvar, -10.0, 10.0)
        return mu + eps * torch.exp(0.5 * logvar)

# Patch qvae reparameterization
QuantumVAE.reparameterize = lambda self, mu, logvar: safe_reparameterize(mu, logvar)

# -----------------------------------
# Debug printing for circuits
# -----------------------------------
def print_circuit_stats(qc, label="QC"):
    """
    Print number of gates, depth, etc.
    Useful when debugging circuit bloat.
    """
    if not DEBUG:
        return
    print(f"[{label}] depth={qc.depth()}, size={qc.size()}, ops={qc.count_ops()}")

# If necessary, plug this into circuit caching to analyze circuits
old_get_encoder = get_encoder_circuit_cached
def get_encoder_circuit_cached_debug(n_qubits, encoder_params, x_angles):
    qc = old_get_encoder(n_qubits, encoder_params, x_angles)
    print_circuit_stats(qc, label="Encoder")
    return qc

# Uncomment if you want verbose circuit debugging
# get_encoder_circuit_cached = get_encoder_circuit_cached_debug

# ---------------------------------------------------------
# Part 5/5
# Final integrity checks and ready-to-run bootstrap
# ---------------------------------------------------------

# =========================================================
# FINAL RUNTIME CHECKS (highly recommended)
# =========================================================

def check_environment():
    print("\n=== Runtime Environment Check ===")
    print("Torch:", torch.__version__)
    try:
        import qiskit
        print("Qiskit:", qiskit.__version__)
    except:
        print("Qiskit version UNKNOWN")

    try:
        import qiskit_aer
        print("Qiskit Aer:", qiskit_aer.__version__)
    except:
        print("Qiskit Aer NOT FOUND (using StatevectorEstimator instead).")

    if torch.cuda.is_available():
        print("CUDA available ✔")
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available ✘ — running on CPU")

    print("Device selected:", device)
    print("=================================")

def check_qiskit_primitives():
    """
    Confirms EstimatorClass is valid and works for a trivial test circuit.
    Avoids runtime crashes deep inside training loops.
    """
    print("\n=== Qiskit Primitive Self-Test ===")
    try:
        est = EstimatorClass()
        qc = QuantumCircuit(1)
        qc.h(0)
        obs = SparsePauliOp("Z")
        job = est.run([qc], [obs])
        res = job.result()
        print("Estimator primitive OK. Test value:", res.values)
    except Exception as e:
        print("ERROR: Estimator primitive failed:", e)
        raise RuntimeError("Estimator primitive is not functional.")
    print("===================================")

def check_dataset_shapes(train_images_all, train_labels_all):
    print("\n=== Dataset Sanity Check ===")
    print("Train images:", train_images_all.shape)
    print("Train labels:", train_labels_all.shape)
    n0 = (train_labels_all == 0).sum().item()
    n1 = (train_labels_all == 1).sum().item()
    print("Class-0 count:", n0)
    print("Class-1 count:", n1)
    if n0 == 0 or n1 == 0:
        raise RuntimeError("One class has zero samples — cannot train binary classifier.")
    print("===================================")


# =========================================================
# BOOSTRAP LOGGER
# =========================================================

def print_banner():
    print(r"""
===========================================================
       QUANTUM ADVERSARIAL GAME (SA-2 OPTIMIZER)
===========================================================
A_mu / A_std optimized via *Simulated Annealing*, not PSO.
Full pipeline:
  ✔ QVAE (encoder/decoder)
  ✔ Quantum CNN classifier
  ✔ Alternating Least Squares adversarial game
  ✔ Simulated Annealing adversarial optimizer
  ✔ α★ post-game attack evaluation
  ✔ Hard defense + quantum fine-tuning
===========================================================
""")

# =========================================================
# SANITY: RUN CHECKS AT IMPORT TIME
# =========================================================
print_banner()
check_environment()
check_qiskit_primitives()
check_dataset_shapes(train_images_all, train_labels_all)

print("\nSystem checks PASSED. Ready to run main loop.\n")

# =========================================================
# EVERYTHING IS WIRED. SCRIPT IS READY.
# =========================================================
# From here, execution flows into the "if __name__ == '__main__'" block in Part 3.


Using qiskit_aer Estimator.
Device: cpu
Loaded MNIST via torchvision.
Datasets prepared.


C:\Users\Aaditya Rajput\AppData\Local\Temp\ipykernel_75124\4189215908.py:220: DeprecationWarning: Estimator has been deprecated as of Aer 0.15, please use EstimatorV2 instead.
  estimator = create_estimator()
C:\Users\Aaditya Rajput\AppData\Local\Temp\ipykernel_75124\4189215908.py:220: DeprecationWarning: Option approximation=False is deprecated as of qiskit-aer 0.13. It will be removed no earlier than 3 months after the release date. Instead, use BackendEstimator from qiskit.primitives.
  estimator = create_estimator()


Running adversarial ALS game (short demo) with Simulated Annealing adversary:
Game 0: payoff_curr=-inf payoff_new=0.8578
 Retrain epoch 1, loss 0.7920
 Retrain epoch 2, loss 0.7839
 Retrain epoch 3, loss 0.7581
Game 1: payoff_curr=0.8578 payoff_new=1.8461
 Retrain epoch 1, loss 0.7515
 Retrain epoch 2, loss 0.7855
 Retrain epoch 3, loss 0.7408
Game finished in 388.5s
A_mu: [ 0.44650771  0.1044491   0.03390839  0.01202229  0.22643739  0.06672995
  0.12284277 -0.5       ]
A_std: [-0.13134412 -0.30664809  0.11695587 -0.22522697 -0.24924806  0.04697888
  0.09779013 -0.04682375]
Final payoff: 1.8460718989372253
Test eval BEFORE defense (baseline):
Acc: 0.4959, TPR0: 1.0000, TPR1: 0.0000

=== ALPHA★ POST-GAME ATTACK EVALUATION ===

--- Alpha★ Attack Results (latent eval) ---
Num attack samples: 120
TPR1 on alpha★ manipulated positives (latent): 0.3000
Attack Success Rate (ASR): 0.7000
Payoff at convergence: 1.8461

Classifier performance on reconstructed alpha★ images (pixel-space):
Accuracy

C:\Users\Aaditya Rajput\AppData\Local\Temp\ipykernel_75124\4189215908.py:984: DeprecationWarning: Estimator has been deprecated as of Aer 0.15, please use EstimatorV2 instead.
  check_qiskit_primitives()
C:\Users\Aaditya Rajput\AppData\Local\Temp\ipykernel_75124\4189215908.py:984: DeprecationWarning: Option approximation=False is deprecated as of qiskit-aer 0.13. It will be removed no earlier than 3 months after the release date. Instead, use BackendEstimator from qiskit.primitives.
  check_qiskit_primitives()


Estimator primitive OK. Test value: [-0.0078125]

=== Dataset Sanity Check ===
Train images: torch.Size([11769, 1, 28, 28])
Train labels: torch.Size([11769])
Class-0 count: 5918
Class-1 count: 5851

System checks PASSED. Ready to run main loop.



In [1]:
# quantum_adversarial_game_qpso_allclasses.py
# ALS + QPSO adversarial example generation for ALL MNIST classes (0..9)
# Produces adversarial images per source digit and saves grids to ./adv_examples/

import os, time, math, hashlib, random, json
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split, ConcatDataset
from sklearn.metrics import confusion_matrix
from PIL import Image

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Prefer Aer Estimator (fast) if available; fallback to StatevectorEstimator
try:
    from qiskit_aer.primitives import Estimator as AerEstimator
    EstimatorClass = AerEstimator
    print("Using qiskit_aer Estimator (fast).")
except Exception:
    try:
        from qiskit.primitives import StatevectorEstimator as EstimatorClass
        print("Using qiskit.primitives.StatevectorEstimator (fallback).")
    except Exception:
        EstimatorClass = None
        print("Warning: No suitable Qiskit estimator found. EstimatorClass=None")

# ------------------------------
# Settings
# ------------------------------
latent_qubits = 8   # latent dim (maps to number of qubits)
batch_size = 64
seed = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

# ------------------------------
# MNIST full loader (0..9)
# ------------------------------
def load_mnist_full(batch_size=64, data_root="./data"):
    try:
        from torchvision import datasets, transforms
        transform = transforms.Compose([transforms.ToTensor()])
        train_full = datasets.MNIST(root=data_root, train=True, download=True, transform=transform)
        test_full  = datasets.MNIST(root=data_root, train=False, download=True, transform=transform)
        train_images = train_full.data.unsqueeze(1).float() / 255.0
        train_labels = train_full.targets.long()
        test_images = test_full.data.unsqueeze(1).float() / 255.0
        test_labels = test_full.targets.long()
        print("Loaded MNIST via torchvision.")
    except Exception as e:
        print("torchvision failed, falling back to sklearn:", e)
        from sklearn.datasets import fetch_openml
        mn = fetch_openml('mnist_784', version=1, as_frame=False)
        X = mn['data'].astype('float32') / 255.0
        y = mn['target'].astype(int)
        X = X.reshape(-1, 1, 28, 28)
        train_images, test_images = X[:60000], X[60000:]
        train_labels, test_labels = y[:60000], y[60000:]
        train_images = torch.from_numpy(train_images)
        train_labels = torch.from_numpy(train_labels).long()
        test_images = torch.from_numpy(test_images)
        test_labels = torch.from_numpy(test_labels).long()
        print("Loaded MNIST via sklearn.")

    # We will not filter — keep full dataset
    total_train = TensorDataset(train_images, train_labels)
    train_size = int(0.9 * len(total_train))
    val_size = len(total_train) - train_size
    train_dataset, val_dataset = random_split(total_train, [train_size, val_size])
    test_dataset = TensorDataset(test_images, test_labels)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print("Datasets prepared: Train", len(train_dataset), "Val", len(val_dataset), "Test", len(test_dataset))
    return train_loader, val_loader, test_loader, train_images, train_labels, test_images, test_labels

train_loader, val_loader, test_loader, train_images_all, train_labels_all, test_images_all, test_labels_all = \
    load_mnist_full(batch_size=batch_size)

# ------------------------------
# Classical feature extractor + decoder
# ------------------------------
input_dim = 28 * 28

class ClassicalFeatureExtractor(nn.Module):
    def __init__(self, latent_qubits):
        super().__init__()
        self.fc = nn.Linear(input_dim, latent_qubits)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return torch.tanh(self.fc(x))

class ClassicalDecoder(nn.Module):
    def __init__(self, latent_qubits):
        super().__init__()
        self.fc1 = nn.Linear(latent_qubits, 128)
        self.fc2 = nn.Linear(128, input_dim)
    def forward(self, z):
        z = F.relu(self.fc1(z))
        out = torch.sigmoid(self.fc2(z))
        return out.view(-1,1,28,28)

feature_extractor = ClassicalFeatureExtractor(latent_qubits).to(device)
classical_decoder = ClassicalDecoder(latent_qubits).to(device)

# ------------------------------
# Circuit caching utilities
# ------------------------------
CIRCUIT_CACHE = {}
def _params_to_list(params):
    if isinstance(params, np.ndarray):
        return params.flatten().tolist()
    elif isinstance(params, (list, tuple)):
        return np.array(params).flatten().tolist()
    else:
        return np.array(params).flatten().tolist()

def cache_key(type_name, params, angles):
    p_list = _params_to_list(params)
    a_list = np.round(np.array(angles).flatten(), 6).tolist()
    raw = f"{type_name}|{p_list}|{a_list}"
    return hashlib.sha256(raw.encode()).hexdigest()

def get_encoder_circuit_cached(n_qubits, encoder_params, x_angles):
    key = cache_key("enc", encoder_params, x_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(x_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(encoder_params[i,0]), i)
        qc.rz(float(encoder_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

def get_decoder_circuit_cached(n_qubits, decoder_params, z_angles):
    key = cache_key("dec", decoder_params, z_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(z_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(decoder_params[i,0]), i)
        qc.rz(float(decoder_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

def get_classifier_circuit_cached(n_qubits, clf_params, x_angles):
    key = cache_key("clf", clf_params, x_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(x_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(clf_params[i,0]), i)
        qc.rz(float(clf_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

# ------------------------------
# Observables: Z individual + grouped
# ------------------------------
z_ops = []
for i in range(latent_qubits):
    s = ['I'] * latent_qubits
    s[latent_qubits - 1 - i] = 'Z'
    z_ops.append(SparsePauliOp(''.join(s)))
Z_GROUPED = z_ops[0]
for i in range(1, latent_qubits):
    Z_GROUPED = Z_GROUPED + z_ops[i]

# ------------------------------
# Estimator instance (wrap factory)
# ------------------------------
def create_estimator():
    if EstimatorClass is None:
        raise RuntimeError("EstimatorClass is not available in this environment.")
    try:
        return EstimatorClass()
    except Exception as e:
        try:
            return EstimatorClass()
        except Exception as e2:
            raise RuntimeError("Could not create estimator: " + str(e2))

estimator = None
try:
    estimator = create_estimator()
except Exception as e:
    print("Estimator creation failed:", e)
    estimator = None

# ------------------------------
# measure_z_expectations_batch
# ------------------------------
def measure_z_expectations_batch(qc_list):
    if len(qc_list) == 0:
        return np.zeros((0, latent_qubits), dtype=float)
    if estimator is None:
        raise RuntimeError("Estimator not available for measuring circuits.")
    n_circuits = len(qc_list)
    grouped_obs_list = [Z_GROUPED] * n_circuits
    job = estimator.run(circuits=qc_list, observables=grouped_obs_list)
    _ = job.result()
    values = np.zeros((n_circuits, latent_qubits), dtype=float)
    for q in range(latent_qubits):
        obs = z_ops[q]
        obs_list = [obs] * n_circuits
        job_q = estimator.run(circuits=qc_list, observables=obs_list)
        res_q = job_q.result()
        vals_q = np.array(res_q.values).reshape(-1)
        values[:, q] = vals_q
    return values

# ------------------------------
# QuantumVAE
# ------------------------------
class QuantumVAE:
    def __init__(self, n_qubits):
        self.n_qubits = n_qubits
        self.encoder_params = np.random.randn(n_qubits, 2) * 0.05
        self.decoder_params = np.random.randn(n_qubits, 2) * 0.05

    def encode(self, x_batch, angles_cache=None):
        x_batch = x_batch.detach().cpu()
        b = x_batch.size(0)
        if angles_cache is None:
            feats = feature_extractor(x_batch.to(device)).detach().cpu().numpy()
            angles = (feats + 1.0) * (np.pi / 2.0)
        else:
            angles = angles_cache
        qc_list = [get_encoder_circuit_cached(self.n_qubits, self.encoder_params, angles[i]) for i in range(b)]
        vals = measure_z_expectations_batch(qc_list)
        mu_angles = np.arcsin(np.clip(vals, -1.0, 1.0))
        mu = torch.tensor(mu_angles, dtype=torch.float32).to(device)
        logvar = torch.zeros_like(mu).to(device)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        eps = torch.randn_like(mu)
        return mu + eps * torch.exp(0.5 * logvar)

    def decode(self, z_batch):
        z_np = z_batch.detach().cpu().numpy()
        qc_list = [get_decoder_circuit_cached(self.n_qubits, self.decoder_params, z_np[i]) for i in range(z_np.shape[0])]
        vals = measure_z_expectations_batch(qc_list)
        return torch.tensor(vals, dtype=torch.float32).to(device)

    def forward(self, x_batch, angles_cache=None):
        mu, logvar = self.encode(x_batch, angles_cache)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

qvae = QuantumVAE(latent_qubits)

# ------------------------------
# QuantumCNN wrapper
# ------------------------------
class QuantumCNN(nn.Module):
    def __init__(self, n_qubits, n_classes=2):
        super().__init__()
        self.n_qubits = n_qubits
        self.clf_params = np.random.randn(n_qubits, 2) * 0.05
        self.fc = nn.Linear(n_qubits, n_classes).to(device)

    def quantum_features_batch(self, x_batch, angles_cache=None):
        x_batch = x_batch.detach().cpu()
        b = x_batch.size(0)
        if angles_cache is None:
            feats = feature_extractor(x_batch.to(device)).detach().cpu().numpy()
            angles = (feats + 1.0) * (np.pi / 2.0)
        else:
            angles = angles_cache
        qc_list = [get_classifier_circuit_cached(self.n_qubits, self.clf_params, angles[i]) for i in range(b)]
        vals = measure_z_expectations_batch(qc_list)
        return torch.tensor(vals, dtype=torch.float32).to(device)

    def forward(self, x_batch, angles_cache=None):
        feats_q = self.quantum_features_batch(x_batch, angles_cache)
        logits = self.fc(feats_q)
        return logits

# We'll build a multiclass wrapper that maps quantum features -> 10-class head
qc_model = QuantumCNN(latent_qubits, n_classes=10)

# ------------------------------
# Metrics & test
# ------------------------------
def test_model(loader, model):
    model.eval()
    all_targets = []
    all_preds = []
    with torch.no_grad():
        for data, targets in loader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs, 1)
            all_targets.extend(targets.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
    cm = confusion_matrix(all_targets, all_preds, labels=list(range(10)))
    acc = (np.array(all_preds) == np.array(all_targets)).mean() if len(all_targets) else 0.0
    # per-class TPR
    tprs = []
    for c in range(10):
        tp = cm[c,c]
        fn = cm[c,:].sum() - tp
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        tprs.append(tpr)
    print(f"Acc: {acc:.4f}, per-class TPR: {[round(t,4) for t in tprs]}")
    return acc, tprs

# ------------------------------
# Adversarial utilities (latent)
# ------------------------------
def quantum_adversarial_latent(mu_pos, logvar_pos, A_mu, A_std):
    if isinstance(A_mu, np.ndarray):
        A_mu = torch.tensor(A_mu, dtype=torch.float32).to(device)
    if isinstance(A_std, np.ndarray):
        A_std = torch.tensor(A_std, dtype=torch.float32).to(device)
    mu_adv = mu_pos + A_mu
    logvar_adv = logvar_pos + A_std
    eps = torch.randn_like(mu_adv)
    z_adv = mu_adv + eps * torch.exp(0.5 * logvar_adv)
    return z_adv, mu_adv, logvar_adv

def calc_cost(a_mean, a_std):
    a_mean_t = torch.tensor(a_mean, dtype=torch.float32) if not isinstance(a_mean, torch.Tensor) else a_mean
    a_std_t = torch.tensor(a_std, dtype=torch.float32) if not isinstance(a_std, torch.Tensor) else a_std
    norm_mean = torch.norm(a_mean_t)
    norm_std = torch.norm(a_std_t)
    total_cost = (norm_mean / latent_qubits) + (norm_std / latent_qubits)
    return total_cost.item()

def adversary_payoff(posnegdata, posindices, negindices, mu_pos, std_pos, qvae_obj, a_mean, a_std, posnegtargets, model):
    # posnegtargets: integer targets array with 1 for pos and 0 for neg (when binary)
    if mu_pos.dim() == 1:
        mu_pos = mu_pos.unsqueeze(0)
    if std_pos.dim() == 1:
        std_pos = std_pos.unsqueeze(0)
    z_adv, _, _ = quantum_adversarial_latent(mu_pos, std_pos, a_mean, a_std)
    # For one-vs-rest: we check whether logits predict the source class index
    logits = model.fc(z_adv.to(device)) if hasattr(model, 'fc') else model(z_adv.to(device))
    _, preds = torch.max(logits, 1)
    # If posnegtargets is binary mask, treat 1 as positive; here for one-vs-rest, we compute TPR for class==source
    # posnegtargets contains true labels (0..9) in our one-vs-rest caller; we'll compute TPR for source label externally.
    # To keep API simple, assume posnegtargets are true labels and source_label is included in closure.
    targets = torch.tensor(posnegtargets, dtype=torch.long).to(device)
    # compute TPR for positives (where targets == source_label)
    src_label = int(targets.max().item()) if False else None  # placeholder; we'll compute wrapper-specific
    # In ALS QPSO calls we will wrap this function appropriately. For safety, fallback:
    true_positives = ((preds == targets) & (targets >= 0)).sum().item()  # generic
    false_negatives = ((preds != targets) & (targets >= 0)).sum().item()
    tpr1 = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0.0
    error = 1.0 - tpr1
    cost = calc_cost(a_mean, a_std)
    payoff = 1.0 + error - cost
    return payoff

# ------------------------------
# QPSO optimizer (replaces PSO / SA)
# ------------------------------
class QPSO:
    def __init__(self, fitness_fn, bounds, num_particles=30, max_iter=300,
                 alpha_start=1.0, alpha_end=0.1, seed=None, verbose=False):
        self.fitness_fn = fitness_fn
        self.bounds = np.array(bounds, dtype=float)
        assert self.bounds.ndim == 2 and self.bounds.shape[1] == 2
        self.D = self.bounds.shape[0]
        self.num_particles = int(num_particles)
        self.max_iter = int(max_iter)
        self.alpha_start = float(alpha_start)
        self.alpha_end = float(alpha_end)
        self.verbose = bool(verbose)
        self.rng = np.random.RandomState(seed)
        self.positions = self.rng.uniform(self.bounds[:,0], self.bounds[:,1], size=(self.num_particles, self.D))
        self.pbest_pos = self.positions.copy()
        self.pbest_val = np.array([self._safe_eval(self.positions[i]) for i in range(self.num_particles)], dtype=float)
        idx = int(np.argmin(self.pbest_val))
        self.gbest_pos = self.pbest_pos[idx].copy()
        self.gbest_val = float(self.pbest_val[idx])
    def _safe_eval(self, x):
        try:
            v = float(self.fitness_fn(x))
            if np.isnan(v) or np.isinf(v):
                return np.finfo(float).max
            return v
        except Exception:
            return np.finfo(float).max
    def _clip(self, x):
        return np.minimum(np.maximum(x, self.bounds[:,0]), self.bounds[:,1])
    def optimize(self):
        alpha = self.alpha_start
        for t in range(1, self.max_iter + 1):
            mbest = np.mean(self.pbest_pos, axis=0)
            frac = (t - 1) / max(1, (self.max_iter - 1))
            alpha = self.alpha_start + frac * (self.alpha_end - self.alpha_start)
            for i in range(self.num_particles):
                phi = self.rng.rand(self.D)
                p_i = phi * self.pbest_pos[i] + (1.0 - phi) * self.gbest_pos
                u = self.rng.rand(self.D)
                u = np.clip(u, 1e-12, 1.0-1e-12)
                sign = self.rng.choice([1.0, -1.0], size=self.D)
                distance = np.abs(mbest - self.positions[i])
                ln_term = np.log(1.0 / u)
                x_new = p_i + sign * (alpha * distance * ln_term)
                x_new = self._clip(x_new)
                fx_new = self._safe_eval(x_new)
                if fx_new < self.pbest_val[i]:
                    self.pbest_val[i] = fx_new
                    self.pbest_pos[i] = x_new.copy()
                if fx_new < self.gbest_val:
                    self.gbest_val = float(fx_new)
                    self.gbest_pos = x_new.copy()
                self.positions[i] = x_new
            if self.verbose and (t % max(1, self.max_iter // 10) == 0):
                print(f"[QPSO] iter {t}/{self.max_iter} gbest={self.gbest_val:.6e}")
        return self.gbest_pos.copy(), float(self.gbest_val)

# ------------------------------
# ALS adversarial game wrapper for one-vs-rest (source_label)
# ------------------------------
def alternating_least_squares_game_one_vs_rest(source_label, qc_model, qvae_obj, train_images, train_labels,
                                               max_game_iter=3, qpso_iter=200, n_pos_samples=80,
                                               qpso_particles=30, qpso_alpha_start=1.0, qpso_alpha_end=0.2,
                                               qpso_seed=1234, retrain_epochs=3):
    # Build pos / neg masks for one-vs-rest
    imgs = train_images
    labs = train_labels
    pos_mask = (labs == source_label)
    neg_mask = (labs != source_label)
    if pos_mask.sum().item() == 0:
        raise RuntimeError(f"No examples for source class {source_label}")
    pos_imgs = imgs[pos_mask][:n_pos_samples].to(device)
    pos_labs = labs[pos_mask][:n_pos_samples].to(device)
    neg_imgs = imgs[neg_mask][:n_pos_samples].to(device)
    neg_labs = labs[neg_mask][:n_pos_samples].to(device)

    combined = torch.cat([pos_imgs, neg_imgs], dim=0)
    combined_targets = torch.cat([pos_labs, neg_labs], dim=0)  # true labels (0..9)
    # we will compute TPR for source_label later when evaluating payoff; define specialized fitness.

    A_mu = np.zeros(latent_qubits, dtype=float)
    A_std = np.zeros(latent_qubits, dtype=float)
    payoff_curr = -np.inf

    D = latent_qubits
    bounds = np.vstack([np.full(D, -0.5), np.full(D, 0.5)]).T

    # Helper to compute payoff with explicit source label TPR
    def payoff_wrapper(alpha_mean, alpha_std):
        # encode positives average
        with torch.no_grad():
            mu_pos_all, logvar_pos_all = qvae_obj.encode(pos_imgs)
        mu_pos_avg = mu_pos_all.mean(dim=0)
        std_pos_avg = logvar_pos_all.mean(dim=0)
        # generate adv latent for the batch (use mu_pos_avg as representative)
        z_adv, _, _ = quantum_adversarial_latent(mu_pos_avg.unsqueeze(0), std_pos_avg.unsqueeze(0), alpha_mean, alpha_std)
        # classify
        logits = qc_model.fc(z_adv.to(device))
        _, preds = torch.max(logits, 1)
        # compute whether predicted == source_label
        pred_eq_src = (preds.cpu().numpy() == source_label).astype(int)
        # Because we used averaged mu, interpret tpr based on repeated pos samples: approximate as mean(pred_eq_src)
        tpr1 = pred_eq_src.mean() if len(pred_eq_src) > 0 else 0.0
        error = 1.0 - tpr1
        cost = calc_cost(alpha_mean, alpha_std)
        payoff = 1.0 + error - cost
        return payoff

    for game_iter in range(max_game_iter):
        with torch.no_grad():
            mu_pos_all, logvar_pos_all = qvae_obj.encode(pos_imgs)
        mu_pos_avg = mu_pos_all.mean(dim=0)
        std_pos_avg = logvar_pos_all.mean(dim=0)

        # Fit mean
        def fit_mean(alpha_vec):
            return -payoff_wrapper(alpha_vec, A_std)

        qpso_mean = QPSO(fit_mean, bounds, num_particles=qpso_particles, max_iter=qpso_iter,
                         alpha_start=qpso_alpha_start, alpha_end=qpso_alpha_end, seed=qpso_seed + source_label + game_iter, verbose=False)
        alpha_star_mean, _ = qpso_mean.optimize()

        # Fit std
        def fit_std(alpha_vec):
            return -payoff_wrapper(alpha_star_mean, alpha_vec)

        qpso_std = QPSO(fit_std, bounds, num_particles=qpso_particles, max_iter=qpso_iter,
                        alpha_start=qpso_alpha_start, alpha_end=qpso_alpha_end, seed=qpso_seed + source_label + game_iter + 100, verbose=False)
        alpha_star_std, _ = qpso_std.optimize()

        payoff_new = payoff_wrapper(alpha_star_mean, alpha_star_std)
        print(f"[ALS QPSO] source={source_label} Game {game_iter}: payoff_curr={payoff_curr:.6f} payoff_new={payoff_new:.6f}")

        if payoff_new > payoff_curr:
            payoff_curr = payoff_new
            A_mu = alpha_star_mean.copy()
            A_std = alpha_star_std.copy()

            # Build adversarial images by applying A to full pos set (mu_pos_all)
            z_adv_full, _, _ = quantum_adversarial_latent(mu_pos_all, logvar_pos_all, A_mu, A_std)
            adv_images = classical_decoder(z_adv_full.to(device))
            adv_targets = torch.full((adv_images.size(0),), source_label, dtype=torch.long, device=device)
            # fine-tune classifier head on a small combined set (one-vs-rest fine-tuning)
            subset_size = min(500, len(train_images))
            train_data_subset = train_images[:subset_size].to(device)
            train_targets_subset = train_labels[:subset_size].to(device)
            combined_images = torch.cat([train_data_subset, adv_images], dim=0)
            combined_targets2 = torch.cat([train_targets_subset, adv_targets], dim=0)
            combined_ds = TensorDataset(combined_images, combined_targets2)
            combined_loader = DataLoader(combined_ds, batch_size=32, shuffle=True)
            qc_model.train()
            optimizer = torch.optim.Adam(qc_model.fc.parameters(), lr=1e-3)
            loss_fn = nn.CrossEntropyLoss()
            for epoch in range(retrain_epochs):
                run_loss = 0.0
                for bimgs, btargets in combined_loader:
                    bimgs = bimgs.to(device); btargets = btargets.to(device)
                    with torch.no_grad():
                        mu_b, logvar_b = qvae_obj.encode(bimgs)
                        z_b = qvae_obj.reparameterize(mu_b, logvar_b)
                    logits = qc_model.fc(z_b.to(device))
                    loss = loss_fn(logits, btargets)
                    optimizer.zero_grad(); loss.backward(); optimizer.step()
                    run_loss += loss.item()
                print(f" Retrain epoch {epoch+1}, loss {run_loss/len(combined_loader):.4f}")
        else:
            print("No improvement — stopping ALS for this source.")
            break

    return A_mu, A_std, payoff_curr

# ------------------------------
# Adversarial example generation & saving
# ------------------------------
def save_image_tensor(img_tensor, path):
    # img_tensor: single example (1,28,28) tensor in [0,1]
    arr = (img_tensor.squeeze().cpu().numpy() * 255.0).astype(np.uint8)
    im = Image.fromarray(arr, mode='L')
    im.save(path)

def generate_and_save_adv_examples_for_source(source_label, A_mu, A_std, n_examples=20, outdir="adv_examples", seed=0):
    """
    For a given source digit label, pick n_examples real images from test set (source class),
    build adversarial latent z_star = mu + A_mu + eps*exp(A_std), reconstruct to pixel space and save.
    Also collect predicted target labels for each adv sample.
    """
    np.random.seed(seed)
    Path(outdir).mkdir(parents=True, exist_ok=True)
    src_dir = Path(outdir) / f"source_{source_label}"
    src_dir.mkdir(parents=True, exist_ok=True)
    # pick samples from test set belonging to source_label
    mask = (test_labels_all == source_label)
    idxs = np.where(mask.cpu().numpy())[0]
    if len(idxs) == 0:
        print(f"No test examples for class {source_label}. Skipping save.")
        return []
    chosen = np.random.choice(idxs, size=min(n_examples, len(idxs)), replace=False)
    preds = []
    for i, idx in enumerate(chosen):
        x = test_images_all[idx].unsqueeze(0).to(device)  # shape (1,1,28,28)
        with torch.no_grad():
            mu, logvar = qvae.encode(x)
            # apply alpha_star to mu/logvar
            mu_a = mu + torch.tensor(A_mu, dtype=torch.float32).to(device)
            logvar_a = logvar + torch.tensor(A_std, dtype=torch.float32).to(device)
            eps = torch.randn_like(mu_a)
            z_star = mu_a + eps * torch.exp(0.5 * logvar_a)
            x_adv = classical_decoder(z_star.to(device))
            # clamp to [0,1]
            x_adv = torch.clamp(x_adv, 0.0, 1.0)
            # predict using classifier head (we use fc on z_star)
            logits = qc_model.fc(z_star.to(device))
            pred = int(torch.argmax(logits, dim=1).cpu().item())
            preds.append(pred)
        # save image
        fname = src_dir / f"src{source_label}_idx{int(idx)}_adv{str(i)}_pred{pred}.png"
        save_image_tensor(x_adv[0], fname)
    # return list of predicted labels
    return preds

def plot_adversarial_grid(outdir="adv_examples", grid_out="adv_grid_all.png", per_source_cols=10):
    """
    Create a combined grid image per source in outdir and an overall combined image.
    Each source folder should contain saved PNGs; we will load up to per_source_cols images each.
    """
    src_folders = sorted([p for p in Path(outdir).iterdir() if p.is_dir()], key=lambda x: x.name)
    per_source_imgs = []
    for sf in src_folders:
        imgs = sorted(sf.glob("*.png"))
        row_imgs = []
        for i in range(per_source_cols):
            if i < len(imgs):
                im = Image.open(imgs[i]).convert("L").resize((28,28))
            else:
                im = Image.new("L", (28,28), color=255)
            row_imgs.append(im)
        # concatenate horizontally for this source
        w = 28 * per_source_cols
        h = 28
        row = Image.new("L", (w,h))
        for i,im in enumerate(row_imgs):
            row.paste(im, (i*28,0))
        per_source_imgs.append(row)
    # concatenate all rows vertically
    if not per_source_imgs:
        print("No adversarial images found to plot.")
        return
    W = per_source_imgs[0].width
    H = per_source_imgs[0].height * len(per_source_imgs)
    grid = Image.new("L", (W, H), color=255)
    for r, row in enumerate(per_source_imgs):
        grid.paste(row, (0, r*row.height))
    grid.save(Path(outdir) / grid_out)
    print(f"Saved adversarial grid: {Path(outdir) / grid_out}")

# ------------------------------
# Runtime checks & main orchestration
# ------------------------------
def check_environment():
    print("\n=== Runtime Environment Check ===")
    print("Torch:", torch.__version__)
    try:
        import qiskit
        print("Qiskit:", qiskit.__version__)
    except:
        print("Qiskit version UNKNOWN")
    try:
        import qiskit_aer
        print("Qiskit Aer:", qiskit_aer.__version__)
    except:
        print("Qiskit Aer NOT FOUND (using estimator fallback).")
    if torch.cuda.is_available():
        print("CUDA available ✔")
        try:
            print("GPU:", torch.cuda.get_device_name(0))
        except:
            pass
    else:
        print("CUDA not available ✘ — running on CPU")
    print("Device selected:", device)
    print("=================================")

def check_qiskit_primitives():
    print("\n=== Qiskit Primitive Self-Test ===")
    if EstimatorClass is None:
        print("No EstimatorClass available; skipping primitive self-test.")
        return
    try:
        est = EstimatorClass()
        qc = QuantumCircuit(1)
        qc.h(0)
        obs = SparsePauliOp("Z")
        job = est.run([qc], [obs])
        res = job.result()
        print("Estimator primitive OK. Test value length:", len(res.values))
    except Exception as e:
        print("ERROR: Estimator primitive failed:", e)
        raise RuntimeError("Estimator primitive is not functional.")
    print("===================================")

def check_dataset_shapes(train_images_all, train_labels_all):
    print("\n=== Dataset Sanity Check ===")
    print("Train images:", train_images_all.shape)
    print("Train labels:", train_labels_all.shape)
    try:
        n_counts = {int(c): int((train_labels_all==c).sum().item()) for c in range(10)}
        print("Class counts (train):", n_counts)
    except Exception as e:
        print("Warning checking class counts:", e)
    print("===================================")

# ------------------------------
# MAIN
# ------------------------------
if __name__ == "__main__":
    print("\n==== ALS + QPSO (one-vs-rest) adversarial example generation for ALL MNIST classes ====")
    check_environment()
    try:
        check_qiskit_primitives()
    except Exception as e:
        print("Estimator self-test failed; continuing but estimator calls may error:", e)
    check_dataset_shapes(train_images_all, train_labels_all)

    # (Optional) Pretrain / warm-up classical extractor + classifier head on full MNIST
    # For speed we do a few epochs of simple training of the fc head & extractor to have a baseline classifier.
    print("\nPretraining feature_extractor + classifier head on full MNIST (few epochs)...")
    optimizer = torch.optim.Adam(list(feature_extractor.parameters()) + list(qc_model.fc.parameters()), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    feature_extractor.train(); qc_model.train()
    pretrain_epochs = 3
    for e in range(pretrain_epochs):
        total_loss = 0.0; n = 0
        for xb, yb in train_loader:
            xb = xb.to(device); yb = yb.to(device)
            feats = feature_extractor(xb)
            logits = qc_model.fc(feats)
            loss = loss_fn(logits, yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item(); n += 1
        print(f" Pretrain epoch {e+1}/{pretrain_epochs}: avg loss {total_loss/max(1,n):.4f}")
    print("Pretraining done. Baseline evaluation on test set:")
    test_model(test_loader, qc_model)

    # create output dir
    outdir = "adv_examples"
    Path(outdir).mkdir(exist_ok=True)

    # Run ALS+QPSO for each source digit 0..9, generate adv examples and save
    summary = {}
    for src in range(10):
        print(f"\n--- Running ALS+QPSO for source digit {src} ---")
        try:
            A_mu_src, A_std_src, payoff_src = alternating_least_squares_game_one_vs_rest(
                source_label=src,
                qc_model=qc_model,
                qvae_obj=qvae,
                train_images=train_images_all,
                train_labels=train_labels_all,
                max_game_iter=2,
                qpso_iter=120,           # smaller for speed; increase if you want better search
                n_pos_samples=80,
                qpso_particles=24,
                qpso_alpha_start=1.0,
                qpso_alpha_end=0.2,
                qpso_seed=seed + src,
                retrain_epochs=2
            )
            print(f" Source {src}: payoff={payoff_src:.6f}")
            # generate and save adversarial examples for this source
            preds = generate_and_save_adv_examples_for_source(src, A_mu_src, A_std_src, n_examples=20, outdir=outdir, seed=seed+src)
            summary[src] = {"A_mu": A_mu_src.tolist(), "A_std": A_std_src.tolist(), "payoff": float(payoff_src), "preds_sample": preds}
        except Exception as e:
            print(f"Error for source {src}: {e}")
            summary[src] = {"error": str(e)}
    # Save summary
    with open(Path(outdir) / "adv_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    print("Saved adv_summary.json")

    # Create grid PNG
    print("Creating adversarial grid overview...")
    plot_adversarial_grid(outdir=outdir, grid_out="adv_grid_all_sources.png", per_source_cols=10)

    print("\nDone. Check the folder ./adv_examples/ for images and adv_grid_all_sources.png")


Using qiskit_aer Estimator (fast).
Device: cpu
Loaded MNIST via torchvision.
Datasets prepared: Train 54000 Val 6000 Test 10000


C:\Users\Aaditya Rajput\AppData\Local\Temp\ipykernel_8840\374987672.py:203: DeprecationWarning: Estimator has been deprecated as of Aer 0.15, please use EstimatorV2 instead.
  estimator = create_estimator()
C:\Users\Aaditya Rajput\AppData\Local\Temp\ipykernel_8840\374987672.py:203: DeprecationWarning: Option approximation=False is deprecated as of qiskit-aer 0.13. It will be removed no earlier than 3 months after the release date. Instead, use BackendEstimator from qiskit.primitives.
  estimator = create_estimator()
C:\Users\Aaditya Rajput\AppData\Local\Temp\ipykernel_8840\374987672.py:696: DeprecationWarning: Estimator has been deprecated as of Aer 0.15, please use EstimatorV2 instead.
  check_qiskit_primitives()
C:\Users\Aaditya Rajput\AppData\Local\Temp\ipykernel_8840\374987672.py:696: DeprecationWarning: Option approximation=False is deprecated as of qiskit-aer 0.13. It will be removed no earlier than 3 months after the release date. Instead, use BackendEstimator from qiskit.primit


==== ALS + QPSO (one-vs-rest) adversarial example generation for ALL MNIST classes ====

=== Runtime Environment Check ===
Torch: 2.9.1+cpu
Qiskit: 2.2.3
Qiskit Aer: 0.17.2
CUDA not available ✘ — running on CPU
Device selected: cpu

=== Qiskit Primitive Self-Test ===
Estimator primitive OK. Test value length: 1

=== Dataset Sanity Check ===
Train images: torch.Size([60000, 1, 28, 28])
Train labels: torch.Size([60000])
Class counts (train): {0: 5923, 1: 6742, 2: 5958, 3: 6131, 4: 5842, 5: 5421, 6: 5918, 7: 6265, 8: 5851, 9: 5949}

Pretraining feature_extractor + classifier head on full MNIST (few epochs)...
 Pretrain epoch 1/3: avg loss 0.8729
 Pretrain epoch 2/3: avg loss 0.4241
 Pretrain epoch 3/3: avg loss 0.3494
Pretraining done. Baseline evaluation on test set:
Acc: 0.0790, per-class TPR: [0.0143, 0.0044, 0.4409, 0.0535, 0.0081, 0.0056, 0.0021, 0.0029, 0.0811, 0.1635]

--- Running ALS+QPSO for source digit 0 ---


KeyboardInterrupt: 

In [ ]:
# quantum_adversarial_game_QPSO.py
# Full merged script: Q-VAE + Q-CNN + ALS adversarial game using QPSO + hard defense + utilities

import os, time, math, hashlib, random, json
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split, ConcatDataset
from sklearn.metrics import confusion_matrix

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Prefer Aer Estimator (fast) if available; fallback to StatevectorEstimator
try:
    from qiskit_aer.primitives import Estimator as AerEstimator
    EstimatorClass = AerEstimator
    print("Using qiskit_aer Estimator (fast).")
except Exception:
    try:
        from qiskit.primitives import StatevectorEstimator as EstimatorClass
        print("Using qiskit.primitives.StatevectorEstimator (fallback).")
    except Exception:
        EstimatorClass = None
        print("Warning: No suitable Qiskit estimator found. EstimatorClass=None")

# ------------------------------
# Settings
# ------------------------------
latent_qubits = 8
batch_size = 64
seed = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

# ------------------------------
# MNIST Loader (6 vs 8)
# ------------------------------
def load_mnist_filtered(label1=6, label2=8, batch_size=64):
    try:
        from torchvision import datasets, transforms
        transform = transforms.Compose([transforms.ToTensor()])
        train_full = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
        test_full  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

        train_images = train_full.data.unsqueeze(1).float() / 255.0
        train_labels = train_full.targets.long()
        test_images = test_full.data.unsqueeze(1).float() / 255.0
        test_labels = test_full.targets.long()

        print("Loaded MNIST via torchvision.")
    except Exception as e:
        print("torchvision failed, falling back to sklearn:", e)
        from sklearn.datasets import fetch_openml
        mn = fetch_openml('mnist_784', version=1, as_frame=False)
        X = mn['data'].astype('float32') / 255.0
        y = mn['target'].astype(int)
        X = X.reshape(-1, 1, 28, 28)

        train_images, test_images = X[:60000], X[60000:]
        train_labels, test_labels = y[:60000], y[60000:]

        train_images = torch.from_numpy(train_images)
        train_labels = torch.from_numpy(train_labels).long()
        test_images = torch.from_numpy(test_images)
        test_labels = torch.from_numpy(test_labels).long()

        print("Loaded MNIST via sklearn.")

    mask_train = (train_labels == label1) | (train_labels == label2)
    mask_test  = (test_labels == label1) | (test_labels == label2)

    train_images = train_images[mask_train]
    train_labels = train_labels[mask_train]
    test_images  = test_images[mask_test]
    test_labels  = test_labels[mask_test]

    train_labels = torch.where(train_labels == label1, torch.tensor(0), torch.tensor(1))
    test_labels  = torch.where(test_labels  == label1, torch.tensor(0), torch.tensor(1))

    total_train = TensorDataset(train_images, train_labels)
    train_size = int(0.8 * len(total_train))
    val_size = len(total_train) - train_size

    train_dataset, val_dataset = random_split(total_train, [train_size, val_size])
    test_dataset = TensorDataset(test_images, test_labels)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print("Datasets prepared: Train", len(train_dataset), "Val", len(val_dataset), "Test", len(test_dataset))
    return train_loader, val_loader, test_loader, train_images, train_labels, test_images, test_labels

train_loader, val_loader, test_loader, train_images_all, train_labels_all, test_images_all, test_labels_all = \
    load_mnist_filtered(batch_size=batch_size)

# ------------------------------
# Classical feature extractor + decoder
# ------------------------------
input_dim = 28 * 28

class ClassicalFeatureExtractor(nn.Module):
    def __init__(self, latent_qubits):
        super().__init__()
        self.fc = nn.Linear(input_dim, latent_qubits)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return torch.tanh(self.fc(x))

class ClassicalDecoder(nn.Module):
    def __init__(self, latent_qubits):
        super().__init__()
        self.fc1 = nn.Linear(latent_qubits, 128)
        self.fc2 = nn.Linear(128, input_dim)
    def forward(self, z):
        z = F.relu(self.fc1(z))
        out = torch.sigmoid(self.fc2(z))
        return out.view(-1,1,28,28)

feature_extractor = ClassicalFeatureExtractor(latent_qubits).to(device)
classical_decoder = ClassicalDecoder(latent_qubits).to(device)

# ------------------------------
# Circuit caching utilities
# ------------------------------
CIRCUIT_CACHE = {}
def _params_to_list(params):
    if isinstance(params, np.ndarray):
        return params.flatten().tolist()
    elif isinstance(params, (list, tuple)):
        return np.array(params).flatten().tolist()
    else:
        return np.array(params).flatten().tolist()

def cache_key(type_name, params, angles):
    p_list = _params_to_list(params)
    a_list = np.round(np.array(angles).flatten(), 6).tolist()
    raw = f"{type_name}|{p_list}|{a_list}"
    return hashlib.sha256(raw.encode()).hexdigest()

def get_encoder_circuit_cached(n_qubits, encoder_params, x_angles):
    key = cache_key("enc", encoder_params, x_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(x_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(encoder_params[i,0]), i)
        qc.rz(float(encoder_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

def get_decoder_circuit_cached(n_qubits, decoder_params, z_angles):
    key = cache_key("dec", decoder_params, z_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(z_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(decoder_params[i,0]), i)
        qc.rz(float(decoder_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

def get_classifier_circuit_cached(n_qubits, clf_params, x_angles):
    key = cache_key("clf", clf_params, x_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(x_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(clf_params[i,0]), i)
        qc.rz(float(clf_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

# ------------------------------
# Observables: Z individual + grouped
# ------------------------------
z_ops = []
for i in range(latent_qubits):
    s = ['I'] * latent_qubits
    s[latent_qubits - 1 - i] = 'Z'
    z_ops.append(SparsePauliOp(''.join(s)))
Z_GROUPED = z_ops[0]
for i in range(1, latent_qubits):
    Z_GROUPED = Z_GROUPED + z_ops[i]

# ------------------------------
# Estimator instance (wrap factory)
# ------------------------------
def create_estimator():
    if EstimatorClass is None:
        raise RuntimeError("EstimatorClass is not available in this environment.")
    try:
        return EstimatorClass()
    except Exception as e:
        try:
            return EstimatorClass()
        except Exception as e2:
            raise RuntimeError("Could not create estimator: " + str(e2))

estimator = None
try:
    estimator = create_estimator()
except Exception as e:
    print("Estimator creation failed:", e)

# ------------------------------
# measure_z_expectations_batch
# ------------------------------
def measure_z_expectations_batch(qc_list):
    if len(qc_list) == 0:
        return np.zeros((0, latent_qubits), dtype=float)
    n_circuits = len(qc_list)
    grouped_obs_list = [Z_GROUPED] * n_circuits
    job = estimator.run(circuits=qc_list, observables=grouped_obs_list)
    _ = job.result()
    values = np.zeros((n_circuits, latent_qubits), dtype=float)
    for q in range(latent_qubits):
        obs = z_ops[q]
        obs_list = [obs] * n_circuits
        job_q = estimator.run(circuits=qc_list, observables=obs_list)
        res_q = job_q.result()
        vals_q = np.array(res_q.values).reshape(-1)
        values[:, q] = vals_q
    return values

# ------------------------------
# QuantumVAE
# ------------------------------
class QuantumVAE:
    def __init__(self, n_qubits):
        self.n_qubits = n_qubits
        self.encoder_params = np.random.randn(n_qubits, 2) * 0.05
        self.decoder_params = np.random.randn(n_qubits, 2) * 0.05

    def encode(self, x_batch, angles_cache=None):
        x_batch = x_batch.detach().cpu()
        b = x_batch.size(0)
        if angles_cache is None:
            feats = feature_extractor(x_batch.to(device)).detach().cpu().numpy()
            angles = (feats + 1.0) * (np.pi / 2.0)
        else:
            angles = angles_cache
        qc_list = [get_encoder_circuit_cached(self.n_qubits, self.encoder_params, angles[i]) for i in range(b)]
        vals = measure_z_expectations_batch(qc_list)
        mu_angles = np.arcsin(np.clip(vals, -1.0, 1.0))
        mu = torch.tensor(mu_angles, dtype=torch.float32).to(device)
        logvar = torch.zeros_like(mu).to(device)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        eps = torch.randn_like(mu)
        return mu + eps * torch.exp(0.5 * logvar)

    def decode(self, z_batch):
        z_np = z_batch.detach().cpu().numpy()
        qc_list = [get_decoder_circuit_cached(self.n_qubits, self.decoder_params, z_np[i]) for i in range(z_np.shape[0])]
        vals = measure_z_expectations_batch(qc_list)
        return torch.tensor(vals, dtype=torch.float32).to(device)

    def forward(self, x_batch, angles_cache=None):
        mu, logvar = self.encode(x_batch, angles_cache)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

qvae = QuantumVAE(latent_qubits)

# ------------------------------
# QuantumCNN wrapper
# ------------------------------
class QuantumCNN(nn.Module):
    def __init__(self, n_qubits, n_classes=2):
        super().__init__()
        self.n_qubits = n_qubits
        self.clf_params = np.random.randn(n_qubits, 2) * 0.05
        self.fc = nn.Linear(n_qubits, n_classes).to(device)

    def quantum_features_batch(self, x_batch, angles_cache=None):
        x_batch = x_batch.detach().cpu()
        b = x_batch.size(0)
        if angles_cache is None:
            feats = feature_extractor(x_batch.to(device)).detach().cpu().numpy()
            angles = (feats + 1.0) * (np.pi / 2.0)
        else:
            angles = angles_cache
        qc_list = [get_classifier_circuit_cached(self.n_qubits, self.clf_params, angles[i]) for i in range(b)]
        vals = measure_z_expectations_batch(qc_list)
        return torch.tensor(vals, dtype=torch.float32).to(device)

    def forward(self, x_batch, angles_cache=None):
        feats_q = self.quantum_features_batch(x_batch, angles_cache)
        logits = self.fc(feats_q)
        return logits

qc_model = QuantumCNN(latent_qubits)

# ------------------------------
# Metrics & test
# ------------------------------
def test_model(loader, model):
    model.eval()
    all_targets = []
    all_preds = []
    with torch.no_grad():
        for data, targets in loader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs, 1)
            all_targets.extend(targets.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
    cm = confusion_matrix(all_targets, all_preds, labels=[0,1])
    if cm.size == 4:
        tp0, fn0 = cm[0,0], cm[0,1]
        tp1, fn1 = cm[1,1], cm[1,0]
    else:
        tp0 = fn0 = tp1 = fn1 = 0
    tpr0 = tp0 / (tp0 + fn0) if (tp0 + fn0) > 0 else 0.0
    tpr1 = tp1 / (tp1 + fn1) if (tp1 + fn1) > 0 else 0.0
    acc = (np.array(all_preds) == np.array(all_targets)).mean() if len(all_targets) else 0.0
    print(f"Acc: {acc:.4f}, TPR0: {tpr0:.4f}, TPR1: {tpr1:.4f}")
    return tpr0, tpr1

# ------------------------------
# Adversarial utilities (latent)
# ------------------------------
def quantum_adversarial_latent(mu_pos, logvar_pos, A_mu, A_std):
    if isinstance(A_mu, np.ndarray):
        A_mu = torch.tensor(A_mu, dtype=torch.float32).to(device)
    if isinstance(A_std, np.ndarray):
        A_std = torch.tensor(A_std, dtype=torch.float32).to(device)
    mu_adv = mu_pos + A_mu
    logvar_adv = logvar_pos + A_std
    eps = torch.randn_like(mu_adv)
    z_adv = mu_adv + eps * torch.exp(0.5 * logvar_adv)
    return z_adv, mu_adv, logvar_adv

def calc_cost(a_mean, a_std):
    a_mean_t = torch.tensor(a_mean, dtype=torch.float32) if not isinstance(a_mean, torch.Tensor) else a_mean
    a_std_t = torch.tensor(a_std, dtype=torch.float32) if not isinstance(a_std, torch.Tensor) else a_std
    norm_mean = torch.norm(a_mean_t)
    norm_std = torch.norm(a_std_t)
    total_cost = (norm_mean / latent_qubits) + (norm_std / latent_qubits)
    return total_cost.item()

def adversary_payoff(posnegdata, posindices, negindices, mu_pos, std_pos, qvae_obj, a_mean, a_std, posnegtargets, model):
    if mu_pos.dim() == 1:
        mu_pos = mu_pos.unsqueeze(0)
    if std_pos.dim() == 1:
        std_pos = std_pos.unsqueeze(0)
    z_adv, _, _ = quantum_adversarial_latent(mu_pos, std_pos, a_mean, a_std)
    logits = model.fc(z_adv.to(device))
    _, preds = torch.max(logits, 1)
    targets = torch.tensor(posnegtargets, dtype=torch.long).to(device)
    true_positives = ((preds == 1) & (targets == 1)).sum().item()
    false_negatives = ((preds == 0) & (targets == 1)).sum().item()
    tpr1 = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0.0
    error = 1.0 - tpr1
    cost = calc_cost(a_mean, a_std)
    payoff = 1.0 + error - cost
    return payoff

# ------------------------------
# QPSO optimizer (replaces PSO / SA)
# ------------------------------
class QPSO:
    """
    Quantum-behaved Particle Swarm Optimizer (QPSO)
    Minimizes fitness_fn(x) where x is a 1D numpy vector.
    """
    def __init__(self, fitness_fn, bounds, num_particles=30, max_iter=300,
                 alpha_start=1.0, alpha_end=0.1, seed=None, verbose=False):
        self.fitness_fn = fitness_fn
        self.bounds = np.array(bounds, dtype=float)
        assert self.bounds.ndim == 2 and self.bounds.shape[1] == 2
        self.D = self.bounds.shape[0]

        self.num_particles = int(num_particles)
        self.max_iter = int(max_iter)
        self.alpha_start = float(alpha_start)
        self.alpha_end = float(alpha_end)
        self.verbose = bool(verbose)

        self.rng = np.random.RandomState(seed)

        # initialize swarm positions uniformly within bounds
        self.positions = self.rng.uniform(self.bounds[:,0], self.bounds[:,1], size=(self.num_particles, self.D))
        self.pbest_pos = self.positions.copy()
        self.pbest_val = np.array([self._safe_eval(self.positions[i]) for i in range(self.num_particles)], dtype=float)

        idx = int(np.argmin(self.pbest_val))
        self.gbest_pos = self.pbest_pos[idx].copy()
        self.gbest_val = float(self.pbest_val[idx])

    def _safe_eval(self, x):
        try:
            v = float(self.fitness_fn(x))
            if np.isnan(v) or np.isinf(v):
                return np.finfo(float).max
            return v
        except Exception:
            return np.finfo(float).max

    def _clip(self, x):
        return np.minimum(np.maximum(x, self.bounds[:,0]), self.bounds[:,1])

    def optimize(self):
        alpha = self.alpha_start
        for t in range(1, self.max_iter + 1):
            mbest = np.mean(self.pbest_pos, axis=0)
            frac = (t - 1) / max(1, (self.max_iter - 1))
            alpha = self.alpha_start + frac * (self.alpha_end - self.alpha_start)

            for i in range(self.num_particles):
                phi = self.rng.rand(self.D)
                p_i = phi * self.pbest_pos[i] + (1.0 - phi) * self.gbest_pos

                u = self.rng.rand(self.D)
                u = np.clip(u, 1e-12, 1.0-1e-12)
                sign = self.rng.choice([1.0, -1.0], size=self.D)

                distance = np.abs(mbest - self.positions[i])
                ln_term = np.log(1.0 / u)

                x_new = p_i + sign * (alpha * distance * ln_term)
                x_new = self._clip(x_new)

                fx_new = self._safe_eval(x_new)

                if fx_new < self.pbest_val[i]:
                    self.pbest_val[i] = fx_new
                    self.pbest_pos[i] = x_new.copy()

                if fx_new < self.gbest_val:
                    self.gbest_val = float(fx_new)
                    self.gbest_pos = x_new.copy()

                self.positions[i] = x_new

            if self.verbose and (t % max(1, self.max_iter // 10) == 0):
                print(f"[QPSO] iter {t}/{self.max_iter} gbest={self.gbest_val:.6e}")

        return self.gbest_pos.copy(), float(self.gbest_val)

# ------------------------------
# Alternating least squares adversarial game (QPSO inner optimizer)
# ------------------------------
def alternating_least_squares_game(qc_model, qvae_obj, train_images_all, train_labels_all,
                                   max_game_iter=3, qpso_iter=200, n_pos_samples=80,
                                   qpso_particles=30, qpso_alpha_start=1.0, qpso_alpha_end=0.2,
                                   qpso_seed=1234, retrain_epochs=3):
    imgs = train_images_all
    labs = train_labels_all
    pos_mask = (labs == 1)
    neg_mask = (labs == 0)
    pos_imgs = imgs[pos_mask][:n_pos_samples].to(device)
    pos_labs = labs[pos_mask][:n_pos_samples].to(device)
    neg_imgs = imgs[neg_mask][:n_pos_samples].to(device)
    neg_labs = labs[neg_mask][:n_pos_samples].to(device)

    combined = torch.cat([pos_imgs, neg_imgs], dim=0)
    combined_targets = torch.cat([pos_labs, neg_labs], dim=0)
    pos_indices = (combined_targets.cpu().numpy() == 1).tolist()
    neg_indices = (combined_targets.cpu().numpy() == 0).tolist()

    A_mu = np.zeros(latent_qubits, dtype=float)
    A_std = np.zeros(latent_qubits, dtype=float)
    payoff_curr = -np.inf

    D = latent_qubits
    bounds = np.vstack([np.full(D, -0.5), np.full(D, 0.5)]).T

    for game_iter in range(max_game_iter):
        with torch.no_grad():
            mu_pos_all, logvar_pos_all = qvae_obj.encode(pos_imgs)
        mu_pos_avg = mu_pos_all.mean(dim=0)
        std_pos_avg = logvar_pos_all.mean(dim=0)

        def fit_mean(alpha_vec):
            return -adversary_payoff(combined, pos_indices, neg_indices,
                                     mu_pos_avg, std_pos_avg, qvae_obj,
                                     alpha_vec, A_std, combined_targets.cpu().numpy(), qc_model)

        qpso_mean = QPSO(fit_mean, bounds,
                         num_particles=qpso_particles, max_iter=qpso_iter,
                         alpha_start=qpso_alpha_start, alpha_end=qpso_alpha_end,
                         seed=qpso_seed + game_iter + 1, verbose=False)
        alpha_star_mean, best_mean_val = qpso_mean.optimize()

        def fit_std(alpha_vec):
            return -adversary_payoff(combined, pos_indices, neg_indices,
                                     mu_pos_avg, std_pos_avg, qvae_obj,
                                     alpha_star_mean, alpha_vec, combined_targets.cpu().numpy(), qc_model)

        qpso_std = QPSO(fit_std, bounds,
                        num_particles=qpso_particles, max_iter=qpso_iter,
                        alpha_start=qpso_alpha_start, alpha_end=qpso_alpha_end,
                        seed=qpso_seed + game_iter + 10, verbose=False)
        alpha_star_std, best_std_val = qpso_std.optimize()

        payoff_new = adversary_payoff(combined, pos_indices, neg_indices,
                                      mu_pos_avg, std_pos_avg, qvae_obj,
                                      alpha_star_mean, alpha_star_std, combined_targets.cpu().numpy(), qc_model)
        print(f"[ALS QPSO] Game {game_iter}: payoff_curr={payoff_curr:.6f} payoff_new={payoff_new:.6f}")

        if payoff_new > payoff_curr:
            payoff_curr = payoff_new
            A_mu = alpha_star_mean.copy()
            A_std = alpha_star_std.copy()

            z_adv, _, _ = quantum_adversarial_latent(mu_pos_all, logvar_pos_all, A_mu, A_std)
            adv_images = classical_decoder(z_adv.to(device))
            adv_targets = torch.ones(adv_images.size(0), dtype=torch.long).to(device)

            subset_size = min(500, len(train_images_all))
            train_data_subset = train_images_all[:subset_size].to(device)
            train_targets_subset = train_labels_all[:subset_size].to(device)
            combined_images = torch.cat([train_data_subset, adv_images], dim=0)
            combined_targets2 = torch.cat([train_targets_subset, adv_targets], dim=0)
            combined_ds = TensorDataset(combined_images, combined_targets2)
            combined_loader = DataLoader(combined_ds, batch_size=32, shuffle=True)

            qc_model.train()
            optimizer = torch.optim.Adam(qc_model.fc.parameters(), lr=1e-3)
            loss_fn = nn.CrossEntropyLoss()
            for epoch in range(retrain_epochs):
                run_loss = 0.0
                for bimgs, btargets in combined_loader:
                    bimgs = bimgs.to(device); btargets = btargets.to(device)
                    with torch.no_grad():
                        mu_b, logvar_b = qvae_obj.encode(bimgs)
                        z_b = qvae_obj.reparameterize(mu_b, logvar_b)
                    logits = qc_model.fc(z_b.to(device))
                    loss = loss_fn(logits, btargets)
                    optimizer.zero_grad(); loss.backward(); optimizer.step()
                    run_loss += loss.item()
                print(f" Retrain epoch {epoch+1}, loss {run_loss/len(combined_loader):.4f}")
        else:
            print("No improvement — stopping ALS.")
            break

    return A_mu, A_std, payoff_curr

# ------------------------------
# Parameter-shift helpers (quantum finetuning)
# ------------------------------
def compute_features_and_grads_batch_for_clf_params(x_batch, clf_params_np, n_qubits):
    with torch.no_grad():
        feats_classical = feature_extractor(x_batch.to(device)).detach().cpu().numpy()
    angles = (feats_classical + 1.0) * (np.pi / 2.0)
    qc_list = [get_classifier_circuit_cached(n_qubits, clf_params_np, angles[i]) for i in range(angles.shape[0])]
    vals = measure_z_expectations_batch(qc_list)
    feats_tensor = torch.tensor(vals, dtype=torch.float32).to(device)
    return feats_tensor, angles

def parameter_shift_update(clf_params_np, x_batch, batch_labels, n_qubits,
                           qc_model, lr_q=1e-2, shift=np.pi/2):
    feats_base, angles = compute_features_and_grads_batch_for_clf_params(x_batch, clf_params_np, n_qubits)
    feats_base = feats_base.clone().detach().requires_grad_(True)
    logits = qc_model.fc(feats_base)
    loss_fn = nn.CrossEntropyLoss()
    loss = loss_fn(logits, batch_labels.to(device))
    qc_model.fc.zero_grad()
    if feats_base.grad is not None:
        feats_base.grad.zero_()
    loss.backward(retain_graph=True)
    dL_dfeat = feats_base.grad.detach().cpu().numpy()  # (B, D)
    B = dL_dfeat.shape[0]
    grads_theta = np.zeros_like(clf_params_np, dtype=float)
    for q in range(n_qubits):
        for p_idx in range(2):
            clf_plus = clf_params_np.copy()
            clf_minus = clf_params_np.copy()
            clf_plus[q, p_idx] += shift
            clf_minus[q, p_idx] -= shift
            f_plus_tensor, _ = compute_features_and_grads_batch_for_clf_params(x_batch, clf_plus, n_qubits)
            f_minus_tensor, _ = compute_features_and_grads_batch_for_clf_params(x_batch, clf_minus, n_qubits)
            f_plus = f_plus_tensor.detach().cpu().numpy()
            f_minus = f_minus_tensor.detach().cpu().numpy()
            df_dtheta = 0.5 * (f_plus - f_minus)
            term = np.sum(dL_dfeat * df_dtheta)
            grads_theta[q, p_idx] = term / float(B)
    clf_params_np = clf_params_np - lr_q * grads_theta
    return clf_params_np, loss.item()

# ------------------------------
# Hard defense + quantum finetune
# ------------------------------
def hard_defense_with_quantum_finetune(qc_model, qvae, feature_extractor, classical_decoder,
                                       train_images_all, train_labels_all,
                                       val_loader=None,
                                       epochs=4,
                                       adv_samples_per_pos=8,
                                       adv_radius=0.5,
                                       batch_size_def=64,
                                       collapse_penalty_coef=12.0,
                                       lr=5e-4,
                                       subset_size=3000,
                                       fine_tune_quantum=True,
                                       lr_q=1e-2,
                                       quantum_batches_per_epoch=3,
                                       paramshift_shift=math.pi/2):
    feature_extractor.train(); qc_model.train()
    params = list(feature_extractor.parameters()) + list(qc_model.fc.parameters())
    optimizer = torch.optim.Adam(params, lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    device_local = device
    total_clean = min(subset_size, len(train_images_all))
    clean_images = train_images_all[:total_clean].to(device_local)
    clean_labels = train_labels_all[:total_clean].to(device_local)
    pos_mask = (train_labels_all == 1)
    pos_imgs_all = train_images_all[pos_mask]
    if len(pos_imgs_all) == 0:
        raise RuntimeError("No positive-class examples found.")
    pos_sample_count = min(200, len(pos_imgs_all))
    pos_imgs = pos_imgs_all[:pos_sample_count].to(device_local)
    clf_params_np = qc_model.clf_params.copy()
    for epoch in range(epochs):
        epoch_loss = 0.0; n_batches = 0; quantum_updates_done = 0
        with torch.no_grad():
            mu_pos_all, logvar_pos_all = qvae.encode(pos_imgs)
            mu_pos_rep = mu_pos_all.repeat_interleave(adv_samples_per_pos, dim=0)
            logvar_rep = logvar_pos_all.repeat_interleave(adv_samples_per_pos, dim=0)
            delta = torch.randn_like(mu_pos_rep, device=device_local) * adv_radius
            std_delta = torch.randn_like(logvar_rep, device=device_local) * (adv_radius * 0.3)
            mu_adv = mu_pos_rep + delta
            std_adv = torch.clamp(logvar_rep + std_delta, min=-3.0, max=3.0)
            eps = torch.randn_like(mu_adv)
            z_adv = mu_adv + eps * torch.exp(0.5 * std_adv)
            adv_images = classical_decoder(z_adv.to(device_local))
            adv_labels = torch.ones(adv_images.size(0), dtype=torch.long, device=device_local)
        idxs = torch.randperm(clean_images.size(0))
        clean_shuffled = clean_images[idxs]; clean_shuffled_labels = clean_labels[idxs]
        combined_images = torch.cat([clean_shuffled, adv_images], dim=0)
        combined_labels = torch.cat([clean_shuffled_labels, adv_labels], dim=0)
        combined_ds = TensorDataset(combined_images, combined_labels)
        combined_loader = DataLoader(combined_ds, batch_size=batch_size_def, shuffle=True)
        for batch_idx, (batch_imgs, batch_labels) in enumerate(combined_loader):
            batch_imgs = batch_imgs.to(device_local); batch_labels = batch_labels.to(device_local)
            feats = feature_extractor(batch_imgs)
            logits = qc_model.fc(feats)
            ce_loss = loss_fn(logits, batch_labels)
            probs1 = torch.softmax(logits, dim=1)[:, 1]
            mean_p1 = probs1.mean()
            collapse_penalty = collapse_penalty_coef * (mean_p1 - 0.5) ** 2
            l2_reg = 0.0
            for p in params:
                l2_reg += 1e-4 * torch.sum(p ** 2)
            loss = ce_loss + collapse_penalty + l2_reg
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            epoch_loss += loss.item(); n_batches += 1
            if fine_tune_quantum and (quantum_updates_done < quantum_batches_per_epoch):
                try:
                    clf_params_np, qloss = parameter_shift_update(clf_params_np, batch_imgs, batch_labels, latent_qubits, qc_model, lr_q=lr_q, shift=paramshift_shift)
                    qc_model.clf_params = clf_params_np.copy()
                    quantum_updates_done += 1
                except Exception as e:
                    print("Parameter-shift update error:", e)
        avg_loss = epoch_loss / max(1, n_batches)
        print(f"[HardDef+QEpoch {epoch+1}/{epochs}] Avg loss: {avg_loss:.4f}, quantum_updates: {quantum_updates_done}")
        if val_loader is not None:
            print("Validation eval:")
            test_model(val_loader, qc_model)
    qc_model.clf_params = clf_params_np.copy()
    print("Hard defense + quantum fine-tune complete.")

# ------------------------------
# Utilities and safety helpers
# ------------------------------
DEBUG = False
ENABLE_PROFILING = False

def debug(*args):
    if DEBUG:
        print("[DEBUG]", *args)

class Timer:
    def __init__(self, name="timer"):
        self.name = name
    def __enter__(self):
        if ENABLE_PROFILING:
            self.start = time.time()
        return self
    def __exit__(self, exc_type, exc, tb):
        if ENABLE_PROFILING:
            dt = time.time() - self.start
            print(f"[PROFILE] {self.name}: {dt:.4f}s")

def safe_reparameterize(mu, logvar):
    with torch.no_grad():
        eps = torch.randn_like(mu)
        logvar = torch.clamp(logvar, -10.0, 10.0)
        return mu + eps * torch.exp(0.5 * logvar)

QuantumVAE.reparameterize = lambda self, mu, logvar: safe_reparameterize(mu, logvar)

# ------------------------------
# Runtime checks
# ------------------------------
def check_environment():
    print("\n=== Runtime Environment Check ===")
    print("Torch:", torch.__version__)
    try:
        import qiskit
        print("Qiskit:", qiskit.__version__)
    except:
        print("Qiskit version UNKNOWN")
    try:
        import qiskit_aer
        print("Qiskit Aer:", qiskit_aer.__version__)
    except:
        print("Qiskit Aer NOT FOUND (using estimator fallback).")
    if torch.cuda.is_available():
        print("CUDA available ✔")
        try:
            print("GPU:", torch.cuda.get_device_name(0))
        except:
            pass
    else:
        print("CUDA not available ✘ — running on CPU")
    print("Device selected:", device)
    print("=================================")

def check_qiskit_primitives():
    print("\n=== Qiskit Primitive Self-Test ===")
    if EstimatorClass is None:
        print("No EstimatorClass available; skipping primitive self-test.")
        return
    try:
        est = EstimatorClass()
        qc = QuantumCircuit(1)
        qc.h(0)
        obs = SparsePauliOp("Z")
        job = est.run([qc], [obs])
        res = job.result()
        print("Estimator primitive OK. Test value length:", len(res.values))
    except Exception as e:
        print("ERROR: Estimator primitive failed:", e)
        raise RuntimeError("Estimator primitive is not functional.")
    print("===================================")

def check_dataset_shapes(train_images_all, train_labels_all):
    print("\n=== Dataset Sanity Check ===")
    print("Train images:", train_images_all.shape)
    print("Train labels:", train_labels_all.shape)
    try:
        n0 = (train_labels_all == 0).sum().item()
        n1 = (train_labels_all == 1).sum().item()
        print("Class-0 count:", n0)
        print("Class-1 count:", n1)
        if n0 == 0 or n1 == 0:
            raise RuntimeError("One class has zero samples — cannot train binary classifier.")
    except Exception as e:
        print("Warning checking class counts:", e)
    print("===================================")

# ------------------------------
# Main demo run: ALS with QPSO then hard defense
# ------------------------------
if __name__ == "__main__":
    print('\nQUANTUM ADVERSARIAL GAME (ALS with QPSO)')
    check_environment()
    try:
        check_qiskit_primitives()
    except Exception as e:
        print("Estimator self-test failed; continuing but execution may fail when calling estimator.")
    check_dataset_shapes(train_images_all, train_labels_all)

    print("\nRunning alternating least squares adversarial game with QPSO (short demo):")
    start = time.time()
    A_mu_final, A_std_final, final_payoff = alternating_least_squares_game(qc_model, qvae, train_images_all, train_labels_all,
                                                                           max_game_iter=2, qpso_iter=150, n_pos_samples=60,
                                                                           qpso_particles=30, qpso_alpha_start=1.0, qpso_alpha_end=0.2,
                                                                           qpso_seed=seed, retrain_epochs=3)
    print("Game finished in {:.1f}s".format(time.time() - start))
    print("A_mu:", A_mu_final)
    print("A_std:", A_std_final)
    print("Final payoff:", final_payoff)

    print("\nTest eval BEFORE defense (baseline):")
    baseline_tpr0, baseline_tpr1 = test_model(test_loader, qc_model)

    # ALPHA★ POST-GAME ATTACK EVALUATION
    print("\n=== ALPHA★ POST-GAME ATTACK EVALUATION ===")
    pos_mask_test = (test_labels_all == 1)
    pos_imgs_test = test_images_all[pos_mask_test]
    if pos_imgs_test.size(0) == 0:
        print("No positive test samples found; skipping alpha-star attack evaluation.")
    else:
        n_attack_samples = min(120, pos_imgs_test.size(0))
        pos_imgs_test = pos_imgs_test[:n_attack_samples].to(device)
        with torch.no_grad():
            mu_pos_test, logvar_pos_test = qvae.encode(pos_imgs_test)
        z_star, mu_star, logvar_star = quantum_adversarial_latent(mu_pos_test, logvar_pos_test, A_mu_final, A_std_final)
        qc_model.eval()
        with torch.no_grad():
            logits_star = qc_model.fc(z_star.to(device))
            _, preds_star = torch.max(logits_star, 1)
            true_labels_star = torch.ones_like(preds_star)
            tp1 = ((preds_star == 1) & (true_labels_star == 1)).sum().item()
            fn1 = ((preds_star == 0) & (true_labels_star == 1)).sum().item()
            tpr1_attack = tp1 / (tp1 + fn1 + 1e-12)
            asr = 1.0 - tpr1_attack
        print("\n--- Alpha★ Attack Results (latent eval) ---")
        print(f"Num attack samples: {n_attack_samples}")
        print(f"TPR1 on alpha★ manipulated positives (latent): {tpr1_attack:.4f}")
        print(f"Attack Success Rate (ASR): {asr:.4f}")
        print(f"Payoff at convergence: {final_payoff:.4f}")
        with torch.no_grad():
            recon_star = classical_decoder(z_star.to(device))
            mu_re, logvar_re = qvae.encode(recon_star)
            z_re = qvae.reparameterize(mu_re, logvar_re)
            logits_re = qc_model.fc(z_re.to(device))
            _, preds_re = torch.max(logits_re, 1)
            acc_recon = (preds_re == torch.ones_like(preds_re)).float().mean().item()
        print("\nClassifier performance on reconstructed alpha★ images (pixel-space):")
        print(f"Accuracy on reconstructions: {acc_recon:.4f}")

    # Save metrics
    metrics = {
        "baseline_tpr0": float(baseline_tpr0),
        "baseline_tpr1": float(baseline_tpr1),
        "A_mu_final": A_mu_final.tolist() if isinstance(A_mu_final, np.ndarray) else A_mu_final,
        "A_std_final": A_std_final.tolist() if isinstance(A_std_final, np.ndarray) else A_std_final,
        "final_payoff": float(final_payoff),
    }
    with open("metrics_qpso.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print("Saved metrics -> metrics_qpso.json")

    # Hard defense + quantum finetune
    print("\nStarting hard defense + quantum fine-tuning (this is expensive):")
    hard_defense_with_quantum_finetune(qc_model, qvae, feature_extractor, classical_decoder,
                                       train_images_all, train_labels_all,
                                       val_loader=val_loader,
                                       epochs=3,
                                       adv_samples_per_pos=8,
                                       adv_radius=0.5,
                                       batch_size_def=64,
                                       collapse_penalty_coef=12.0,
                                       lr=5e-4,
                                       subset_size=2000,
                                       fine_tune_quantum=True,
                                       lr_q=1e-2,
                                       quantum_batches_per_epoch=2,
                                       paramshift_shift=math.pi/2)

    print("Final evaluation on test set AFTER defense:")
    test_model(test_loader, qc_model)

    # Save artifacts
    np.save("qvae_encoder_params_qpso.npy", qvae.encoder_params)
    np.save("qvae_decoder_params_qpso.npy", qvae.decoder_params)
    np.save("qcnn_clf_params_qpso.npy", qc_model.clf_params)
    torch.save(qc_model.fc.state_dict(), "qcnn_fc_head_qpso.pth")
    print("Saved parameters. Done.")


In [1]:
# quantum_adversarial_game_QPSO.py
# (Updated with paper-style adversarial grid plotting and saving)
# Full merged script: Q-VAE + Q-CNN + ALS adversarial game using QPSO + hard defense + utilities

import os, time, math, hashlib, random, json
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split, ConcatDataset
from sklearn.metrics import confusion_matrix

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Prefer Aer Estimator (fast) if available; fallback to StatevectorEstimator
try:
    from qiskit_aer.primitives import Estimator as AerEstimator
    EstimatorClass = AerEstimator
    print("Using qiskit_aer Estimator (fast).")
except Exception:
    try:
        from qiskit.primitives import StatevectorEstimator as EstimatorClass
        print("Using qiskit.primitives.StatevectorEstimator (fallback).")
    except Exception:
        EstimatorClass = None
        print("Warning: No suitable Qiskit estimator found. EstimatorClass=None")

# ------------------------------
# Settings
# ------------------------------
latent_qubits = 8
batch_size = 64
seed = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

# ------------------------------
# MNIST Loader (6 vs 8)
# ------------------------------
def load_mnist_filtered(label1=6, label2=8, batch_size=64):
    try:
        from torchvision import datasets, transforms
        transform = transforms.Compose([transforms.ToTensor()])
        train_full = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
        test_full  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

        train_images = train_full.data.unsqueeze(1).float() / 255.0
        train_labels = train_full.targets.long()
        test_images = test_full.data.unsqueeze(1).float() / 255.0
        test_labels = test_full.targets.long()

        print("Loaded MNIST via torchvision.")
    except Exception as e:
        print("torchvision failed, falling back to sklearn:", e)
        from sklearn.datasets import fetch_openml
        mn = fetch_openml('mnist_784', version=1, as_frame=False)
        X = mn['data'].astype('float32') / 255.0
        y = mn['target'].astype(int)
        X = X.reshape(-1, 1, 28, 28)

        train_images, test_images = X[:60000], X[60000:]
        train_labels, test_labels = y[:60000], y[60000:]

        train_images = torch.from_numpy(train_images)
        train_labels = torch.from_numpy(train_labels).long()
        test_images = torch.from_numpy(test_images)
        test_labels = torch.from_numpy(test_labels).long()

        print("Loaded MNIST via sklearn.")

    mask_train = (train_labels == label1) | (train_labels == label2)
    mask_test  = (test_labels == label1) | (test_labels == label2)

    train_images = train_images[mask_train]
    train_labels = train_labels[mask_train]
    test_images  = test_images[mask_test]
    test_labels  = test_labels[mask_test]

    train_labels = torch.where(train_labels == label1, torch.tensor(0), torch.tensor(1))
    test_labels  = torch.where(test_labels  == label1, torch.tensor(0), torch.tensor(1))

    total_train = TensorDataset(train_images, train_labels)
    train_size = int(0.8 * len(total_train))
    val_size = len(total_train) - train_size

    train_dataset, val_dataset = random_split(total_train, [train_size, val_size])
    test_dataset = TensorDataset(test_images, test_labels)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print("Datasets prepared: Train", len(train_dataset), "Val", len(val_dataset), "Test", len(test_dataset))
    return train_loader, val_loader, test_loader, train_images, train_labels, test_images, test_labels

train_loader, val_loader, test_loader, train_images_all, train_labels_all, test_images_all, test_labels_all = \
    load_mnist_filtered(batch_size=batch_size)

# ------------------------------
# Classical feature extractor + decoder
# ------------------------------
input_dim = 28 * 28

class ClassicalFeatureExtractor(nn.Module):
    def __init__(self, latent_qubits):
        super().__init__()
        self.fc = nn.Linear(input_dim, latent_qubits)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return torch.tanh(self.fc(x))

class ClassicalDecoder(nn.Module):
    def __init__(self, latent_qubits):
        super().__init__()
        self.fc1 = nn.Linear(latent_qubits, 128)
        self.fc2 = nn.Linear(128, input_dim)
    def forward(self, z):
        z = F.relu(self.fc1(z))
        out = torch.sigmoid(self.fc2(z))
        return out.view(-1,1,28,28)

feature_extractor = ClassicalFeatureExtractor(latent_qubits).to(device)
classical_decoder = ClassicalDecoder(latent_qubits).to(device)

# ------------------------------
# Circuit caching utilities
# ------------------------------
CIRCUIT_CACHE = {}
def _params_to_list(params):
    if isinstance(params, np.ndarray):
        return params.flatten().tolist()
    elif isinstance(params, (list, tuple)):
        return np.array(params).flatten().tolist()
    else:
        return np.array(params).flatten().tolist()

def cache_key(type_name, params, angles):
    p_list = _params_to_list(params)
    a_list = np.round(np.array(angles).flatten(), 6).tolist()
    raw = f"{type_name}|{p_list}|{a_list}"
    return hashlib.sha256(raw.encode()).hexdigest()

def get_encoder_circuit_cached(n_qubits, encoder_params, x_angles):
    key = cache_key("enc", encoder_params, x_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(x_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(encoder_params[i,0]), i)
        qc.rz(float(encoder_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

def get_decoder_circuit_cached(n_qubits, decoder_params, z_angles):
    key = cache_key("dec", decoder_params, z_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(z_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(decoder_params[i,0]), i)
        qc.rz(float(decoder_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

def get_classifier_circuit_cached(n_qubits, clf_params, x_angles):
    key = cache_key("clf", clf_params, x_angles)
    if key in CIRCUIT_CACHE:
        return CIRCUIT_CACHE[key].copy()
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(float(x_angles[i]), i)
    for i in range(n_qubits):
        qc.ry(float(clf_params[i,0]), i)
        qc.rz(float(clf_params[i,1]), i)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    CIRCUIT_CACHE[key] = qc
    return qc.copy()

# ------------------------------
# Observables: Z individual + grouped
# ------------------------------
z_ops = []
for i in range(latent_qubits):
    s = ['I'] * latent_qubits
    s[latent_qubits - 1 - i] = 'Z'
    z_ops.append(SparsePauliOp(''.join(s)))
Z_GROUPED = z_ops[0]
for i in range(1, latent_qubits):
    Z_GROUPED = Z_GROUPED + z_ops[i]

# ------------------------------
# Estimator instance (wrap factory)
# ------------------------------
def create_estimator():
    if EstimatorClass is None:
        raise RuntimeError("EstimatorClass is not available in this environment.")
    try:
        return EstimatorClass()
    except Exception as e:
        try:
            return EstimatorClass()
        except Exception as e2:
            raise RuntimeError("Could not create estimator: " + str(e2))

estimator = None
try:
    estimator = create_estimator()
except Exception as e:
    print("Estimator creation failed:", e)

# ------------------------------
# measure_z_expectations_batch
# ------------------------------
def measure_z_expectations_batch(qc_list):
    if len(qc_list) == 0:
        return np.zeros((0, latent_qubits), dtype=float)
    n_circuits = len(qc_list)
    grouped_obs_list = [Z_GROUPED] * n_circuits
    job = estimator.run(circuits=qc_list, observables=grouped_obs_list)
    _ = job.result()
    values = np.zeros((n_circuits, latent_qubits), dtype=float)
    for q in range(latent_qubits):
        obs = z_ops[q]
        obs_list = [obs] * n_circuits
        job_q = estimator.run(circuits=qc_list, observables=obs_list)
        res_q = job_q.result()
        vals_q = np.array(res_q.values).reshape(-1)
        values[:, q] = vals_q
    return values

# ------------------------------
# QuantumVAE
# ------------------------------
class QuantumVAE:
    def __init__(self, n_qubits):
        self.n_qubits = n_qubits
        self.encoder_params = np.random.randn(n_qubits, 2) * 0.05
        self.decoder_params = np.random.randn(n_qubits, 2) * 0.05

    def encode(self, x_batch, angles_cache=None):
        x_batch = x_batch.detach().cpu()
        b = x_batch.size(0)
        if angles_cache is None:
            feats = feature_extractor(x_batch.to(device)).detach().cpu().numpy()
            angles = (feats + 1.0) * (np.pi / 2.0)
        else:
            angles = angles_cache
        qc_list = [get_encoder_circuit_cached(self.n_qubits, self.encoder_params, angles[i]) for i in range(b)]
        vals = measure_z_expectations_batch(qc_list)
        mu_angles = np.arcsin(np.clip(vals, -1.0, 1.0))
        mu = torch.tensor(mu_angles, dtype=torch.float32).to(device)
        logvar = torch.zeros_like(mu).to(device)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        eps = torch.randn_like(mu)
        return mu + eps * torch.exp(0.5 * logvar)

    def decode(self, z_batch):
        z_np = z_batch.detach().cpu().numpy()
        qc_list = [get_decoder_circuit_cached(self.n_qubits, self.decoder_params, z_np[i]) for i in range(z_np.shape[0])]
        vals = measure_z_expectations_batch(qc_list)
        return torch.tensor(vals, dtype=torch.float32).to(device)

    def forward(self, x_batch, angles_cache=None):
        mu, logvar = self.encode(x_batch, angles_cache)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

qvae = QuantumVAE(latent_qubits)

# ------------------------------
# QuantumCNN wrapper
# ------------------------------
class QuantumCNN(nn.Module):
    def __init__(self, n_qubits, n_classes=2):
        super().__init__()
        self.n_qubits = n_qubits
        self.clf_params = np.random.randn(n_qubits, 2) * 0.05
        self.fc = nn.Linear(n_qubits, n_classes).to(device)

    def quantum_features_batch(self, x_batch, angles_cache=None):
        x_batch = x_batch.detach().cpu()
        b = x_batch.size(0)
        if angles_cache is None:
            feats = feature_extractor(x_batch.to(device)).detach().cpu().numpy()
            angles = (feats + 1.0) * (np.pi / 2.0)
        else:
            angles = angles_cache
        qc_list = [get_classifier_circuit_cached(self.n_qubits, self.clf_params, angles[i]) for i in range(b)]
        vals = measure_z_expectations_batch(qc_list)
        return torch.tensor(vals, dtype=torch.float32).to(device)

    def forward(self, x_batch, angles_cache=None):
        feats_q = self.quantum_features_batch(x_batch, angles_cache)
        logits = self.fc(feats_q)
        return logits

qc_model = QuantumCNN(latent_qubits)

# ------------------------------
# Metrics & test
# ------------------------------
def test_model(loader, model):
    model.eval()
    all_targets = []
    all_preds = []
    with torch.no_grad():
        for data, targets in loader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs, 1)
            all_targets.extend(targets.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
    cm = confusion_matrix(all_targets, all_preds, labels=[0,1])
    if cm.size == 4:
        tp0, fn0 = cm[0,0], cm[0,1]
        tp1, fn1 = cm[1,1], cm[1,0]
    else:
        tp0 = fn0 = tp1 = fn1 = 0
    tpr0 = tp0 / (tp0 + fn0) if (tp0 + fn0) > 0 else 0.0
    tpr1 = tp1 / (tp1 + fn1) if (tp1 + fn1) > 0 else 0.0
    acc = (np.array(all_preds) == np.array(all_targets)).mean() if len(all_targets) else 0.0
    print(f"Acc: {acc:.4f}, TPR0: {tpr0:.4f}, TPR1: {tpr1:.4f}")
    return tpr0, tpr1

# ------------------------------
# Adversarial utilities (latent)
# ------------------------------
def quantum_adversarial_latent(mu_pos, logvar_pos, A_mu, A_std):
    if isinstance(A_mu, np.ndarray):
        A_mu = torch.tensor(A_mu, dtype=torch.float32).to(device)
    if isinstance(A_std, np.ndarray):
        A_std = torch.tensor(A_std, dtype=torch.float32).to(device)
    mu_adv = mu_pos + A_mu
    logvar_adv = logvar_pos + A_std
    eps = torch.randn_like(mu_adv)
    z_adv = mu_adv + eps * torch.exp(0.5 * logvar_adv)
    return z_adv, mu_adv, logvar_adv

def calc_cost(a_mean, a_std):
    a_mean_t = torch.tensor(a_mean, dtype=torch.float32) if not isinstance(a_mean, torch.Tensor) else a_mean
    a_std_t = torch.tensor(a_std, dtype=torch.float32) if not isinstance(a_std, torch.Tensor) else a_std
    norm_mean = torch.norm(a_mean_t)
    norm_std = torch.norm(a_std_t)
    total_cost = (norm_mean / latent_qubits) + (norm_std / latent_qubits)
    return total_cost.item()

def adversary_payoff(posnegdata, posindices, negindices, mu_pos, std_pos, qvae_obj, a_mean, a_std, posnegtargets, model):
    if mu_pos.dim() == 1:
        mu_pos = mu_pos.unsqueeze(0)
    if std_pos.dim() == 1:
        std_pos = std_pos.unsqueeze(0)
    z_adv, _, _ = quantum_adversarial_latent(mu_pos, std_pos, a_mean, a_std)
    logits = model.fc(z_adv.to(device))
    _, preds = torch.max(logits, 1)
    targets = torch.tensor(posnegtargets, dtype=torch.long).to(device)
    true_positives = ((preds == 1) & (targets == 1)).sum().item()
    false_negatives = ((preds == 0) & (targets == 1)).sum().item()
    tpr1 = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0.0
    error = 1.0 - tpr1
    cost = calc_cost(a_mean, a_std)
    payoff = 1.0 + error - cost
    return payoff

# ------------------------------
# QPSO optimizer (replaces PSO / SA)
# ------------------------------
class QPSO:
    """
    Quantum-behaved Particle Swarm Optimizer (QPSO)
    Minimizes fitness_fn(x) where x is a 1D numpy vector.
    """
    def __init__(self, fitness_fn, bounds, num_particles=30, max_iter=300,
                 alpha_start=1.0, alpha_end=0.1, seed=None, verbose=False):
        self.fitness_fn = fitness_fn
        self.bounds = np.array(bounds, dtype=float)
        assert self.bounds.ndim == 2 and self.bounds.shape[1] == 2
        self.D = self.bounds.shape[0]

        self.num_particles = int(num_particles)
        self.max_iter = int(max_iter)
        self.alpha_start = float(alpha_start)
        self.alpha_end = float(alpha_end)
        self.verbose = bool(verbose)

        self.rng = np.random.RandomState(seed)

        # initialize swarm positions uniformly within bounds
        self.positions = self.rng.uniform(self.bounds[:,0], self.bounds[:,1], size=(self.num_particles, self.D))
        self.pbest_pos = self.positions.copy()
        self.pbest_val = np.array([self._safe_eval(self.positions[i]) for i in range(self.num_particles)], dtype=float)

        idx = int(np.argmin(self.pbest_val))
        self.gbest_pos = self.pbest_pos[idx].copy()
        self.gbest_val = float(self.pbest_val[idx])

    def _safe_eval(self, x):
        try:
            v = float(self.fitness_fn(x))
            if np.isnan(v) or np.isinf(v):
                return np.finfo(float).max
            return v
        except Exception:
            return np.finfo(float).max

    def _clip(self, x):
        return np.minimum(np.maximum(x, self.bounds[:,0]), self.bounds[:,1])

    def optimize(self):
        alpha = self.alpha_start
        for t in range(1, self.max_iter + 1):
            mbest = np.mean(self.pbest_pos, axis=0)
            frac = (t - 1) / max(1, (self.max_iter - 1))
            alpha = self.alpha_start + frac * (self.alpha_end - self.alpha_start)

            for i in range(self.num_particles):
                phi = self.rng.rand(self.D)
                p_i = phi * self.pbest_pos[i] + (1.0 - phi) * self.gbest_pos

                u = self.rng.rand(self.D)
                u = np.clip(u, 1e-12, 1.0-1e-12)
                sign = self.rng.choice([1.0, -1.0], size=self.D)

                distance = np.abs(mbest - self.positions[i])
                ln_term = np.log(1.0 / u)

                x_new = p_i + sign * (alpha * distance * ln_term)
                x_new = self._clip(x_new)

                fx_new = self._safe_eval(x_new)

                if fx_new < self.pbest_val[i]:
                    self.pbest_val[i] = fx_new
                    self.pbest_pos[i] = x_new.copy()

                if fx_new < self.gbest_val:
                    self.gbest_val = float(fx_new)
                    self.gbest_pos = x_new.copy()

                self.positions[i] = x_new

            if self.verbose and (t % max(1, self.max_iter // 10) == 0):
                print(f"[QPSO] iter {t}/{self.max_iter} gbest={self.gbest_val:.6e}")

        return self.gbest_pos.copy(), float(self.gbest_val)

# ------------------------------
# Alternating least squares adversarial game (QPSO inner optimizer)
# ------------------------------
def alternating_least_squares_game(qc_model, qvae_obj, train_images_all, train_labels_all,
                                   max_game_iter=3, qpso_iter=200, n_pos_samples=80,
                                   qpso_particles=30, qpso_alpha_start=1.0, qpso_alpha_end=0.2,
                                   qpso_seed=1234, retrain_epochs=3):
    imgs = train_images_all
    labs = train_labels_all
    pos_mask = (labs == 1)
    neg_mask = (labs == 0)
    pos_imgs = imgs[pos_mask][:n_pos_samples].to(device)
    pos_labs = labs[pos_mask][:n_pos_samples].to(device)
    neg_imgs = imgs[neg_mask][:n_pos_samples].to(device)
    neg_labs = labs[neg_mask][:n_pos_samples].to(device)

    combined = torch.cat([pos_imgs, neg_imgs], dim=0)
    combined_targets = torch.cat([pos_labs, neg_labs], dim=0)
    pos_indices = (combined_targets.cpu().numpy() == 1).tolist()
    neg_indices = (combined_targets.cpu().numpy() == 0).tolist()

    A_mu = np.zeros(latent_qubits, dtype=float)
    A_std = np.zeros(latent_qubits, dtype=float)
    payoff_curr = -np.inf

    D = latent_qubits
    bounds = np.vstack([np.full(D, -0.5), np.full(D, 0.5)]).T

    for game_iter in range(max_game_iter):
        with torch.no_grad():
            mu_pos_all, logvar_pos_all = qvae_obj.encode(pos_imgs)
        mu_pos_avg = mu_pos_all.mean(dim=0)
        std_pos_avg = logvar_pos_all.mean(dim=0)

        def fit_mean(alpha_vec):
            return -adversary_payoff(combined, pos_indices, neg_indices,
                                     mu_pos_avg, std_pos_avg, qvae_obj,
                                     alpha_vec, A_std, combined_targets.cpu().numpy(), qc_model)

        qpso_mean = QPSO(fit_mean, bounds,
                         num_particles=qpso_particles, max_iter=qpso_iter,
                         alpha_start=qpso_alpha_start, alpha_end=qpso_alpha_end,
                         seed=qpso_seed + game_iter + 1, verbose=False)
        alpha_star_mean, best_mean_val = qpso_mean.optimize()

        def fit_std(alpha_vec):
            return -adversary_payoff(combined, pos_indices, neg_indices,
                                     mu_pos_avg, std_pos_avg, qvae_obj,
                                     alpha_star_mean, alpha_vec, combined_targets.cpu().numpy(), qc_model)

        qpso_std = QPSO(fit_std, bounds,
                        num_particles=qpso_particles, max_iter=qpso_iter,
                        alpha_start=qpso_alpha_start, alpha_end=qpso_alpha_end,
                        seed=qpso_seed + game_iter + 10, verbose=False)
        alpha_star_std, best_std_val = qpso_std.optimize()

        payoff_new = adversary_payoff(combined, pos_indices, neg_indices,
                                      mu_pos_avg, std_pos_avg, qvae_obj,
                                      alpha_star_mean, alpha_star_std, combined_targets.cpu().numpy(), qc_model)
        print(f"[ALS QPSO] Game {game_iter}: payoff_curr={payoff_curr:.6f} payoff_new={payoff_new:.6f}")

        if payoff_new > payoff_curr:
            payoff_curr = payoff_new
            A_mu = alpha_star_mean.copy()
            A_std = alpha_star_std.copy()

            z_adv, _, _ = quantum_adversarial_latent(mu_pos_all, logvar_pos_all, A_mu, A_std)
            adv_images = classical_decoder(z_adv.to(device))
            adv_targets = torch.ones(adv_images.size(0), dtype=torch.long).to(device)

            subset_size = min(500, len(train_images_all))
            train_data_subset = train_images_all[:subset_size].to(device)
            train_targets_subset = train_labels_all[:subset_size].to(device)
            combined_images = torch.cat([train_data_subset, adv_images], dim=0)
            combined_targets2 = torch.cat([train_targets_subset, adv_targets], dim=0)
            combined_ds = TensorDataset(combined_images, combined_targets2)
            combined_loader = DataLoader(combined_ds, batch_size=32, shuffle=True)

            qc_model.train()
            optimizer = torch.optim.Adam(qc_model.fc.parameters(), lr=1e-3)
            loss_fn = nn.CrossEntropyLoss()
            for epoch in range(retrain_epochs):
                run_loss = 0.0
                for bimgs, btargets in combined_loader:
                    bimgs = bimgs.to(device); btargets = btargets.to(device)
                    with torch.no_grad():
                        mu_b, logvar_b = qvae_obj.encode(bimgs)
                        z_b = qvae_obj.reparameterize(mu_b, logvar_b)
                    logits = qc_model.fc(z_b.to(device))
                    loss = loss_fn(logits, btargets)
                    optimizer.zero_grad(); loss.backward(); optimizer.step()
                    run_loss += loss.item()
                print(f" Retrain epoch {epoch+1}, loss {run_loss/len(combined_loader):.4f}")
        else:
            print("No improvement — stopping ALS.")
            break

    return A_mu, A_std, payoff_curr

# ------------------------------
# Parameter-shift helpers (quantum finetuning)
# ------------------------------
def compute_features_and_grads_batch_for_clf_params(x_batch, clf_params_np, n_qubits):
    with torch.no_grad():
        feats_classical = feature_extractor(x_batch.to(device)).detach().cpu().numpy()
    angles = (feats_classical + 1.0) * (np.pi / 2.0)
    qc_list = [get_classifier_circuit_cached(n_qubits, clf_params_np, angles[i]) for i in range(angles.shape[0])]
    vals = measure_z_expectations_batch(qc_list)
    feats_tensor = torch.tensor(vals, dtype=torch.float32).to(device)
    return feats_tensor, angles

def parameter_shift_update(clf_params_np, x_batch, batch_labels, n_qubits,
                           qc_model, lr_q=1e-2, shift=np.pi/2):
    feats_base, angles = compute_features_and_grads_batch_for_clf_params(x_batch, clf_params_np, n_qubits)
    feats_base = feats_base.clone().detach().requires_grad_(True)
    logits = qc_model.fc(feats_base)
    loss_fn = nn.CrossEntropyLoss()
    loss = loss_fn(logits, batch_labels.to(device))
    qc_model.fc.zero_grad()
    if feats_base.grad is not None:
        feats_base.grad.zero_()
    loss.backward(retain_graph=True)
    dL_dfeat = feats_base.grad.detach().cpu().numpy()  # (B, D)
    B = dL_dfeat.shape[0]
    grads_theta = np.zeros_like(clf_params_np, dtype=float)
    for q in range(n_qubits):
        for p_idx in range(2):
            clf_plus = clf_params_np.copy()
            clf_minus = clf_params_np.copy()
            clf_plus[q, p_idx] += shift
            clf_minus[q, p_idx] -= shift
            f_plus_tensor, _ = compute_features_and_grads_batch_for_clf_params(x_batch, clf_plus, n_qubits)
            f_minus_tensor, _ = compute_features_and_grads_batch_for_clf_params(x_batch, clf_minus, n_qubits)
            f_plus = f_plus_tensor.detach().cpu().numpy()
            f_minus = f_minus_tensor.detach().cpu().numpy()
            df_dtheta = 0.5 * (f_plus - f_minus)
            term = np.sum(dL_dfeat * df_dtheta)
            grads_theta[q, p_idx] = term / float(B)
    clf_params_np = clf_params_np - lr_q * grads_theta
    return clf_params_np, loss.item()

# ------------------------------
# Hard defense + quantum finetune
# ------------------------------
def hard_defense_with_quantum_finetune(qc_model, qvae, feature_extractor, classical_decoder,
                                       train_images_all, train_labels_all,
                                       val_loader=None,
                                       epochs=4,
                                       adv_samples_per_pos=8,
                                       adv_radius=0.5,
                                       batch_size_def=64,
                                       collapse_penalty_coef=12.0,
                                       lr=5e-4,
                                       subset_size=3000,
                                       fine_tune_quantum=True,
                                       lr_q=1e-2,
                                       quantum_batches_per_epoch=3,
                                       paramshift_shift=math.pi/2):
    feature_extractor.train(); qc_model.train()
    params = list(feature_extractor.parameters()) + list(qc_model.fc.parameters())
    optimizer = torch.optim.Adam(params, lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    device_local = device
    total_clean = min(subset_size, len(train_images_all))
    clean_images = train_images_all[:total_clean].to(device_local)
    clean_labels = train_labels_all[:total_clean].to(device_local)
    pos_mask = (train_labels_all == 1)
    pos_imgs_all = train_images_all[pos_mask]
    if len(pos_imgs_all) == 0:
        raise RuntimeError("No positive-class examples found.")
    pos_sample_count = min(200, len(pos_imgs_all))
    pos_imgs = pos_imgs_all[:pos_sample_count].to(device_local)
    clf_params_np = qc_model.clf_params.copy()
    for epoch in range(epochs):
        epoch_loss = 0.0; n_batches = 0; quantum_updates_done = 0
        with torch.no_grad():
            mu_pos_all, logvar_pos_all = qvae.encode(pos_imgs)
            mu_pos_rep = mu_pos_all.repeat_interleave(adv_samples_per_pos, dim=0)
            logvar_rep = logvar_pos_all.repeat_interleave(adv_samples_per_pos, dim=0)
            delta = torch.randn_like(mu_pos_rep, device=device_local) * adv_radius
            std_delta = torch.randn_like(logvar_rep, device=device_local) * (adv_radius * 0.3)
            mu_adv = mu_pos_rep + delta
            std_adv = torch.clamp(logvar_rep + std_delta, min=-3.0, max=3.0)
            eps = torch.randn_like(mu_adv)
            z_adv = mu_adv + eps * torch.exp(0.5 * std_adv)
            adv_images = classical_decoder(z_adv.to(device_local))
            adv_labels = torch.ones(adv_images.size(0), dtype=torch.long, device=device_local)
        idxs = torch.randperm(clean_images.size(0))
        clean_shuffled = clean_images[idxs]; clean_shuffled_labels = clean_labels[idxs]
        combined_images = torch.cat([clean_shuffled, adv_images], dim=0)
        combined_labels = torch.cat([clean_shuffled_labels, adv_labels], dim=0)
        combined_ds = TensorDataset(combined_images, combined_labels)
        combined_loader = DataLoader(combined_ds, batch_size=batch_size_def, shuffle=True)
        for batch_idx, (batch_imgs, batch_labels) in enumerate(combined_loader):
            batch_imgs = batch_imgs.to(device_local); batch_labels = batch_labels.to(device_local)
            feats = feature_extractor(batch_imgs)
            logits = qc_model.fc(feats)
            ce_loss = loss_fn(logits, batch_labels)
            probs1 = torch.softmax(logits, dim=1)[:, 1]
            mean_p1 = probs1.mean()
            collapse_penalty = collapse_penalty_coef * (mean_p1 - 0.5) ** 2
            l2_reg = 0.0
            for p in params:
                l2_reg += 1e-4 * torch.sum(p ** 2)
            loss = ce_loss + collapse_penalty + l2_reg
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            epoch_loss += loss.item(); n_batches += 1
            if fine_tune_quantum and (quantum_updates_done < quantum_batches_per_epoch):
                try:
                    clf_params_np, qloss = parameter_shift_update(clf_params_np, batch_imgs, batch_labels, latent_qubits, qc_model, lr_q=lr_q, shift=paramshift_shift)
                    qc_model.clf_params = clf_params_np.copy()
                    quantum_updates_done += 1
                except Exception as e:
                    print("Parameter-shift update error:", e)
        avg_loss = epoch_loss / max(1, n_batches)
        print(f"[HardDef+QEpoch {epoch+1}/{epochs}] Avg loss: {avg_loss:.4f}, quantum_updates: {quantum_updates_done}")
        if val_loader is not None:
            print("Validation eval:")
            test_model(val_loader, qc_model)
    qc_model.clf_params = clf_params_np.copy()
    print("Hard defense + quantum fine-tune complete.")

# ------------------------------
# Utilities and safety helpers
# ------------------------------
DEBUG = False
ENABLE_PROFILING = False

def debug(*args):
    if DEBUG:
        print("[DEBUG]", *args)

class Timer:
    def __init__(self, name="timer"):
        self.name = name
    def __enter__(self):
        if ENABLE_PROFILING:
            self.start = time.time()
        return self
    def __exit__(self, exc_type, exc, tb):
        if ENABLE_PROFILING:
            dt = time.time() - self.start
            print(f"[PROFILE] {self.name}: {dt:.4f}s")

def safe_reparameterize(mu, logvar):
    with torch.no_grad():
        eps = torch.randn_like(mu)
        logvar = torch.clamp(logvar, -10.0, 10.0)
        return mu + eps * torch.exp(0.5 * logvar)

QuantumVAE.reparameterize = lambda self, mu, logvar: safe_reparameterize(mu, logvar)

# ------------------------------
# Runtime checks
# ------------------------------
def check_environment():
    print("\n=== Runtime Environment Check ===")
    print("Torch:", torch.__version__)
    try:
        import qiskit
        print("Qiskit:", qiskit.__version__)
    except:
        print("Qiskit version UNKNOWN")
    try:
        import qiskit_aer
        print("Qiskit Aer:", qiskit_aer.__version__)
    except:
        print("Qiskit Aer NOT FOUND (using estimator fallback).")
    if torch.cuda.is_available():
        print("CUDA available ✔")
        try:
            print("GPU:", torch.cuda.get_device_name(0))
        except:
            pass
    else:
        print("CUDA not available ✘ — running on CPU")
    print("Device selected:", device)
    print("=================================")

def check_qiskit_primitives():
    print("\n=== Qiskit Primitive Self-Test ===")
    if EstimatorClass is None:
        print("No EstimatorClass available; skipping primitive self-test.")
        return
    try:
        est = EstimatorClass()
        qc = QuantumCircuit(1)
        qc.h(0)
        obs = SparsePauliOp("Z")
        job = est.run([qc], [obs])
        res = job.result()
        print("Estimator primitive OK. Test value length:", len(res.values))
    except Exception as e:
        print("ERROR: Estimator primitive failed:", e)
        raise RuntimeError("Estimator primitive is not functional.")
    print("===================================")

def check_dataset_shapes(train_images_all, train_labels_all):
    print("\n=== Dataset Sanity Check ===")
    print("Train images:", train_images_all.shape)
    print("Train labels:", train_labels_all.shape)
    try:
        n0 = (train_labels_all == 0).sum().item()
        n1 = (train_labels_all == 1).sum().item()
        print("Class-0 count:", n0)
        print("Class-1 count:", n1)
        if n0 == 0 or n1 == 0:
            raise RuntimeError("One class has zero samples — cannot train binary classifier.")
    except Exception as e:
        print("Warning checking class counts:", e)
    print("===================================")

# ------------------------------
# Adversarial Grid Plotting (paper-style)
# ------------------------------
import matplotlib.pyplot as plt
from PIL import Image

def plot_adversarial_grid(out_dir="adv_examples", target_class=8, n_cols=10,
                          save_name="adv_grid_target8.png"):
    fig, axes = plt.subplots(nrows=10, ncols=n_cols, figsize=(n_cols*1.2, 10*1.2))
    fig.suptitle(f"Targeted Adversarial Reconstructions → Class {target_class}", fontsize=18)
    for source_c in range(10):
        class_dir = os.path.join(out_dir, f"target_{target_class}", f"source_{source_c}")
        if not os.path.exists(class_dir):
            for col in range(n_cols): axes[source_c, col].axis("off")
            continue
        png_files = sorted([f for f in os.listdir(class_dir) if f.endswith(".png")])[:n_cols]
        for col in range(n_cols):
            ax = axes[source_c, col]; ax.axis("off")
            if col < len(png_files):
                img = Image.open(os.path.join(class_dir, png_files[col])).convert("L")
                ax.imshow(img, cmap="gray")
            else:
                ax.imshow(np.zeros((28,28)), cmap="gray")
        axes[source_c, 0].set_ylabel(f"Src {source_c}", fontsize=10)
    plt.tight_layout(rect=[0,0,1,0.97])
    plt.savefig(save_name, dpi=300)
    plt.savefig(save_name.replace(".png", ".pdf"))
    print(f"Saved adversarial grid → {save_name}")

# ------------------------------
# Main demo run: ALS with QPSO then hard defense
# ------------------------------
if __name__ == "__main__":
    print('\nQUANTUM ADVERSARIAL GAME (ALS with QPSO)')
    check_environment()
    try:
        check_qiskit_primitives()
    except Exception as e:
        print("Estimator self-test failed; continuing but execution may fail when calling estimator.")
    check_dataset_shapes(train_images_all, train_labels_all)

    print("\nRunning alternating least squares adversarial game with QPSO (short demo):")
    start = time.time()
    A_mu_final, A_std_final, final_payoff = alternating_least_squares_game(qc_model, qvae, train_images_all, train_labels_all,
                                                                           max_game_iter=2, qpso_iter=150, n_pos_samples=60,
                                                                           qpso_particles=30, qpso_alpha_start=1.0, qpso_alpha_end=0.2,
                                                                           qpso_seed=seed, retrain_epochs=3)
    print("Game finished in {:.1f}s".format(time.time() - start))
    print("A_mu:", A_mu_final)
    print("A_std:", A_std_final)
    print("Final payoff:", final_payoff)

    print("\nTest eval BEFORE defense (baseline):")
    baseline_tpr0, baseline_tpr1 = test_model(test_loader, qc_model)

    # ALPHA★ POST-GAME ATTACK EVALUATION
    print("\n=== ALPHA★ POST-GAME ATTACK EVALUATION ===")
    pos_mask_test = (test_labels_all == 1)
    pos_imgs_test = test_images_all[pos_mask_test]
    if pos_imgs_test.size(0) == 0:
        print("No positive test samples found; skipping alpha-star attack evaluation.")
    else:
        n_attack_samples = min(120, pos_imgs_test.size(0))
        pos_imgs_test = pos_imgs_test[:n_attack_samples].to(device)
        with torch.no_grad():
            mu_pos_test, logvar_pos_test = qvae.encode(pos_imgs_test)
        z_star, mu_star, logvar_star = quantum_adversarial_latent(mu_pos_test, logvar_pos_test, A_mu_final, A_std_final)
        qc_model.eval()
        with torch.no_grad():
            logits_star = qc_model.fc(z_star.to(device))
            _, preds_star = torch.max(logits_star, 1)
            true_labels_star = torch.ones_like(preds_star)
            tp1 = ((preds_star == 1) & (true_labels_star == 1)).sum().item()
            fn1 = ((preds_star == 0) & (true_labels_star == 1)).sum().item()
            tpr1_attack = tp1 / (tp1 + fn1 + 1e-12)
            asr = 1.0 - tpr1_attack
        print("\n--- Alpha★ Attack Results (latent eval) ---")
        print(f"Num attack samples: {n_attack_samples}")
        print(f"TPR1 on alpha★ manipulated positives (latent): {tpr1_attack:.4f}")
        print(f"Attack Success Rate (ASR): {asr:.4f}")
        print(f"Payoff at convergence: {final_payoff:.4f}")
        with torch.no_grad():
            recon_star = classical_decoder(z_star.to(device))
            mu_re, logvar_re = qvae.encode(recon_star)
            z_re = qvae.reparameterize(mu_re, logvar_re)
            logits_re = qc_model.fc(z_re.to(device))
            _, preds_re = torch.max(logits_re, 1)
            acc_recon = (preds_re == torch.ones_like(preds_re)).float().mean().item()
        print("\nClassifier performance on reconstructed alpha★ images (pixel-space):")
        print(f"Accuracy on reconstructions: {acc_recon:.4f}")

    # Save metrics
    metrics = {
        "baseline_tpr0": float(baseline_tpr0),
        "baseline_tpr1": float(baseline_tpr1),
        "A_mu_final": A_mu_final.tolist() if isinstance(A_mu_final, np.ndarray) else A_mu_final,
        "A_std_final": A_std_final.tolist() if isinstance(A_std_final, np.ndarray) else A_std_final,
        "final_payoff": float(final_payoff),
    }
    with open("metrics_qpso.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print("Saved metrics -> metrics_qpso.json")

    # Hard defense + quantum finetune
    print("\nStarting hard defense + quantum fine-tuning (this is expensive):")
    hard_defense_with_quantum_finetune(qc_model, qvae, feature_extractor, classical_decoder,
                                       train_images_all, train_labels_all,
                                       val_loader=val_loader,
                                       epochs=3,
                                       adv_samples_per_pos=8,
                                       adv_radius=0.5,
                                       batch_size_def=64,
                                       collapse_penalty_coef=12.0,
                                       lr=5e-4,
                                       subset_size=2000,
                                       fine_tune_quantum=True,
                                       lr_q=1e-2,
                                       quantum_batches_per_epoch=2,
                                       paramshift_shift=math.pi/2)

    print("Final evaluation on test set AFTER defense:")
    test_model(test_loader, qc_model)

    # Save artifacts
    np.save("qvae_encoder_params_qpso.npy", qvae.encoder_params)
    np.save("qvae_decoder_params_qpso.npy", qvae.decoder_params)
    np.save("qcnn_clf_params_qpso.npy", qc_model.clf_params)
    torch.save(qc_model.fc.state_dict(), "qcnn_fc_head_qpso.pth")
    print("Saved parameters. Done.")


Using qiskit_aer Estimator (fast).
Device: cpu
Loaded MNIST via torchvision.
Datasets prepared: Train 9415 Val 2354 Test 1932


C:\Users\Aaditya Rajput\AppData\Local\Temp\ipykernel_32084\827874111.py:217: DeprecationWarning: Estimator has been deprecated as of Aer 0.15, please use EstimatorV2 instead.
  estimator = create_estimator()
C:\Users\Aaditya Rajput\AppData\Local\Temp\ipykernel_32084\827874111.py:217: DeprecationWarning: Option approximation=False is deprecated as of qiskit-aer 0.13. It will be removed no earlier than 3 months after the release date. Instead, use BackendEstimator from qiskit.primitives.
  estimator = create_estimator()



QUANTUM ADVERSARIAL GAME (ALS with QPSO)

=== Runtime Environment Check ===
Torch: 2.9.1+cpu
Qiskit: 2.2.3
Qiskit Aer: 0.17.2
CUDA not available ✘ — running on CPU
Device selected: cpu

=== Qiskit Primitive Self-Test ===


C:\Users\Aaditya Rajput\AppData\Local\Temp\ipykernel_32084\827874111.py:807: DeprecationWarning: Estimator has been deprecated as of Aer 0.15, please use EstimatorV2 instead.
  check_qiskit_primitives()
C:\Users\Aaditya Rajput\AppData\Local\Temp\ipykernel_32084\827874111.py:807: DeprecationWarning: Option approximation=False is deprecated as of qiskit-aer 0.13. It will be removed no earlier than 3 months after the release date. Instead, use BackendEstimator from qiskit.primitives.
  check_qiskit_primitives()


Estimator primitive OK. Test value length: 1

=== Dataset Sanity Check ===
Train images: torch.Size([11769, 1, 28, 28])
Train labels: torch.Size([11769])
Class-0 count: 5918
Class-1 count: 5851

Running alternating least squares adversarial game with QPSO (short demo):
[ALS QPSO] Game 0: payoff_curr=-inf payoff_new=1.999999
 Retrain epoch 1, loss 0.7886
 Retrain epoch 2, loss 0.7791
 Retrain epoch 3, loss 0.7433
[ALS QPSO] Game 1: payoff_curr=1.999999 payoff_new=1.999997
No improvement — stopping ALS.
Game finished in 249.1s
A_mu: [-5.60693873e-07  1.38811387e-07  6.94861329e-07  2.13720701e-07
 -5.29816178e-07  1.74899395e-06  1.37096022e-07  1.86910279e-07]
A_std: [ 3.82048846e-07  7.26911910e-07  2.52943781e-07 -9.91797406e-08
 -3.20134100e-07 -9.35335530e-07  1.99929596e-06 -3.17312317e-07]
Final payoff: 1.9999994404907966

Test eval BEFORE defense (baseline):
Acc: 0.4959, TPR0: 1.0000, TPR1: 0.0000

=== ALPHA★ POST-GAME ATTACK EVALUATION ===

--- Alpha★ Attack Results (latent eval